In [74]:
# Import libraries and define configurations
import json
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
import joblib
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay,
    calibration_curve,
)
from sklearn.exceptions import ConvergenceWarning

RANDOM_STATE = 121
N_JOBS = -1
LOGISTIC_BASELINE_MAX_ITER = 2_000
REGULARIZED_LOGISTIC_MAX_ITER = 2_000
REGULARIZED_LOGISTIC_N_JOBS = min(
    joblib.cpu_count(),
    8,
)
DECISION_TREE_MAX_DEPTH = 5
DECISION_TREE_MIN_SAMPLES_SPLIT = 100
DECISION_TREE_MIN_SAMPLES_LEAF = 50
RANDOM_FOREST_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)
HIST_GRADIENT_BOOSTING_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)
FEATURE_SET_COMPARISON_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)
DONATION_REPRESENTATION_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)
REDUNDANCY_EXPERIMENT_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)


np.random.seed(RANDOM_STATE)


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_WORKING_DIRECTORY.parent
    if CURRENT_WORKING_DIRECTORY.name == "notebooks"
    else CURRENT_WORKING_DIRECTORY
)

PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
REPORTS_DIRECTORY = PROJECT_ROOT / "reports"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

ENGINEERED_FEATURES_PARQUET_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.parquet"
)
ENGINEERED_FEATURES_CSV_PATH = (
    PROCESSED_DATA_DIRECTORY / "donor_features.csv"
)
CLEANED_DONOR_DATA_PATH = (
    PROCESSED_DATA_DIRECTORY / "cleaned_donor_data.csv"
)
FEATURE_DICTIONARY_PATH = (
    REPORTS_DIRECTORY / "feature_dictionary.csv"
)

MODEL_PREDICTIONS_PATH = (
    PROCESSED_DATA_DIRECTORY / "model_predictions.csv"
)
FINAL_PRIMARY_PIPELINE_PATH = (
    MODELS_DIRECTORY / "final_primary_donor_pipeline.joblib"
)
MODEL_COMPARISON_RESULTS_PATH = (
    REPORTS_DIRECTORY / "05_model_comparison_results.csv"
)
CLASSIFICATION_MODELING_REPORT_PATH = (
    REPORTS_DIRECTORY / "05_classification_modeling.md"
)

MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
REPORTS_DIRECTORY.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

sns.set_theme(style="whitegrid")
set_config(display="diagram")

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Load the engineered dataset and feature dictionary
def format_project_path(path):
    relative_path = Path(path).resolve().relative_to(PROJECT_ROOT)
    return f"/{PROJECT_ROOT.name}/{relative_path.as_posix()}"


artifact_availability = pd.DataFrame({
    "artifact": [
        "Engineered features Parquet",
        "Engineered features CSV fallback",
        "Feature dictionary",
        "Cleaned donor dataset",
    ],
    "path_object": [
        ENGINEERED_FEATURES_PARQUET_PATH,
        ENGINEERED_FEATURES_CSV_PATH,
        FEATURE_DICTIONARY_PATH,
        CLEANED_DONOR_DATA_PATH,
    ],
})

artifact_availability["path"] = artifact_availability["path_object"].apply(
    format_project_path
)
artifact_availability["available"] = artifact_availability["path_object"].apply(
    Path.exists
)

display(artifact_availability[
    ["artifact", "path", "available"]
].style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
)

if ENGINEERED_FEATURES_PARQUET_PATH.exists():
    modeling_data = pd.read_parquet(
        ENGINEERED_FEATURES_PARQUET_PATH
    )
    modeling_data_source = "Parquet"
    modeling_data_path = ENGINEERED_FEATURES_PARQUET_PATH

elif ENGINEERED_FEATURES_CSV_PATH.exists():
    modeling_data = pd.read_csv(
        ENGINEERED_FEATURES_CSV_PATH
    )
    modeling_data_source = "CSV fallback"
    modeling_data_path = ENGINEERED_FEATURES_CSV_PATH

else:
    raise FileNotFoundError(
        "Neither donor_features.parquet nor donor_features.csv "
        "was found in the processed data directory."
    )

if not FEATURE_DICTIONARY_PATH.exists():
    raise FileNotFoundError(
        f"Feature dictionary not found: "
        f"{format_project_path(FEATURE_DICTIONARY_PATH)}"
    )

feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_PATH
)

print(f"\nModeling dataset source: {modeling_data_source}")
print(
    "Modeling dataset path:",
    format_project_path(modeling_data_path),
)
print(
    "Feature dictionary path:",
    format_project_path(FEATURE_DICTIONARY_PATH),
)

print(
    "\nModeling dataset loaded:",
    f"{modeling_data.shape[0]:,} rows × "
    f"{modeling_data.shape[1]:,} columns",
)

print(
    "Feature dictionary loaded:",
    f"{feature_dictionary.shape[0]:,} rows × "
    f"{feature_dictionary.shape[1]:,} columns\n",
)

artifact,path,available
Engineered features Parquet,/red-cross-donor-prediction/data/processed/donor_features.parquet,True
Engineered features CSV fallback,/red-cross-donor-prediction/data/processed/donor_features.csv,True
Feature dictionary,/red-cross-donor-prediction/reports/feature_dictionary.csv,True
Cleaned donor dataset,/red-cross-donor-prediction/data/processed/cleaned_donor_data.csv,True



Modeling dataset source: Parquet
Modeling dataset path: /red-cross-donor-prediction/data/processed/donor_features.parquet
Feature dictionary path: /red-cross-donor-prediction/reports/feature_dictionary.csv

Modeling dataset loaded: 34,403 rows × 55 columns
Feature dictionary loaded: 77 rows × 7 columns



In [3]:
# Validate the Phase 4 modeling export
EXPECTED_ROW_COUNT = 34_403
EXPECTED_COLUMN_COUNT = 55
EXPECTED_PREDICTOR_COUNT = 53

TRACKING_IDENTIFIER_COLUMN = "donor_unique_id"
PRIMARY_TARGET_COLUMN = "target_current_fiscal_year_donor_flag"

DIRECT_LEAKAGE_COLUMNS = {
    "current_fiscal_year_donation",
    "cumulative_donation_amount",
}

EXPECTED_TARGET_COUNTS = {
    0: 32_499,
    1: 1_904,
}

excluded_modeling_columns = {
    TRACKING_IDENTIFIER_COLUMN,
    PRIMARY_TARGET_COLUMN,
}

predictor_columns = [
    column
    for column in modeling_data.columns
    if column not in excluded_modeling_columns
]

unexpected_direct_leakage_columns = sorted(
    set(predictor_columns).intersection(DIRECT_LEAKAGE_COLUMNS)
)

numeric_columns = modeling_data.select_dtypes(
    include=[np.number]
).columns

infinite_value_count = int(
    np.isinf(modeling_data[numeric_columns]).sum().sum()
)

target_distribution = (
    modeling_data[PRIMARY_TARGET_COLUMN]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("target_class")
    .reset_index(name="record_count")
)

target_distribution["percentage"] = (
    target_distribution["record_count"]
    / len(modeling_data)
    * 100
)

target_distribution["expected_record_count"] = (
    target_distribution["target_class"]
    .map(EXPECTED_TARGET_COUNTS)
)

target_distribution["matches_expected"] = (
    target_distribution["record_count"]
    == target_distribution["expected_record_count"]
)

actual_target_counts = target_distribution.set_index(
    "target_class"
)["record_count"].to_dict()

validation_results = pd.DataFrame({
    "validation_check": [
        "Record count",
        "Total column count",
        "Predictor count",
        "Tracking identifier count",
        "Primary target count",
        "Missing tracking identifiers",
        "Duplicate tracking identifiers",
        "Unexpected direct-leakage columns",
        "Infinite numeric values",
        "Primary target class 0 count",
        "Primary target class 1 count",
    ],
    "expected": [
        EXPECTED_ROW_COUNT,
        EXPECTED_COLUMN_COUNT,
        EXPECTED_PREDICTOR_COUNT,
        1,
        1,
        0,
        0,
        "None",
        0,
        EXPECTED_TARGET_COUNTS[0],
        EXPECTED_TARGET_COUNTS[1],
    ],
    "actual": [
        modeling_data.shape[0],
        modeling_data.shape[1],
        len(predictor_columns),
        list(modeling_data.columns).count(
            TRACKING_IDENTIFIER_COLUMN
        ),
        list(modeling_data.columns).count(
            PRIMARY_TARGET_COLUMN
        ),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].isna().sum(),
        modeling_data[
            TRACKING_IDENTIFIER_COLUMN
        ].duplicated().sum(),
        (
            ", ".join(unexpected_direct_leakage_columns)
            if unexpected_direct_leakage_columns
            else "None"
        ),
        infinite_value_count,
        actual_target_counts[0],
        actual_target_counts[1],
    ],
})

validation_results["passed"] = [
    modeling_data.shape[0] == EXPECTED_ROW_COUNT,
    modeling_data.shape[1] == EXPECTED_COLUMN_COUNT,
    len(predictor_columns) == EXPECTED_PREDICTOR_COUNT,
    list(modeling_data.columns).count(
        TRACKING_IDENTIFIER_COLUMN
    ) == 1,
    list(modeling_data.columns).count(
        PRIMARY_TARGET_COLUMN
    ) == 1,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].isna().sum() == 0,
    modeling_data[
        TRACKING_IDENTIFIER_COLUMN
    ].duplicated().sum() == 0,
    len(unexpected_direct_leakage_columns) == 0,
    infinite_value_count == 0,
    actual_target_counts[0] == EXPECTED_TARGET_COUNTS[0],
    actual_target_counts[1] == EXPECTED_TARGET_COUNTS[1],
]

def format_validation_value(value):
    if isinstance(value, (bool, np.bool_)):
        return str(value)

    if isinstance(value, (int, np.integer)):
        return f"{value:,.0f}"

    if isinstance(value, (float, np.floating)):
        if pd.isna(value):
            return ""

        if value.is_integer():
            return f"{value:,.0f}"

        if value != 0 and abs(value) < 0.01:
            return f"{value:,.6f}".rstrip("0").rstrip(".")

        return f"{value:,.2f}"

    return str(value)


display(validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(target_distribution.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center"})
    .set_table_styles([{
        "selector": "th",
        "props": [("text-align", "center")],
    }])
    .format({
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "expected_record_count": "{:,.0f}",
    })
)

failed_validation_checks = validation_results.loc[
    ~validation_results["passed"],
    "validation_check",
].tolist()

if failed_validation_checks:
    raise AssertionError(
        "Phase 4 modeling export validation failed for: "
        + ", ".join(failed_validation_checks)
    )

print("\nAll Phase 4 modeling export validation checks passed.\n")

validation_check,expected,actual,passed
Record count,"34,403","34,403",True
Total column count,55,55,True
Predictor count,53,53,True
Tracking identifier count,1,1,True
Primary target count,1,1,True
Missing tracking identifiers,0,0,True
Duplicate tracking identifiers,0,0,True
Unexpected direct-leakage columns,None,None,True
Infinite numeric values,0,0,True
Primary target class 0 count,"32,499","32,499",True


target_class,record_count,percentage,expected_record_count,matches_expected
0,"32,499",94.47%,"32,499",True
1,"1,904",5.53%,"1,904",True



All Phase 4 modeling export validation checks passed.



## Modeling Setup and Data Validation

The Phase 5 modeling environment was configured using a consistent random state of `121`, reusable project paths, and the libraries required for preprocessing, classification, model evaluation, visualization, and model persistence.

Project paths are defined relative to the repository root so the notebook does not display user-specific local directories.

The engineered modeling dataset was loaded from the Parquet export, with the CSV retained as a fallback. The feature dictionary and original cleaned dataset were also confirmed to be available.

The modeling export contains:

* 34,403 records
* 55 total columns
* 53 leakage-safe predictors
* 1 tracking identifier
* 1 primary target

The `donor_unique_id` field contains no missing or duplicate values. No direct-leakage columns or infinite numeric values were found.

The primary target distribution remains unchanged:

| Target Class | Records | Percentage |
| ------------ | ------: | ---------: |
| 0            |  32,499 |     94.47% |
| 1            |   1,904 |      5.53% |

All Phase 4 modeling export validation checks passed. The severe class imbalance confirms that Phase 5 should use stratified splitting and evaluation metrics beyond accuracy.


In [4]:
# Define the primary target, tracking identifier, and predictor matrix
tracking_donor_ids = modeling_data[
    TRACKING_IDENTIFIER_COLUMN
].copy()

target_primary_donor_flag = modeling_data[
    PRIMARY_TARGET_COLUMN
].copy()

features_primary_model = modeling_data.drop(
    columns=[
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
    ]
).copy()

primary_model_objects_summary = pd.DataFrame({
    "object_name": [
        "tracking_donor_ids",
        "target_primary_donor_flag",
        "features_primary_model",
    ],
    "record_count": [
        tracking_donor_ids.shape[0],
        target_primary_donor_flag.shape[0],
        features_primary_model.shape[0],
    ],
    "column_count": [
        1,
        1,
        features_primary_model.shape[1],
    ],
})

display(primary_model_objects_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["object_name"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "record_count": "{:,.0f}",
        "column_count": "{:,.0f}",
    })
)

assert len(tracking_donor_ids) == len(modeling_data)
assert len(target_primary_donor_flag) == len(modeling_data)
assert len(features_primary_model) == len(modeling_data)

assert tracking_donor_ids.index.equals(
    target_primary_donor_flag.index
)
assert tracking_donor_ids.index.equals(
    features_primary_model.index
)

assert TRACKING_IDENTIFIER_COLUMN not in features_primary_model.columns
assert PRIMARY_TARGET_COLUMN not in features_primary_model.columns
assert features_primary_model.shape[1] == EXPECTED_PREDICTOR_COUNT

print(
    "\nPrimary modeling objects were created and validated successfully."
)

object_name,record_count,column_count
tracking_donor_ids,"34,403",1
target_primary_donor_flag,"34,403",1
features_primary_model,"34,403",53



Primary modeling objects were created and validated successfully.


In [5]:
# Build and validate predictor list
required_feature_dictionary_columns = {
    "feature_name",
    "leakage_status",
}

missing_feature_dictionary_columns = sorted(
    required_feature_dictionary_columns
    - set(feature_dictionary.columns)
)

if missing_feature_dictionary_columns:
    raise KeyError(
        "Missing required feature dictionary columns: "
        + ", ".join(missing_feature_dictionary_columns)
    )

feature_dictionary_leakage_status = (
    feature_dictionary["leakage_status"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

leakage_safe_predictor_columns = feature_dictionary.loc[
    feature_dictionary_leakage_status.eq("safe"),
    "feature_name",
].tolist()

non_safe_dictionary_features = set(
    feature_dictionary.loc[
        ~feature_dictionary_leakage_status.eq("safe"),
        "feature_name",
    ]
)

missing_safe_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(features_primary_model.columns)
)

unexpected_predictor_columns = sorted(
    set(features_primary_model.columns)
    - set(leakage_safe_predictor_columns)
)

duplicate_safe_predictors = sorted(
    pd.Series(leakage_safe_predictor_columns)[
        pd.Series(leakage_safe_predictor_columns).duplicated()
    ].unique()
)

duplicate_predictor_matrix_columns = sorted(
    features_primary_model.columns[
        features_primary_model.columns.duplicated()
    ].unique()
)

identifier_or_target_predictors = sorted(
    set(leakage_safe_predictor_columns).intersection({
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
    })
)

non_safe_predictors_included = sorted(
    set(leakage_safe_predictor_columns).intersection(
        non_safe_dictionary_features
    )
)

predictor_validation_results = pd.DataFrame({
    "validation_check": [
        "Safe predictors exist in modeling export",
        "No unexpected predictors in modeling matrix",
        "No duplicate names in safe predictor list",
        "No duplicate columns in predictor matrix",
        "No identifier or target included",
        "No excluded or timing-sensitive features included",
        "Final safe predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        0,
        "None",
        "None",
        EXPECTED_PREDICTOR_COUNT,
    ],
    "actual": [
        (
            ", ".join(missing_safe_predictors)
            if missing_safe_predictors
            else "None missing"
        ),
        (
            ", ".join(unexpected_predictor_columns)
            if unexpected_predictor_columns
            else "None"
        ),
        len(duplicate_safe_predictors),
        len(duplicate_predictor_matrix_columns),
        (
            ", ".join(identifier_or_target_predictors)
            if identifier_or_target_predictors
            else "None"
        ),
        (
            ", ".join(non_safe_predictors_included)
            if non_safe_predictors_included
            else "None"
        ),
        len(leakage_safe_predictor_columns),
    ],
})

predictor_validation_results["passed"] = [
    len(missing_safe_predictors) == 0,
    len(unexpected_predictor_columns) == 0,
    len(duplicate_safe_predictors) == 0,
    len(duplicate_predictor_matrix_columns) == 0,
    len(identifier_or_target_predictors) == 0,
    len(non_safe_predictors_included) == 0,
    len(leakage_safe_predictor_columns) == EXPECTED_PREDICTOR_COUNT,
]

display(predictor_validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_predictor_checks = predictor_validation_results.loc[
    ~predictor_validation_results["passed"],
    "validation_check",
].tolist()

if failed_predictor_checks:
    raise AssertionError(
        "Leakage-safe predictor validation failed for: "
        + ", ".join(failed_predictor_checks)
    )

features_primary_model = features_primary_model.loc[
    :,
    leakage_safe_predictor_columns,
].copy()

print(
    f"\nLeakage-safe predictor list validated with "
    f"{len(leakage_safe_predictor_columns):,} features.\n"
)

validation_check,expected,actual,passed
Safe predictors exist in modeling export,None missing,None missing,True
No unexpected predictors in modeling matrix,None,None,True
No duplicate names in safe predictor list,0,0,True
No duplicate columns in predictor matrix,0,0,True
No identifier or target included,None,None,True
No excluded or timing-sensitive features included,None,None,True
Final safe predictor count,53,53,True



Leakage-safe predictor list validated with 53 features.



In [6]:
# Recreate and validate feature set variants
safe_baseline_historical_features = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "is_alumnus_flag",
    "is_parent_flag",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

safe_aggregate_rfm_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "feature_past_5yr_max_donation",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

safe_trend_enhanced_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "feature_past_5yr_max_donation",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
]

full_leakage_safe_candidate_features = [
    "donor_age",
    "is_alumnus_flag",
    "is_parent_flag",
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
    "feature_max_donation_streak_past_5yr",
    "feature_intermittent_donor_flag",
    "feature_years_since_last_donation_past_5yr",
    "feature_donated_last_year_flag",
    "feature_donated_within_2_years_flag",
    "feature_lapsed_donor_flag",
    "feature_never_donated_past_5yr_flag",
    "feature_most_recent_positive_donation",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_previous_year_donation_zero_flag",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_log_past_5yr_total_donation",
    "feature_log_past_5yr_average_donation",
    "feature_log_past_5yr_max_donation",
    "feature_log_most_recent_positive_donation",
    "feature_age_group",
    "feature_age_decade",
    "feature_age_squared",
    "feature_gender_identity",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_type",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

feature_set_variants = {
    "Safe Baseline Historical Set": safe_baseline_historical_features,
    "Safe Aggregate RFM Set": safe_aggregate_rfm_features,
    "Safe Trend-Enhanced Set": safe_trend_enhanced_features,
    "Full Leakage-Safe Candidate Set": full_leakage_safe_candidate_features,
}

expected_feature_set_counts = {
    "Safe Baseline Historical Set": 10,
    "Safe Aggregate RFM Set": 8,
    "Safe Trend-Enhanced Set": 15,
    "Full Leakage-Safe Candidate Set": 53,
}

safe_dictionary_features = set(
    feature_dictionary.loc[
        feature_dictionary["leakage_status"]
        .astype(str)
        .str.strip()
        .str.casefold()
        .eq("safe"),
        "feature_name",
    ]
)

feature_set_validation_records = []
feature_set_validation_details = {}

for feature_set_name, feature_list in feature_set_variants.items():
    duplicate_features = sorted(
        pd.Series(feature_list)[
            pd.Series(feature_list).duplicated()
        ].unique()
    )

    missing_from_model = sorted(
        set(feature_list) - set(features_primary_model.columns)
    )

    missing_from_dictionary = sorted(
        set(feature_list) - set(feature_dictionary["feature_name"])
    )

    non_safe_features = sorted(
        set(feature_list) - safe_dictionary_features
    )

    forbidden_features = sorted(
        set(feature_list).intersection({
            TRACKING_IDENTIFIER_COLUMN,
            PRIMARY_TARGET_COLUMN,
        })
    )

    outside_full_safe_list = sorted(
        set(feature_list) - set(leakage_safe_predictor_columns)
    )

    expected_count = expected_feature_set_counts[feature_set_name]

    feature_set_passed = all([
        len(feature_list) == expected_count,
        len(duplicate_features) == 0,
        len(missing_from_model) == 0,
        len(missing_from_dictionary) == 0,
        len(non_safe_features) == 0,
        len(forbidden_features) == 0,
        len(outside_full_safe_list) == 0,
    ])

    feature_set_validation_records.append({
        "feature_set": feature_set_name,
        "expected_count": expected_count,
        "actual_count": len(feature_list),
        "duplicate_count": len(duplicate_features),
        "missing_model_count": len(missing_from_model),
        "missing_dictionary_count": len(missing_from_dictionary),
        "non_safe_count": len(non_safe_features),
        "forbidden_count": len(forbidden_features),
        "passed": feature_set_passed,
    })

    feature_set_validation_details[feature_set_name] = {
        "duplicates": duplicate_features,
        "missing_from_model": missing_from_model,
        "missing_from_dictionary": missing_from_dictionary,
        "non_safe_features": non_safe_features,
        "forbidden_features": forbidden_features,
        "outside_full_safe_list": outside_full_safe_list,
    }

feature_set_validation_results = pd.DataFrame(
    feature_set_validation_records
)

full_candidate_matches_safe_export = (
    set(full_leakage_safe_candidate_features)
    == set(leakage_safe_predictor_columns)
)

display(feature_set_validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["feature_set"],
        **{"text-align": "center"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected_count": "{:,.0f}",
        "actual_count": "{:,.0f}",
        "duplicate_count": "{:,.0f}",
        "missing_model_count": "{:,.0f}",
        "missing_dictionary_count": "{:,.0f}",
        "non_safe_count": "{:,.0f}",
        "forbidden_count": "{:,.0f}",
    })
)

failed_feature_sets = feature_set_validation_results.loc[
    ~feature_set_validation_results["passed"],
    "feature_set",
].tolist()

if failed_feature_sets:
    failed_details = {
        feature_set_name: feature_set_validation_details[feature_set_name]
        for feature_set_name in failed_feature_sets
    }

    raise AssertionError(
        "Feature-set validation failed: "
        + json.dumps(failed_details, indent=2)
    )

if not full_candidate_matches_safe_export:
    raise AssertionError(
        "The Full Leakage-Safe Candidate Set does not match "
        "the 53-feature safe predictor list."
    )

print(
    "\nAll leakage-safe feature-set variants were recreated "
    "and validated successfully."
)

feature_set,expected_count,actual_count,duplicate_count,missing_model_count,missing_dictionary_count,non_safe_count,forbidden_count,passed
Safe Baseline Historical Set,10,10,0,0,0,0,0,True
Safe Aggregate RFM Set,8,8,0,0,0,0,0,True
Safe Trend-Enhanced Set,15,15,0,0,0,0,0,True
Full Leakage-Safe Candidate Set,53,53,0,0,0,0,0,True



All leakage-safe feature-set variants were recreated and validated successfully.


In [7]:
# Classify predictors by primary data type and modeling role
numeric_continuous_features = [
    "donor_age",
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_median_donation",
    "feature_past_5yr_max_donation",
    "feature_past_5yr_min_donation",
    "feature_past_5yr_donation_std",
    "feature_past_5yr_active_average_donation",
    "feature_most_recent_positive_donation",
    "feature_recent_vs_older_donation_difference",
    "feature_recent_vs_older_donation_ratio",
    "feature_past_5yr_donation_trend_slope",
    "feature_last_year_donation_amount_change",
    "feature_past_5yr_donation_coefficient_of_variation",
    "feature_last_vs_previous_donation_ratio",
    "feature_log1p_last_vs_previous_donation_ratio",
    "feature_log_past_5yr_total_donation",
    "feature_log_past_5yr_average_donation",
    "feature_log_past_5yr_max_donation",
    "feature_log_most_recent_positive_donation",
    "feature_age_squared",
]

numeric_binary_features = [
    "is_alumnus_flag",
    "is_parent_flag",
    "feature_any_past_donation_flag",
    "feature_multiple_year_donor_flag",
    "feature_consistent_donor_flag",
    "feature_intermittent_donor_flag",
    "feature_donated_last_year_flag",
    "feature_donated_within_2_years_flag",
    "feature_lapsed_donor_flag",
    "feature_never_donated_past_5yr_flag",
    "feature_previous_year_donation_zero_flag",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

ordinal_discrete_numeric_features = [
    "feature_years_donated_past_5yr",
    "feature_past_5yr_donation_frequency_rate",
    "feature_max_donation_streak_past_5yr",
    "feature_years_since_last_donation_past_5yr",
]

categorical_features = [
    "feature_age_group",
    "feature_age_decade",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

structurally_missing_features = [
    "feature_recent_vs_older_donation_ratio",
]

near_constant_features = [
    "feature_past_5yr_min_donation",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

primary_feature_type_groups = {
    "Numeric continuous": numeric_continuous_features,
    "Numeric binary": numeric_binary_features,
    "Ordinal or discrete numeric": ordinal_discrete_numeric_features,
    "Categorical": categorical_features,
}

feature_to_primary_type = {
    feature_name: feature_type
    for feature_type, feature_list in primary_feature_type_groups.items()
    for feature_name in feature_list
}

special_consideration_map = {
    "feature_recent_vs_older_donation_ratio": (
        "Structural missingness; impute within the pipeline and "
        "consider a missingness indicator"
    ),
    "feature_years_since_last_donation_past_5yr": (
        "Value 6 is a sentinel for no donation in the five-year window"
    ),
    "feature_past_5yr_min_donation": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_address_type_other": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_postal_code_missing_flag": (
        "Near constant; compare retention and removal by model"
    ),
    "feature_last_vs_previous_donation_ratio": (
        "May contain extreme values when the previous donation is small"
    ),
    "feature_log1p_last_vs_previous_donation_ratio": (
        "Log-transformed version retained for model-specific comparison"
    ),
}

predictor_modeling_roles = pd.DataFrame({
    "feature_name": leakage_safe_predictor_columns,
})

predictor_modeling_roles["primary_data_type"] = (
    predictor_modeling_roles["feature_name"]
    .map(feature_to_primary_type)
)

predictor_modeling_roles["missing_count"] = (
    predictor_modeling_roles["feature_name"]
    .map(features_primary_model.isna().sum())
)

predictor_modeling_roles["missing_percentage"] = (
    predictor_modeling_roles["missing_count"]
    / len(features_primary_model)
    * 100
)

predictor_modeling_roles["requires_scaling_linear_model"] = (
    predictor_modeling_roles["primary_data_type"].isin([
        "Numeric continuous",
        "Ordinal or discrete numeric",
    ])
)

predictor_modeling_roles["requires_imputation"] = (
    predictor_modeling_roles["missing_count"] > 0
)

predictor_modeling_roles["requires_one_hot_encoding"] = (
    predictor_modeling_roles["primary_data_type"].eq("Categorical")
)

predictor_modeling_roles["structurally_missing"] = (
    predictor_modeling_roles["feature_name"].isin(
        structurally_missing_features
    )
)

predictor_modeling_roles["near_constant"] = (
    predictor_modeling_roles["feature_name"].isin(
        near_constant_features
    )
)

predictor_modeling_roles["special_consideration"] = (
    predictor_modeling_roles["feature_name"]
    .map(special_consideration_map)
    .fillna("None")
)

In [8]:
# Validate predictor classifications and summarize preprocessing roles
all_classified_features = [
    feature_name
    for feature_list in primary_feature_type_groups.values()
    for feature_name in feature_list
]

duplicate_classified_features = sorted(
    pd.Series(all_classified_features)[
        pd.Series(all_classified_features).duplicated()
    ].unique()
)

unclassified_safe_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(all_classified_features)
)

unexpected_classified_features = sorted(
    set(all_classified_features)
    - set(leakage_safe_predictor_columns)
)

numeric_type_mismatches = sorted([
    feature_name
    for feature_name in (
        numeric_continuous_features
        + numeric_binary_features
        + ordinal_discrete_numeric_features
    )
    if not pd.api.types.is_numeric_dtype(
        features_primary_model[feature_name]
    )
])

actual_missing_features = sorted(
    features_primary_model.columns[
        features_primary_model.isna().any()
    ].tolist()
)

structural_missingness_matches = (
    set(actual_missing_features)
    == set(structurally_missing_features)
)

near_constant_dominant_percentages = {
    feature_name: (
        features_primary_model[feature_name]
        .value_counts(dropna=False, normalize=True)
        .max()
        * 100
    )
    for feature_name in near_constant_features
}

near_constant_status_valid = all(
    dominant_percentage >= 99.50
    for dominant_percentage
    in near_constant_dominant_percentages.values()
)

predictor_classification_validation = pd.DataFrame({
    "validation_check": [
        "All safe predictors classified",
        "No unexpected features classified",
        "No feature assigned to multiple primary types",
        "All numeric groups contain numeric columns",
        "Structural missingness list matches observed missingness",
        "Near-constant features meet dominance threshold",
        "Final classified predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        "None",
        "Exact match",
        "At least 99.50%",
        EXPECTED_PREDICTOR_COUNT,
    ],
    "actual": [
        (
            ", ".join(unclassified_safe_predictors)
            if unclassified_safe_predictors
            else "None missing"
        ),
        (
            ", ".join(unexpected_classified_features)
            if unexpected_classified_features
            else "None"
        ),
        len(duplicate_classified_features),
        (
            ", ".join(numeric_type_mismatches)
            if numeric_type_mismatches
            else "None"
        ),
        (
            "Exact match"
            if structural_missingness_matches
            else ", ".join(actual_missing_features)
        ),
        (
            "All passed"
            if near_constant_status_valid
            else "Review required"
        ),
        len(all_classified_features),
    ],
})

predictor_classification_validation["passed"] = [
    len(unclassified_safe_predictors) == 0,
    len(unexpected_classified_features) == 0,
    len(duplicate_classified_features) == 0,
    len(numeric_type_mismatches) == 0,
    structural_missingness_matches,
    near_constant_status_valid,
    len(all_classified_features) == EXPECTED_PREDICTOR_COUNT,
]

preprocessing_role_summary = pd.DataFrame({
    "primary_data_type": [
        "Numeric continuous",
        "Numeric binary",
        "Ordinal or discrete numeric",
        "Categorical",
    ],
    "feature_count": [
        len(numeric_continuous_features),
        len(numeric_binary_features),
        len(ordinal_discrete_numeric_features),
        len(categorical_features),
    ],
    "scaling_for_linear_models": [
        "Required",
        "Not required",
        "Required",
        "Not applicable",
    ],
    "imputation": [
        "Median when missing",
        "Most frequent if missing",
        "Median if missing",
        "Most frequent if missing",
    ],
    "encoding": [
        "None",
        "None",
        "None",
        "One-hot encoding",
    ],
})

special_feature_summary = predictor_modeling_roles.loc[
    predictor_modeling_roles["special_consideration"].ne("None"),
    [
        "feature_name",
        "primary_data_type",
        "missing_count",
        "missing_percentage",
        "structurally_missing",
        "near_constant",
        "special_consideration",
    ],
].copy()

display(predictor_classification_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("\n")

display(preprocessing_role_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["primary_data_type"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "feature_count": "{:,.0f}",
    })
)

print("\n")

display(special_feature_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["feature_name", "special_consideration"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
        {
            "selector": "th.col6",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "missing_count": "{:,.0f}",
        "missing_percentage": "{:,.2f}%",
    })
)

failed_classification_checks = (
    predictor_classification_validation.loc[
        ~predictor_classification_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_classification_checks:
    raise AssertionError(
        "Predictor classification validation failed for: "
        + ", ".join(failed_classification_checks)
    )

print(
    "\nAll predictors were classified and preprocessing roles "
    "were documented successfully.\n"
)

validation_check,expected,actual,passed
All safe predictors classified,None missing,None missing,True
No unexpected features classified,None,None,True
No feature assigned to multiple primary types,0,0,True
All numeric groups contain numeric columns,None,None,True
Structural missingness list matches observed missingness,Exact match,Exact match,True
Near-constant features meet dominance threshold,At least 99.50%,All passed,True
Final classified predictor count,53,53,True


primary_data_type,feature_count,scaling_for_linear_models,imputation,encoding
Numeric continuous,26,Required,Median when missing,None
Numeric binary,19,Not required,Most frequent if missing,None
Ordinal or discrete numeric,4,Required,Median if missing,None
Categorical,4,Not applicable,Most frequent if missing,One-hot encoding


feature_name,primary_data_type,missing_count,missing_percentage,structurally_missing,near_constant,special_consideration
feature_past_5yr_min_donation,Numeric continuous,0,0.00%,False,True,Near constant; compare retention and removal by model
feature_years_since_last_donation_past_5yr,Ordinal or discrete numeric,0,0.00%,False,False,Value 6 is a sentinel for no donation in the five-year window
feature_recent_vs_older_donation_ratio,Numeric continuous,"28,497",82.83%,True,False,Structural missingness; impute within the pipeline and consider a missingness indicator
feature_last_vs_previous_donation_ratio,Numeric continuous,0,0.00%,False,False,May contain extreme values when the previous donation is small
feature_log1p_last_vs_previous_donation_ratio,Numeric continuous,0,0.00%,False,False,Log-transformed version retained for model-specific comparison
feature_postal_code_missing_flag,Numeric binary,0,0.00%,False,True,Near constant; compare retention and removal by model
feature_address_type_other,Numeric binary,0,0.00%,False,True,Near constant; compare retention and removal by model



All predictors were classified and preprocessing roles were documented successfully.



## Primary Modeling Data and Predictor Structure

The modeling dataset was separated into three explicitly named objects:

- `tracking_donor_ids` contains the donor identifiers used only for record alignment, joins, and final prediction outputs.
- `target_primary_donor_flag` contains the binary current-fiscal-year donation target.
- `features_primary_model` contains the 53 predictors used for modeling.

The tracking identifier and target were removed from the predictor matrix before any preprocessing or model training. All three objects contain 34,403 aligned records.

The feature dictionary was used as the source of truth to construct the leakage-safe predictor list. All 53 predictors were found in the modeling export, and the list contained no duplicate names, targets, identifiers, excluded fields, or timing-sensitive features.

Four leakage-safe feature-set variants were recreated for later model comparison:

| Feature Set                     | Feature Count |
| ------------------------------- | ------------: |
| Safe Baseline Historical Set    |            10 |
| Safe Aggregate RFM Set          |             8 |
| Safe Trend-Enhanced Set         |            15 |
| Full Leakage-Safe Candidate Set |            53 |

Each feature set was validated against both the modeling dataset and feature dictionary. All expected features were present, and no unsafe or forbidden fields were included.

The 53 predictors were then separated into mutually exclusive primary data-type groups:

| Primary Data Type           | Feature Count | Planned Preprocessing                                       |
| --------------------------- | ------------: | ----------------------------------------------------------- |
| Numeric continuous          |            26 | Median imputation when needed and scaling for linear models |
| Numeric binary              |            19 | Most-frequent imputation if needed; no scaling required     |
| Ordinal or discrete numeric |             4 | Median imputation if needed and scaling for linear models   |
| Categorical                 |             4 | Most-frequent imputation and one-hot encoding               |

Tree-based pipelines will use the same feature classifications but will generally omit numeric scaling.

Several predictors require additional consideration during modeling:

- `feature_recent_vs_older_donation_ratio` contains structural missingness for 28,497 records, or 82.83% of the dataset. Its imputation must occur inside the modeling pipeline, and a missingness indicator may be evaluated.
- `feature_years_since_last_donation_past_5yr` uses the value `6` as a sentinel for no donation during the five-year historical window.
- `feature_past_5yr_min_donation`, `feature_address_type_other`, and `feature_postal_code_missing_flag` are near-constant and will be evaluated through model-specific comparisons rather than removed automatically.
- The original and log-transformed donation-ratio features were retained.
- Ratio features may contain extreme but valid values when the comparison-period donation amount is zero or very small.

All predictors were successfully classified, and the resulting preprocessing roles provide the structure needed to build separate linear and tree-based modeling pipelines.


In [9]:
# Join and validate historical donor status benchmark target
BENCHMARK_TARGET_COLUMN = "donor_indicator_flag"

EXPECTED_BENCHMARK_TARGET_COUNTS = {
    0: 13_034,
    1: 21_369,
}

benchmark_target_source = pd.read_csv(
    CLEANED_DONOR_DATA_PATH,
    usecols=[
        TRACKING_IDENTIFIER_COLUMN,
        BENCHMARK_TARGET_COLUMN,
    ],
)

benchmark_target_join = modeling_data[
    [TRACKING_IDENTIFIER_COLUMN]
].merge(
    benchmark_target_source,
    on=TRACKING_IDENTIFIER_COLUMN,
    how="left",
    validate="one_to_one",
    indicator=True,
)

target_benchmark_donor_indicator_flag = benchmark_target_join[
    BENCHMARK_TARGET_COLUMN
].copy()

benchmark_target_distribution = (
    target_benchmark_donor_indicator_flag
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis("target_class")
    .reset_index(name="record_count")
)

benchmark_target_distribution["percentage"] = (
    benchmark_target_distribution["record_count"]
    / len(target_benchmark_donor_indicator_flag)
    * 100
)

benchmark_target_distribution["expected_record_count"] = (
    benchmark_target_distribution["target_class"]
    .map(EXPECTED_BENCHMARK_TARGET_COUNTS)
)

benchmark_target_distribution["matches_expected"] = (
    benchmark_target_distribution["record_count"]
    == benchmark_target_distribution["expected_record_count"]
)

actual_benchmark_target_counts = (
    benchmark_target_distribution
    .set_index("target_class")["record_count"]
    .to_dict()
)

benchmark_in_primary_feature_sets = sorted({
    feature_set_name
    for feature_set_name, feature_list in feature_set_variants.items()
    if BENCHMARK_TARGET_COLUMN in feature_list
})

benchmark_target_validation = pd.DataFrame({
    "validation_check": [
        "Benchmark source identifier missing count",
        "Benchmark source identifier duplicate count",
        "Join matched record count",
        "Joined row count",
        "Benchmark target missing count",
        "Benchmark target contains only 0 and 1",
        "Benchmark target class 0 count",
        "Benchmark target class 1 count",
        "Benchmark target excluded from primary predictor matrix",
        "Benchmark target excluded from safe predictor list",
        "Benchmark target excluded from primary feature sets",
    ],
    "expected": [
        0,
        0,
        EXPECTED_ROW_COUNT,
        EXPECTED_ROW_COUNT,
        0,
        "True",
        EXPECTED_BENCHMARK_TARGET_COUNTS[0],
        EXPECTED_BENCHMARK_TARGET_COUNTS[1],
        "True",
        "True",
        "True",
    ],
    "actual": [
        benchmark_target_source[
            TRACKING_IDENTIFIER_COLUMN
        ].isna().sum(),
        benchmark_target_source[
            TRACKING_IDENTIFIER_COLUMN
        ].duplicated().sum(),
        benchmark_target_join["_merge"].eq("both").sum(),
        len(benchmark_target_join),
        target_benchmark_donor_indicator_flag.isna().sum(),
        str(
            set(
                target_benchmark_donor_indicator_flag
                .dropna()
                .unique()
            ).issubset({0, 1})
        ),
        actual_benchmark_target_counts[0],
        actual_benchmark_target_counts[1],
        str(
            BENCHMARK_TARGET_COLUMN
            not in features_primary_model.columns
        ),
        str(
            BENCHMARK_TARGET_COLUMN
            not in leakage_safe_predictor_columns
        ),
        str(len(benchmark_in_primary_feature_sets) == 0),
    ],
})

benchmark_target_validation["passed"] = [
    benchmark_target_source[
        TRACKING_IDENTIFIER_COLUMN
    ].isna().sum() == 0,
    benchmark_target_source[
        TRACKING_IDENTIFIER_COLUMN
    ].duplicated().sum() == 0,
    benchmark_target_join["_merge"].eq("both").sum()
    == EXPECTED_ROW_COUNT,
    len(benchmark_target_join) == EXPECTED_ROW_COUNT,
    target_benchmark_donor_indicator_flag.isna().sum() == 0,
    set(
        target_benchmark_donor_indicator_flag
        .dropna()
        .unique()
    ).issubset({0, 1}),
    actual_benchmark_target_counts[0]
    == EXPECTED_BENCHMARK_TARGET_COUNTS[0],
    actual_benchmark_target_counts[1]
    == EXPECTED_BENCHMARK_TARGET_COUNTS[1],
    BENCHMARK_TARGET_COLUMN
    not in features_primary_model.columns,
    BENCHMARK_TARGET_COLUMN
    not in leakage_safe_predictor_columns,
    len(benchmark_in_primary_feature_sets) == 0,
]

display(benchmark_target_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(benchmark_target_distribution.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("text-align", "center"),
            ("padding", "8px 16px"),
        ],
    }])
    .format({
        "target_class": "{:,.0f}",
        "record_count": "{:,.0f}",
        "percentage": "{:,.2f}%",
        "expected_record_count": "{:,.0f}",
    })
)

failed_benchmark_target_checks = benchmark_target_validation.loc[
    ~benchmark_target_validation["passed"],
    "validation_check",
].tolist()

if failed_benchmark_target_checks:
    raise AssertionError(
        "Benchmark target validation failed for: "
        + ", ".join(failed_benchmark_target_checks)
    )

print(
    "\nHistorical donor-status benchmark target joined "
    "and validated successfully."
)

validation_check,expected,actual,passed
Benchmark source identifier missing count,0,0,True
Benchmark source identifier duplicate count,0,0,True
Join matched record count,"34,403","34,403",True
Joined row count,"34,403","34,403",True
Benchmark target missing count,0,0,True
Benchmark target contains only 0 and 1,True,True,True
Benchmark target class 0 count,"13,034","13,034",True
Benchmark target class 1 count,"21,369","21,369",True
Benchmark target excluded from primary predictor matrix,True,True,True
Benchmark target excluded from safe predictor list,True,True,True


target_class,record_count,percentage,expected_record_count,matches_expected
0,"13,034",37.89%,"13,034",True
1,"21,369",62.11%,"21,369",True



Historical donor-status benchmark target joined and validated successfully.


In [10]:
# Audit benchmark target against donation based construction rules
historical_donation_columns = [
    "last_fiscal_year_donation",
    "donation_2_fiscal_years_ago",
    "donation_3_fiscal_years_ago",
    "donation_4_fiscal_years_ago",
    "donation_5_fiscal_years_ago",
]

benchmark_audit_source = pd.read_csv(
    CLEANED_DONOR_DATA_PATH,
    usecols=[
        TRACKING_IDENTIFIER_COLUMN,
        BENCHMARK_TARGET_COLUMN,
        "cumulative_donation_amount",
        *historical_donation_columns,
    ],
)

benchmark_construction_audit = tracking_donor_ids.to_frame(
    name=TRACKING_IDENTIFIER_COLUMN
).merge(
    benchmark_audit_source,
    on=TRACKING_IDENTIFIER_COLUMN,
    how="left",
    validate="one_to_one",
)

benchmark_construction_audit["historical_positive_year_count"] = (
    benchmark_construction_audit[historical_donation_columns]
    .gt(0)
    .sum(axis=1)
)

benchmark_construction_audit["historical_5yr_total"] = (
    benchmark_construction_audit[historical_donation_columns]
    .sum(axis=1)
)

benchmark_actual_target = benchmark_construction_audit[
    BENCHMARK_TARGET_COLUMN
].astype("int8")

historical_missing_rows = (
    benchmark_construction_audit[historical_donation_columns]
    .isna()
    .any(axis=1)
    .sum()
)

cumulative_missing_rows = (
    benchmark_construction_audit[
        "cumulative_donation_amount"
    ]
    .isna()
    .sum()
)

benchmark_candidate_rules = {
    "Any positive historical donation": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(1).astype("int8"),
        historical_missing_rows,
    ),
    "Positive five-year historical total": (
        benchmark_construction_audit[
            "historical_5yr_total"
        ].gt(0).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 2 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(2).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 3 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(3).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in at least 4 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].ge(4).astype("int8"),
        historical_missing_rows,
    ),
    "Donated in all 5 historical years": (
        benchmark_construction_audit[
            "historical_positive_year_count"
        ].eq(5).astype("int8"),
        historical_missing_rows,
    ),
    "Positive cumulative donation": (
        benchmark_construction_audit[
            "cumulative_donation_amount"
        ].gt(0).astype("int8"),
        cumulative_missing_rows,
    ),
}

benchmark_reconstruction_records = []

for rule_name, (
    candidate_target,
    source_missing_rows,
) in benchmark_candidate_rules.items():

    disagreement_count = int(
        candidate_target.ne(benchmark_actual_target).sum()
    )

    benchmark_reconstruction_records.append({
        "candidate_rule": rule_name,
        "source_missing_rows": source_missing_rows,
        "predicted_positive_count": candidate_target.sum(),
        "predicted_positive_percentage": (
            candidate_target.mean() * 100
        ),
        "disagreement_count": disagreement_count,
        "agreement_percentage": (
            candidate_target.eq(benchmark_actual_target).mean()
            * 100
        ),
        "exact_reconstruction": (
            source_missing_rows == 0
            and disagreement_count == 0
        ),
    })

benchmark_reconstruction_results = pd.DataFrame(
    benchmark_reconstruction_records
)

exact_benchmark_reconstruction_rules = (
    benchmark_reconstruction_results.loc[
        benchmark_reconstruction_results[
            "exact_reconstruction"
        ],
        "candidate_rule",
    ].tolist()
)

display(benchmark_reconstruction_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["candidate_rule"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "source_missing_rows": "{:,.0f}",
        "predicted_positive_count": "{:,.0f}",
        "predicted_positive_percentage": "{:,.2f}%",
        "disagreement_count": "{:,.0f}",
        "agreement_percentage": "{:,.2f}%",
    })
)

if exact_benchmark_reconstruction_rules:
    print(
        "\nExact benchmark reconstruction rule(s): "
        + ", ".join(exact_benchmark_reconstruction_rules)
    )
else:
    print(
        "No tested donation-based rule exactly reconstructs "
        "donor_indicator_flag."
    )

candidate_rule,source_missing_rows,predicted_positive_count,predicted_positive_percentage,disagreement_count,agreement_percentage,exact_reconstruction
Any positive historical donation,0,"9,087",26.41%,"12,282",64.30%,False
Positive five-year historical total,0,"9,087",26.41%,"12,282",64.30%,False
Donated in at least 2 historical years,0,"1,894",5.51%,"19,475",43.39%,False
Donated in at least 3 historical years,0,407,1.18%,"20,962",39.07%,False
Donated in at least 4 historical years,0,60,0.17%,"21,309",38.06%,False
Donated in all 5 historical years,0,10,0.03%,"21,359",37.92%,False
Positive cumulative donation,0,"21,369",62.11%,0,100.00%,True



Exact benchmark reconstruction rule(s): Positive cumulative donation


In [11]:
# Define and validate conservative benchmark predictor set
conservative_benchmark_predictor_columns = [
    "donor_age",
    "is_alumnus_flag",
    "is_parent_flag",
    "feature_age_group",
    "feature_age_decade",
    "feature_age_squared",
    "feature_gender_identity",
    "feature_gender_missing_flag",
    "feature_gender_unknown_flag",
    "feature_preferred_address_type",
    "feature_preferred_address_missing_flag",
    "feature_is_business_address",
    "feature_is_home_address",
    "feature_is_campus_address",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

if exact_benchmark_reconstruction_rules:
    benchmark_predictor_strategy = (
        "Donation-based construction identified; historical donation "
        "features and derivatives excluded to avoid circular prediction"
    )
else:
    benchmark_predictor_strategy = (
        "Construction not verified; conservative non-donation "
        "predictor set used"
    )

benchmark_predictor_columns = (
    conservative_benchmark_predictor_columns.copy()
)

features_benchmark_model = features_primary_model[
    benchmark_predictor_columns
].copy()

benchmark_excluded_donation_predictors = sorted(
    set(leakage_safe_predictor_columns)
    - set(benchmark_predictor_columns)
)

missing_benchmark_predictors = sorted(
    set(benchmark_predictor_columns)
    - set(features_primary_model.columns)
)

non_safe_benchmark_predictors = sorted(
    set(benchmark_predictor_columns)
    - set(leakage_safe_predictor_columns)
)

duplicate_benchmark_predictors = sorted(
    pd.Series(benchmark_predictor_columns)[
        pd.Series(
            benchmark_predictor_columns
        ).duplicated()
    ].unique()
)

benchmark_forbidden_predictors = sorted(
    set(benchmark_predictor_columns).intersection({
        TRACKING_IDENTIFIER_COLUMN,
        PRIMARY_TARGET_COLUMN,
        BENCHMARK_TARGET_COLUMN,
        "cumulative_donation_amount",
        *historical_donation_columns,
    })
)

benchmark_predictor_validation = pd.DataFrame({
    "validation_check": [
        "Benchmark predictors exist in primary feature matrix",
        "Benchmark predictors are leakage-safe",
        "No duplicate benchmark predictors",
        "No identifier or target included",
        "Historical donation fields excluded",
        "Benchmark predictor count",
    ],
    "expected": [
        "None missing",
        "None",
        0,
        "None",
        "Excluded",
        len(conservative_benchmark_predictor_columns),
    ],
    "actual": [
        (
            ", ".join(missing_benchmark_predictors)
            if missing_benchmark_predictors
            else "None missing"
        ),
        (
            ", ".join(non_safe_benchmark_predictors)
            if non_safe_benchmark_predictors
            else "None"
        ),
        len(duplicate_benchmark_predictors),
        (
            ", ".join(benchmark_forbidden_predictors)
            if benchmark_forbidden_predictors
            else "None"
        ),
        (
            "Excluded"
            if not benchmark_forbidden_predictors
            else "Review required"
        ),
        len(benchmark_predictor_columns),
    ],
})

benchmark_predictor_validation["passed"] = [
    len(missing_benchmark_predictors) == 0,
    len(non_safe_benchmark_predictors) == 0,
    len(duplicate_benchmark_predictors) == 0,
    len(benchmark_forbidden_predictors) == 0,
    len(benchmark_forbidden_predictors) == 0,
    (
        len(benchmark_predictor_columns)
        == len(conservative_benchmark_predictor_columns)
    ),
]

benchmark_predictor_summary = pd.DataFrame({
    "predictor_strategy": [benchmark_predictor_strategy],
    "included_predictor_count": [
        len(benchmark_predictor_columns)
    ],
    "excluded_donation_predictor_count": [
        len(benchmark_excluded_donation_predictors)
    ],
})

display(benchmark_predictor_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

print("")

display(benchmark_predictor_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["predictor_strategy"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "included_predictor_count": "{:,.0f}",
        "excluded_donation_predictor_count": "{:,.0f}",
    })
)

failed_benchmark_predictor_checks = (
    benchmark_predictor_validation.loc[
        ~benchmark_predictor_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_benchmark_predictor_checks:
    raise AssertionError(
        "Benchmark predictor validation failed for: "
        + ", ".join(failed_benchmark_predictor_checks)
    )

print(
    "\nHistorical donor-status benchmark predictor strategy "
    "validated successfully."
)

validation_check,expected,actual,passed
Benchmark predictors exist in primary feature matrix,None missing,None missing,True
Benchmark predictors are leakage-safe,None,None,True
No duplicate benchmark predictors,0,0,True
No identifier or target included,None,None,True
Historical donation fields excluded,Excluded,Excluded,True
Benchmark predictor count,16,16,True


predictor_strategy,included_predictor_count,excluded_donation_predictor_count
Donation-based construction identified; historical donation features and derivatives excluded to avoid circular prediction,16,37



Historical donor-status benchmark predictor strategy validated successfully.


## Historical Donor-Status Benchmark Preparation

The original `donor_indicator_flag` was added as a separate benchmark target by joining it from the cleaned donor dataset using `donor_unique_id`.

The benchmark target validation confirmed:

- The join was one-to-one.
- The row count remained 34,403.
- No benchmark target values were missing.
- The target contained only binary values.
- The class distribution matched the Phase 4 documentation.
- The benchmark target was not added to `features_primary_model`, the leakage-safe predictor list, or any primary-model feature set.

The benchmark target distribution is:

| Target Class | Records | Percentage |
| ------------ | ------: | ---------: |
| 0            |  13,034 |     37.89% |
| 1            |  21,369 |     62.11% |

Because the construction of `donor_indicator_flag` was previously uncertain, several plausible donation-based rules were tested.

The five-year historical donation rules did not reproduce the target exactly. For example, classifying anyone with at least one positive donation during the five completed historical fiscal years as a donor produced only 64.30% agreement.

However, the following rule reproduced the benchmark target exactly:

`cumulative_donation_amount > 0`

This rule produced:

- 21,369 positive records
- 0 disagreements
- 100.00% agreement with `donor_indicator_flag`

This confirms that, within the available dataset, `donor_indicator_flag` functions as an indicator of whether cumulative donation is positive. It therefore represents historical donor status rather than the forward-looking current-fiscal-year outcome used by the primary model.

To avoid circular prediction, the benchmark model excludes:

- `cumulative_donation_amount`
- The five individual historical donation amount fields
- Donation aggregates
- Donation frequency and streak features
- Donation recency features
- Donation trend and trajectory features
- Log-transformed donation features

These fields were used only to investigate how the benchmark target was constructed and are excluded from the benchmark predictor set to avoid circular prediction.

A conservative benchmark predictor set containing 16 non-donation features was therefore defined. These consist of demographic, alumni and parent status, age representations, gender features, preferred-address features, and postal-code missingness.

The benchmark workflow will remain separate from the primary future-donation model because the two targets represent different prediction problems, use different predictor restrictions, and have different business interpretations.


In [12]:
# Define modeling experiment configuration
TEST_SIZE = 0.20
CV_FOLDS = 5
PRIMARY_SCORING = "average_precision"
DEFAULT_PROBABILITY_THRESHOLD = 0.50
OUTREACH_CAPACITY_PERCENTAGES = (0.01, 0.05, 0.10, 0.20)

modeling_experiment_config = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "n_jobs": N_JOBS,
    "primary_scoring": PRIMARY_SCORING,
    "default_probability_threshold": DEFAULT_PROBABILITY_THRESHOLD,
    "outreach_capacity_percentages": OUTREACH_CAPACITY_PERCENTAGES,
}

modeling_experiment_summary = pd.DataFrame({
    "setting": [
        "Random state",
        "Test-set proportion",
        "Cross-validation folds",
        "Parallel-processing jobs",
        "Primary scoring metric",
        "Default probability threshold",
        "Outreach-capacity percentages",
    ],
    "value": [
        f"{RANDOM_STATE:,}",
        f"{TEST_SIZE:.2%}",
        f"{CV_FOLDS:,}",
        str(N_JOBS),
        "PR-AUC (average_precision)",
        f"{DEFAULT_PROBABILITY_THRESHOLD:.2f}",
        ", ".join(
            f"{capacity:.0%}"
            for capacity in OUTREACH_CAPACITY_PERCENTAGES
        ),
    ],
})

display(modeling_experiment_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["setting"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
)

assert 0 < TEST_SIZE < 1
assert CV_FOLDS >= 2
assert 0 <= DEFAULT_PROBABILITY_THRESHOLD <= 1
assert all(
    0 < capacity <= 1
    for capacity in OUTREACH_CAPACITY_PERCENTAGES
)

print(
    "\nModeling experiment configuration defined successfully."
)

setting,value
Random state,121
Test-set proportion,20.00%
Cross-validation folds,5
Parallel-processing jobs,-1
Primary scoring metric,PR-AUC (average_precision)
Default probability threshold,0.50
Outreach-capacity percentages,"1%, 5%, 10%, 20%"



Modeling experiment configuration defined successfully.


In [13]:
# Create stratified development and test split
(
    features_primary_development,
    features_primary_test,
    target_primary_development,
    target_primary_test,
    tracking_donor_ids_development,
    tracking_donor_ids_test,
) = train_test_split(
    features_primary_model,
    target_primary_donor_flag,
    tracking_donor_ids,
    test_size=TEST_SIZE,
    stratify=target_primary_donor_flag,
    random_state=RANDOM_STATE,
)

overall_positive_rate = target_primary_donor_flag.mean() * 100
development_positive_rate = target_primary_development.mean() * 100
test_positive_rate = target_primary_test.mean() * 100

primary_split_summary = pd.DataFrame({
    "dataset_partition": [
        "Full dataset",
        "Development set",
        "Test set",
    ],
    "record_count": [
        len(features_primary_model),
        len(features_primary_development),
        len(features_primary_test),
    ],
    "class_0_count": [
        target_primary_donor_flag.eq(0).sum(),
        target_primary_development.eq(0).sum(),
        target_primary_test.eq(0).sum(),
    ],
    "class_1_count": [
        target_primary_donor_flag.eq(1).sum(),
        target_primary_development.eq(1).sum(),
        target_primary_test.eq(1).sum(),
    ],
    "class_1_percentage": [
        overall_positive_rate,
        development_positive_rate,
        test_positive_rate,
    ],
})

display(primary_split_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["dataset_partition"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "record_count": "{:,.0f}",
        "class_0_count": "{:,.0f}",
        "class_1_count": "{:,.0f}",
        "class_1_percentage": "{:,.2f}%",
    })
)

primary_split_validation = pd.DataFrame({
    "validation_check": [
        "Development objects aligned",
        "Test objects aligned",
        "Development and test IDs do not overlap",
        "All donor IDs retained",
        "Development and test rows sum to full dataset",
        "Development positive rate preserved",
        "Test positive rate preserved",
        "Test-set proportion matches configuration",
    ],
    "expected": [
        "True",
        "True",
        "True",
        EXPECTED_ROW_COUNT,
        EXPECTED_ROW_COUNT,
        f"{overall_positive_rate:.2f}%",
        f"{overall_positive_rate:.2f}%",
        f"{TEST_SIZE:.2%}",
    ],
    "actual": [
        str(
            features_primary_development.index.equals(
                target_primary_development.index
            )
            and features_primary_development.index.equals(
                tracking_donor_ids_development.index
            )
        ),
        str(
            features_primary_test.index.equals(
                target_primary_test.index
            )
            and features_primary_test.index.equals(
                tracking_donor_ids_test.index
            )
        ),
        str(
            set(tracking_donor_ids_development).isdisjoint(
                set(tracking_donor_ids_test)
            )
        ),
        (
            len(tracking_donor_ids_development)
            + len(tracking_donor_ids_test)
        ),
        (
            len(features_primary_development)
            + len(features_primary_test)
        ),
        f"{development_positive_rate:.2f}%",
        f"{test_positive_rate:.2f}%",
        f"{len(features_primary_test) / len(features_primary_model):.2%}",
    ],
})

primary_split_validation["passed"] = [
    (
        features_primary_development.index.equals(
            target_primary_development.index
        )
        and features_primary_development.index.equals(
            tracking_donor_ids_development.index
        )
    ),
    (
        features_primary_test.index.equals(
            target_primary_test.index
        )
        and features_primary_test.index.equals(
            tracking_donor_ids_test.index
        )
    ),
    set(tracking_donor_ids_development).isdisjoint(
        set(tracking_donor_ids_test)
    ),
    (
        len(tracking_donor_ids_development)
        + len(tracking_donor_ids_test)
        == EXPECTED_ROW_COUNT
    ),
    (
        len(features_primary_development)
        + len(features_primary_test)
        == EXPECTED_ROW_COUNT
    ),
    abs(
        development_positive_rate
        - overall_positive_rate
    ) < 0.01,
    abs(
        test_positive_rate
        - overall_positive_rate
    ) < 0.01,
    abs(
        len(features_primary_test)
        / len(features_primary_model)
        - TEST_SIZE
    ) < 0.001,
]

print("")

display(primary_split_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_primary_split_checks = primary_split_validation.loc[
    ~primary_split_validation["passed"],
    "validation_check",
].tolist()

if failed_primary_split_checks:
    raise AssertionError(
        "Primary development/test split validation failed for: "
        + ", ".join(failed_primary_split_checks)
    )

print(
    "\nStratified primary development and test split "
    "created and validated successfully."
)

dataset_partition,record_count,class_0_count,class_1_count,class_1_percentage
Full dataset,"34,403","32,499","1,904",5.53%
Development set,"27,522","25,999","1,523",5.53%
Test set,"6,881","6,500",381,5.54%


validation_check,expected,actual,passed
Development objects aligned,True,True,True
Test objects aligned,True,True,True
Development and test IDs do not overlap,True,True,True
All donor IDs retained,"34,403","34,403",True
Development and test rows sum to full dataset,"34,403","34,403",True
Development positive rate preserved,5.53%,5.53%,True
Test positive rate preserved,5.53%,5.54%,True
Test-set proportion matches configuration,20.00%,20.00%,True



Stratified primary development and test split created and validated successfully.


In [14]:
# Define and validate stratified cross validation strategy

primary_cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

primary_cv_splits = list(
    primary_cv.split(
        features_primary_development,
        target_primary_development,
    )
)

cv_fold_records = []

for fold_number, (train_indices, validation_indices) in enumerate(
    primary_cv_splits,
    start=1,
):
    target_fold_train = target_primary_development.iloc[
        train_indices
    ]
    target_fold_validation = target_primary_development.iloc[
        validation_indices
    ]

    cv_fold_records.append({
        "fold": fold_number,
        "train_count": len(train_indices),
        "train_class_1_count": target_fold_train.eq(1).sum(),
        "train_class_1_percentage": target_fold_train.mean() * 100,
        "validation_count": len(validation_indices),
        "validation_class_1_count": (
            target_fold_validation.eq(1).sum()
        ),
        "validation_class_1_percentage": (
            target_fold_validation.mean() * 100
        ),
    })

primary_cv_summary = pd.DataFrame(cv_fold_records)

display(primary_cv_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("text-align", "center"),
            ("padding", "8px 16px"),
        ],
    }])
    .format({
        "fold": "{:,.0f}",
        "train_count": "{:,.0f}",
        "train_class_1_count": "{:,.0f}",
        "train_class_1_percentage": "{:,.2f}%",
        "validation_count": "{:,.0f}",
        "validation_class_1_count": "{:,.0f}",
        "validation_class_1_percentage": "{:,.2f}%",
    })
)

all_validation_indices = np.concatenate([
    validation_indices
    for _, validation_indices in primary_cv_splits
])

cv_validation_results = pd.DataFrame({
    "validation_check": [
        "Cross-validation fold count",
        "All folds use development data only",
        "Training and validation indices do not overlap",
        "Every development record appears once in validation",
        "All development records retained across validation folds",
        "Validation class balance preserved",
    ],
    "expected": [
        CV_FOLDS,
        "True",
        "True",
        "True",
        len(features_primary_development),
        f"{target_primary_development.mean() * 100:.2f}%",
    ],
    "actual": [
        len(primary_cv_splits),
        str(
            max(all_validation_indices)
            < len(features_primary_development)
        ),
        str(all(
            set(train_indices).isdisjoint(validation_indices)
            for train_indices, validation_indices
            in primary_cv_splits
        )),
        str(
            len(np.unique(all_validation_indices))
            == len(features_primary_development)
        ),
        len(all_validation_indices),
        (
            f"{primary_cv_summary[
                'validation_class_1_percentage'
            ].mean():.2f}%"
        ),
    ],
})

cv_validation_results["passed"] = [
    len(primary_cv_splits) == CV_FOLDS,
    max(all_validation_indices)
    < len(features_primary_development),
    all(
        set(train_indices).isdisjoint(validation_indices)
        for train_indices, validation_indices
        in primary_cv_splits
    ),
    (
        len(np.unique(all_validation_indices))
        == len(features_primary_development)
    ),
    (
        len(all_validation_indices)
        == len(features_primary_development)
    ),
    abs(
        primary_cv_summary[
            "validation_class_1_percentage"
        ].mean()
        - target_primary_development.mean() * 100
    ) < 0.01,
]

print("")

display(cv_validation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_cv_checks = cv_validation_results.loc[
    ~cv_validation_results["passed"],
    "validation_check",
].tolist()

if failed_cv_checks:
    raise AssertionError(
        "Cross-validation strategy validation failed for: "
        + ", ".join(failed_cv_checks)
    )

print(
    "\nStratified cross-validation strategy created "
    "and validated successfully."
)

fold,train_count,train_class_1_count,train_class_1_percentage,validation_count,validation_class_1_count,validation_class_1_percentage
1,"22,017","1,218",5.53%,"5,505",305,5.54%
2,"22,017","1,218",5.53%,"5,505",305,5.54%
3,"22,018","1,219",5.54%,"5,504",304,5.52%
4,"22,018","1,219",5.54%,"5,504",304,5.52%
5,"22,018","1,218",5.53%,"5,504",305,5.54%


validation_check,expected,actual,passed
Cross-validation fold count,5,5,True
All folds use development data only,True,True,True
Training and validation indices do not overlap,True,True,True
Every development record appears once in validation,True,True,True
All development records retained across validation folds,"27,522","27,522",True
Validation class balance preserved,5.53%,5.53%,True



Stratified cross-validation strategy created and validated successfully.


## Modeling Experiment Configuration and Validation Strategy

The primary modeling workflow uses a consistent experiment configuration so that feature sets, model families, imbalance strategies, and hyperparameter settings can be compared under the same conditions.

The shared configuration is:

- Random state: `121`
- Final test-set proportion: 20%
- Cross-validation folds: 5
- Parallel processing: all available cores
- Primary scoring metric: PR-AUC using `average_precision`
- Default probability threshold: 0.50
- Outreach-capacity scenarios: 1%, 5%, 10%, and 20%

PR-AUC is used as the primary scoring metric because the positive class represents only about 5.53% of the dataset. Accuracy alone would not provide a meaningful measure of performance under this level of class imbalance.

The primary modeling data were then split into a development set and a final untouched test set using stratification.

| Dataset Partition | Records | Class 0 | Class 1 | Class 1 Percentage |
| ----------------- | ------: | ------: | ------: | -----------------: |
| Full dataset      |  34,403 |  32,499 |   1,904 |              5.53% |
| Development set   |  27,522 |  25,999 |   1,523 |              5.53% |
| Test set          |   6,881 |   6,500 |     381 |              5.54% |

The development set will be used for:

- Model training
- Cross-validation
- Feature-set comparisons
- Class-imbalance experiments
- Hyperparameter tuning
- Probability-threshold selection

The test set will remain untouched until the final model and operating threshold have been selected. Donor identifiers were split alongside the features and targets so record alignment is preserved.

A reproducible five-fold `StratifiedKFold` strategy was created using only the development set. The same fold assignments will be reused across modeling experiments to ensure that performance differences are caused by modeling choices rather than different validation samples.

Each validation fold contains approximately 5,504–5,505 records and preserves the positive-class rate at approximately 5.53%. Every development record appears exactly once in a validation fold, training and validation indices do not overlap, and the final test set is not involved in cross-validation.

This configuration establishes the fixed evaluation framework that will be used throughout primary model development.

In [15]:
# Build reusable classification evaluation functions
def calculate_classification_metrics(
    target_true,
    probability_positive,
    threshold=DEFAULT_PROBABILITY_THRESHOLD,
):
    target_true = pd.Series(target_true).reset_index(drop=True)
    probability_positive = np.asarray(probability_positive, dtype=float)

    if len(target_true) != len(probability_positive):
        raise ValueError(
            "Target values and predicted probabilities must have "
            "the same number of records."
        )

    if not set(target_true.dropna().unique()).issubset({0, 1}):
        raise ValueError("Target values must contain only 0 and 1.")

    if np.isnan(probability_positive).any():
        raise ValueError("Predicted probabilities contain missing values.")

    if np.any((probability_positive < 0) | (probability_positive > 1)):
        raise ValueError(
            "Predicted probabilities must be between 0 and 1."
        )

    if not 0 <= threshold <= 1:
        raise ValueError("Probability threshold must be between 0 and 1.")

    predicted_class = (
        probability_positive >= threshold
    ).astype("int8")

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(
            target_true,
            predicted_class,
            labels=[0, 1],
        ).ravel()
    )

    specificity = (
        true_negative / (true_negative + false_positive)
        if (true_negative + false_positive) > 0
        else np.nan
    )

    metrics = {
        "pr_auc_average_precision": average_precision_score(
            target_true,
            probability_positive,
        ),
        "roc_auc": roc_auc_score(
            target_true,
            probability_positive,
        ),
        "recall": recall_score(
            target_true,
            predicted_class,
            zero_division=0,
        ),
        "precision": precision_score(
            target_true,
            predicted_class,
            zero_division=0,
        ),
        "f1_score": f1_score(
            target_true,
            predicted_class,
            zero_division=0,
        ),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(
            target_true,
            predicted_class,
        ),
        "predicted_positive_rate": predicted_class.mean(),
        "brier_score": brier_score_loss(
            target_true,
            probability_positive,
        ),
        "probability_threshold": threshold,
    }

    return pd.Series(metrics, name="metric_value")

In [16]:
# Create consistent confusion matrix summary
def create_confusion_matrix_summary(
    target_true,
    probability_positive,
    threshold=DEFAULT_PROBABILITY_THRESHOLD,
):
    predicted_class = (
        np.asarray(probability_positive) >= threshold
    ).astype("int8")

    matrix = confusion_matrix(
        target_true,
        predicted_class,
        labels=[0, 1],
    )

    return pd.DataFrame(
        matrix,
        index=[
            "Actual Class 0",
            "Actual Class 1",
        ],
        columns=[
            "Predicted Class 0",
            "Predicted Class 1",
        ],
    )

In [17]:
# Create calibration summary from predicted probabilities
def create_calibration_summary(
    target_true,
    probability_positive,
    n_bins=10,
    strategy="quantile",
):
    observed_positive_rate, mean_predicted_probability = (
        calibration_curve(
            target_true,
            probability_positive,
            n_bins=n_bins,
            strategy=strategy,
        )
    )

    calibration_summary = pd.DataFrame({
        "bin": np.arange(
            1,
            len(mean_predicted_probability) + 1,
        ),
        "mean_predicted_probability": (
            mean_predicted_probability
        ),
        "observed_positive_rate": observed_positive_rate,
    })

    calibration_summary["calibration_difference"] = (
        calibration_summary["observed_positive_rate"]
        - calibration_summary["mean_predicted_probability"]
    )

    return calibration_summary

In [18]:
# Combine classification evaluation outputs into one reusable utility
def evaluate_classifier_predictions(
    target_true,
    probability_positive,
    threshold=DEFAULT_PROBABILITY_THRESHOLD,
    include_calibration=False,
    calibration_bins=10,
):
    evaluation_results = {
        "metrics": calculate_classification_metrics(
            target_true=target_true,
            probability_positive=probability_positive,
            threshold=threshold,
        ),
        "confusion_matrix": create_confusion_matrix_summary(
            target_true=target_true,
            probability_positive=probability_positive,
            threshold=threshold,
        ),
    }

    if include_calibration:
        evaluation_results["calibration"] = create_calibration_summary(
            target_true=target_true,
            probability_positive=probability_positive,
            n_bins=calibration_bins,
        )

    return evaluation_results

In [19]:
# Calculate outreach metrics at fixed campaign capacities
def calculate_outreach_capacity_metrics(
    target_true,
    probability_positive,
    capacities=OUTREACH_CAPACITY_PERCENTAGES,
):
    target_true = np.asarray(target_true, dtype=int)
    probability_positive = np.asarray(
        probability_positive,
        dtype=float,
    )

    if len(target_true) != len(probability_positive):
        raise ValueError(
            "Target values and predicted probabilities must have "
            "the same number of records."
        )

    if not set(np.unique(target_true)).issubset({0, 1}):
        raise ValueError("Target values must contain only 0 and 1.")

    if np.isnan(probability_positive).any():
        raise ValueError(
            "Predicted probabilities contain missing values."
        )

    if np.any(
        (probability_positive < 0)
        | (probability_positive > 1)
    ):
        raise ValueError(
            "Predicted probabilities must be between 0 and 1."
        )

    if not all(0 < capacity <= 1 for capacity in capacities):
        raise ValueError(
            "Outreach capacities must be greater than 0 "
            "and less than or equal to 1."
        )

    ranking_order = np.argsort(
        -probability_positive,
        kind="stable",
    )

    ranked_target = target_true[ranking_order]

    total_records = len(ranked_target)
    total_true_donors = ranked_target.sum()
    overall_donor_rate = ranked_target.mean()

    outreach_records = []

    for capacity in capacities:
        contacted_count = min(
            int(np.ceil(total_records * capacity)),
            total_records,
        )

        selected_target = ranked_target[:contacted_count]

        true_donors_captured = int(selected_target.sum())

        precision_at_capacity = (
            true_donors_captured / contacted_count
            if contacted_count > 0
            else np.nan
        )

        recall_at_capacity = (
            true_donors_captured / total_true_donors
            if total_true_donors > 0
            else np.nan
        )

        lift_over_random = (
            precision_at_capacity / overall_donor_rate
            if overall_donor_rate > 0
            else np.nan
        )

        expected_random_donors = (
            contacted_count * overall_donor_rate
        )

        outreach_records.append({
            "outreach_capacity": capacity,
            "people_contacted": contacted_count,
            "true_donors_captured": true_donors_captured,
            "precision_at_capacity": precision_at_capacity,
            "recall_at_capacity": recall_at_capacity,
            "donor_capture_rate": recall_at_capacity,
            "lift_over_random": lift_over_random,
            "cumulative_gain": recall_at_capacity,
            "expected_random_donors": expected_random_donors,
        })

    return pd.DataFrame(outreach_records)

In [20]:
# Create cumulative gains data across donor ranking
def create_cumulative_gains_data(
    target_true,
    probability_positive,
):
    target_true = np.asarray(target_true, dtype=int)
    probability_positive = np.asarray(
        probability_positive,
        dtype=float,
    )

    if len(target_true) != len(probability_positive):
        raise ValueError(
            "Target values and predicted probabilities must have "
            "the same number of records."
        )

    ranking_order = np.argsort(
        -probability_positive,
        kind="stable",
    )

    ranked_target = target_true[ranking_order]
    ranked_probability = probability_positive[ranking_order]

    total_records = len(ranked_target)
    total_true_donors = ranked_target.sum()

    cumulative_donors = np.cumsum(ranked_target)
    people_contacted = np.arange(1, total_records + 1)

    cumulative_gains_data = pd.DataFrame({
        "rank": people_contacted,
        "probability_positive": ranked_probability,
        "people_contacted": people_contacted,
        "outreach_percentage": (
            people_contacted / total_records
        ),
        "cumulative_true_donors": cumulative_donors,
        "cumulative_gain": (
            cumulative_donors / total_true_donors
            if total_true_donors > 0
            else np.nan
        ),
    })

    cumulative_gains_data["random_expected_gain"] = (
        cumulative_gains_data["outreach_percentage"]
    )

    return cumulative_gains_data

In [21]:
# Combine donor outreach ranking outputs into one reusable utility
def evaluate_outreach_ranking(
    target_true,
    probability_positive,
    capacities=OUTREACH_CAPACITY_PERCENTAGES,
    include_cumulative_gains=True,
):
    outreach_results = {
        "capacity_metrics": calculate_outreach_capacity_metrics(
            target_true=target_true,
            probability_positive=probability_positive,
            capacities=capacities,
        ),
    }

    if include_cumulative_gains:
        outreach_results["cumulative_gains"] = (
            create_cumulative_gains_data(
                target_true=target_true,
                probability_positive=probability_positive,
            )
        )

    return outreach_results

In [22]:
# Build reusable linear model preprocessing
def build_linear_preprocessor(feature_columns):
    feature_columns = list(feature_columns)

    structural_numeric = [
        feature
        for feature in structurally_missing_features
        if feature in feature_columns
    ]

    continuous_numeric = [
        feature
        for feature in numeric_continuous_features
        if feature in feature_columns
        and feature not in structural_numeric
    ]

    binary_numeric = [
        feature
        for feature in numeric_binary_features
        if feature in feature_columns
    ]

    ordinal_numeric = [
        feature
        for feature in ordinal_discrete_numeric_features
        if feature in feature_columns
    ]

    categorical = [
        feature
        for feature in categorical_features
        if feature in feature_columns
    ]

    transformers = []

    if continuous_numeric:
        transformers.append((
            "continuous_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            continuous_numeric,
        ))

    if structural_numeric:
        transformers.append((
            "structural_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
                ("scaler", StandardScaler()),
            ]),
            structural_numeric,
        ))

    if binary_numeric:
        transformers.append((
            "binary_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
            ]),
            binary_numeric,
        ))

    if ordinal_numeric:
        transformers.append((
            "ordinal_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            ordinal_numeric,
        ))

    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "one_hot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                    ),
                ),
            ]),
            categorical,
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

In [23]:
# Create linear preprocessor for leakage safe feature set
linear_preprocessor_full = build_linear_preprocessor(
    full_leakage_safe_candidate_features
)

In [24]:
# Validate linear model preprocessing structure
linear_preprocessing_summary = pd.DataFrame({
    "preprocessing_group": [
        "Continuous numeric",
        "Structurally missing numeric",
        "Binary numeric",
        "Ordinal or discrete numeric",
        "Categorical",
    ],
    "feature_count": [
        len([
            feature
            for feature in numeric_continuous_features
            if feature not in structurally_missing_features
        ]),
        len(structurally_missing_features),
        len(numeric_binary_features),
        len(ordinal_discrete_numeric_features),
        len(categorical_features),
    ],
    "imputation": [
        "Median",
        "Median + missingness indicator",
        "Most frequent",
        "Median",
        "Most frequent",
    ],
    "scaling": [
        "StandardScaler",
        "StandardScaler",
        "None",
        "StandardScaler",
        "None",
    ],
    "encoding": [
        "None",
        "None",
        "None",
        "None",
        "One-hot; unknown categories ignored",
    ],
})

display(linear_preprocessing_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["preprocessing_group"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "feature_count": "{:,.0f}",
    })
)

linear_preprocessing_feature_count = (
    linear_preprocessing_summary["feature_count"].sum()
)

assert linear_preprocessing_feature_count == EXPECTED_PREDICTOR_COUNT
assert not hasattr(linear_preprocessor_full, "transformers_")

print(
    "\nLinear-model preprocessing pipeline defined and "
    "validated without fitting."
)

preprocessing_group,feature_count,imputation,scaling,encoding
Continuous numeric,25,Median,StandardScaler,None
Structurally missing numeric,1,Median + missingness indicator,StandardScaler,None
Binary numeric,19,Most frequent,None,None
Ordinal or discrete numeric,4,Median,StandardScaler,None
Categorical,4,Most frequent,None,One-hot; unknown categories ignored



Linear-model preprocessing pipeline defined and validated without fitting.


In [25]:
# Build reusable tree model preprocessing
def build_tree_preprocessor(feature_columns):
    feature_columns = list(feature_columns)

    structural_numeric = [
        feature
        for feature in structurally_missing_features
        if feature in feature_columns
    ]

    continuous_numeric = [
        feature
        for feature in numeric_continuous_features
        if feature in feature_columns
        and feature not in structural_numeric
    ]

    binary_numeric = [
        feature
        for feature in numeric_binary_features
        if feature in feature_columns
    ]

    ordinal_numeric = [
        feature
        for feature in ordinal_discrete_numeric_features
        if feature in feature_columns
    ]

    categorical = [
        feature
        for feature in categorical_features
        if feature in feature_columns
    ]

    transformers = []

    if continuous_numeric:
        transformers.append((
            "continuous_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            continuous_numeric,
        ))

    if structural_numeric:
        transformers.append((
            "structural_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
            ]),
            structural_numeric,
        ))

    if binary_numeric:
        transformers.append((
            "binary_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
            ]),
            binary_numeric,
        ))

    if ordinal_numeric:
        transformers.append((
            "ordinal_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            ordinal_numeric,
        ))

    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "one_hot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )

In [26]:
# Create tree preprocessor for leakage safe feature set
tree_preprocessor_full = build_tree_preprocessor(
    full_leakage_safe_candidate_features
)

In [27]:
# Validate tree model preprocessing structure
tree_preprocessing_summary = pd.DataFrame({
    "preprocessing_group": [
        "Continuous numeric",
        "Structurally missing numeric",
        "Binary numeric",
        "Ordinal or discrete numeric",
        "Categorical",
    ],
    "feature_count": [
        len([
            feature
            for feature in numeric_continuous_features
            if feature not in structurally_missing_features
        ]),
        len(structurally_missing_features),
        len(numeric_binary_features),
        len(ordinal_discrete_numeric_features),
        len(categorical_features),
    ],
    "imputation": [
        "Median",
        "Median + missingness indicator",
        "Most frequent",
        "Median",
        "Most frequent",
    ],
    "scaling": [
        "None",
        "None",
        "None",
        "None",
        "None",
    ],
    "encoding": [
        "None",
        "None",
        "None",
        "None",
        "One-hot; dense output; unknown categories ignored",
    ],
})

display(tree_preprocessing_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["preprocessing_group"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "feature_count": "{:,.0f}",
    })
)

tree_preprocessing_feature_count = (
    tree_preprocessing_summary["feature_count"].sum()
)

assert tree_preprocessing_feature_count == EXPECTED_PREDICTOR_COUNT
assert not hasattr(tree_preprocessor_full, "transformers_")

print(
    "\nTree-model preprocessing pipeline defined and "
    "validated without fitting."
)

preprocessing_group,feature_count,imputation,scaling,encoding
Continuous numeric,25,Median,None,None
Structurally missing numeric,1,Median + missingness indicator,None,None
Binary numeric,19,Most frequent,None,None
Ordinal or discrete numeric,4,Median,None,None
Categorical,4,Most frequent,None,One-hot; dense output; unknown categories ignored



Tree-model preprocessing pipeline defined and validated without fitting.


# Evaluation Framework and Preprocessing Pipelines

Reusable evaluation utilities were created so every classification model can be assessed with the same set of metrics and output structure.

The classification evaluation functions calculate:

- PR-AUC using average precision
- ROC-AUC
- Recall
- Precision
- F1 score
- Specificity
- Balanced accuracy
- Predicted-positive rate
- Brier score
- Probability threshold used for classification

Accuracy is not used as the primary model-selection metric because the positive class represents only about 5.53% of the primary target.

Separate utilities were also created for:

- Confusion matrix summaries
- Calibration summaries using predicted-probability bins
- Combined classifier evaluation outputs

Calibration results will be used mainly for shortlisted or final models when probability quality becomes operationally important.

A separate donor-outreach evaluation framework was also created to measure how useful model rankings may be under limited campaign capacity. The outreach functions calculate:

- Precision at fixed outreach percentages
- Recall at fixed outreach percentages
- Donor capture rate
- Lift over random outreach
- Cumulative gains
- True donors captured
- Number of people contacted
- Expected donors captured through random selection

The planned outreach capacities are 1%, 5%, 10%, and 20% of the evaluated population.

Two preprocessing pipelines were then defined for the primary model workflow. Neither pipeline has been fitted yet, ensuring that imputation values, scaling parameters, missingness indicators, and categorical levels will be learned only within training folds.

For linear models, preprocessing uses:

- Median imputation and standard scaling for continuous numeric features
- Median imputation, a missingness indicator, and scaling for the structurally missing ratio feature
- Most-frequent imputation for binary features
- Median imputation and scaling for ordinal or discrete numeric features
- Most-frequent imputation and one-hot encoding for categorical features
- Safe handling of previously unseen categories

The linear preprocessing structure accounts for all 53 predictors:

| Preprocessing Group          | Feature Count |
| ---------------------------- | ------------: |
| Continuous numeric           |            25 |
| Structurally missing numeric |             1 |
| Binary numeric               |            19 |
| Ordinal or discrete numeric  |             4 |
| Categorical                  |             4 |

A separate tree-model preprocessing pipeline uses the same feature groups but omits scaling because tree-based models do not require standardized numeric inputs.

The tree pipeline also uses dense one-hot encoded categorical output so it remains compatible with the planned decision tree, random forest, and boosting implementations.

Both preprocessing pipelines remain unfitted and will later be placed inside complete scikit-learn model pipelines so all learned transformations occur only within the appropriate training data.


In [28]:
# Establish dummy classifier baselines
dummy_baseline_strategies = {
    "Most frequent": DummyClassifier(
        strategy="most_frequent",
    ),
    "Stratified random": DummyClassifier(
        strategy="stratified",
        random_state=RANDOM_STATE,
    ),
    "Prior probability": DummyClassifier(
        strategy="prior",
    ),
}

dummy_baseline_oof_probabilities = {}
dummy_baseline_records = []

for strategy_name, dummy_model in dummy_baseline_strategies.items():
    oof_probability_positive = cross_val_predict(
        dummy_model,
        features_primary_development,
        target_primary_development,
        cv=primary_cv_splits,
        method="predict_proba",
        n_jobs=N_JOBS,
    )[:, 1]

    cv_results = cross_validate(
        dummy_model,
        features_primary_development,
        target_primary_development,
        cv=primary_cv_splits,
        scoring=PRIMARY_SCORING,
        n_jobs=N_JOBS,
        return_train_score=False,
    )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    dummy_baseline_oof_probabilities[strategy_name] = (
        oof_probability_positive
    )

    dummy_baseline_records.append({
        "strategy": strategy_name,
        "cv_pr_auc_mean": cv_results["test_score"].mean(),
        "cv_pr_auc_std": cv_results["test_score"].std(),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "oof_roc_auc": evaluation_metrics["roc_auc"],
        "recall": evaluation_metrics["recall"],
        "precision": evaluation_metrics["precision"],
        "f1_score": evaluation_metrics["f1_score"],
        "specificity": evaluation_metrics["specificity"],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": evaluation_metrics[
            "predicted_positive_rate"
        ],
        "brier_score": evaluation_metrics["brier_score"],
    })

dummy_baseline_results = pd.DataFrame(
    dummy_baseline_records
)

display(dummy_baseline_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["strategy"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "oof_roc_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "specificity": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
    })
)

strategy,cv_pr_auc_mean,cv_pr_auc_std,oof_pr_auc,oof_roc_auc,recall,precision,f1_score,specificity,balanced_accuracy,predicted_positive_rate,brier_score
Most frequent,0.0553,0.0001,0.0553,0.5000,0.0000,0.0000,0.0000,1.0000,0.5000,0.00%,0.0553
Stratified random,0.0550,0.0002,0.0549,0.4956,0.0525,0.0478,0.0500,0.9387,0.4956,6.09%,0.1104
Prior probability,0.0553,0.0001,0.0553,0.4996,0.0000,0.0000,0.0000,1.0000,0.5000,0.00%,0.0523


In [29]:
# Validate dummy baseline evaluation
development_positive_rate = (
    target_primary_development.mean()
)

dummy_baseline_validation = pd.DataFrame({
    "validation_check": [
        "Dummy strategy count",
        "All out-of-fold probability arrays match development size",
        "No missing dummy probabilities",
        "All dummy probabilities are between 0 and 1",
        "Cross-validation folds reused",
        "Primary metric is PR-AUC",
        "Development positive rate",
    ],
    "expected": [
        3,
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "average_precision",
        f"{development_positive_rate:.2%}",
    ],
    "actual": [
        len(dummy_baseline_strategies),
        min(
            len(probabilities)
            for probabilities
            in dummy_baseline_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in dummy_baseline_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in dummy_baseline_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        PRIMARY_SCORING,
        f"{development_positive_rate:.2%}",
    ],
})

dummy_baseline_validation["passed"] = [
    len(dummy_baseline_strategies) == 3,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in dummy_baseline_oof_probabilities.values()
    ),
    all(
        not np.isnan(probabilities).any()
        for probabilities
        in dummy_baseline_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in dummy_baseline_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    PRIMARY_SCORING == "average_precision",
    np.isclose(
        development_positive_rate,
        target_primary_development.mean(),
    ),
]

display(dummy_baseline_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_dummy_baseline_checks = dummy_baseline_validation.loc[
    ~dummy_baseline_validation["passed"],
    "validation_check",
].tolist()

if failed_dummy_baseline_checks:
    raise AssertionError(
        "Dummy baseline validation failed for: "
        + ", ".join(failed_dummy_baseline_checks)
    )

print(
    "\nDummy classifier baselines evaluated and "
    "validated successfully."
)

validation_check,expected,actual,passed
Dummy strategy count,3,3,True
All out-of-fold probability arrays match development size,"27,522","27,522",True
No missing dummy probabilities,0,0,True
All dummy probabilities are between 0 and 1,True,True,True
Cross-validation folds reused,5,5,True
Primary metric is PR-AUC,average_precision,average_precision,True
Development positive rate,5.53%,5.53%,True



Dummy classifier baselines evaluated and validated successfully.


In [30]:
# Build scaled and unscaled logistic regression baseline pipelines
def build_unscaled_logistic_preprocessor(feature_columns):
    feature_columns = list(feature_columns)

    structural_numeric = [
        feature
        for feature in structurally_missing_features
        if feature in feature_columns
    ]

    continuous_numeric = [
        feature
        for feature in numeric_continuous_features
        if feature in feature_columns
        and feature not in structural_numeric
    ]

    binary_numeric = [
        feature
        for feature in numeric_binary_features
        if feature in feature_columns
    ]

    ordinal_numeric = [
        feature
        for feature in ordinal_discrete_numeric_features
        if feature in feature_columns
    ]

    categorical = [
        feature
        for feature in categorical_features
        if feature in feature_columns
    ]

    transformers = []

    if continuous_numeric:
        transformers.append((
            "continuous_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            continuous_numeric,
        ))

    if structural_numeric:
        transformers.append((
            "structural_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True,
                    ),
                ),
            ]),
            structural_numeric,
        ))

    if binary_numeric:
        transformers.append((
            "binary_numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
            ]),
            binary_numeric,
        ))

    if ordinal_numeric:
        transformers.append((
            "ordinal_numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            ordinal_numeric,
        ))

    if categorical:
        transformers.append((
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent"),
                ),
                (
                    "one_hot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                    ),
                ),
            ]),
            categorical,
        ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


logistic_baseline_scaled_pipeline = Pipeline([
    (
        "preprocessor",
        build_linear_preprocessor(
            safe_baseline_historical_features
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            penalty=None,
            solver="lbfgs",
            class_weight=None,
            max_iter=LOGISTIC_BASELINE_MAX_ITER,
        ),
    ),
])

logistic_baseline_unscaled_pipeline = Pipeline([
    (
        "preprocessor",
        build_unscaled_logistic_preprocessor(
            safe_baseline_historical_features
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            penalty=None,
            solver="lbfgs",
            class_weight=None,
            max_iter=LOGISTIC_BASELINE_MAX_ITER,
        ),
    ),
])

In [31]:
# Compare scaled and unscaled logistic regression baselines

logistic_baseline_pipelines = {
    "Scaled": logistic_baseline_scaled_pipeline,
    "Unscaled": logistic_baseline_unscaled_pipeline,
}

logistic_baseline_oof_probabilities = {}
logistic_baseline_cv_results = {}
logistic_baseline_records = []

dummy_reference_pr_auc = dummy_baseline_results.loc[
    dummy_baseline_results["strategy"].eq("Prior probability"),
    "cv_pr_auc_mean",
].iloc[0]

for preprocessing_name, pipeline in logistic_baseline_pipelines.items():
    evaluation_n_jobs = (
        N_JOBS
        if preprocessing_name == "Scaled"
        else 1
    )

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        cv_results = cross_validate(
            pipeline,
            features_primary_development[
                safe_baseline_historical_features
            ],
            target_primary_development,
            cv=primary_cv_splits,
            scoring=PRIMARY_SCORING,
            n_jobs=evaluation_n_jobs,
            return_estimator=True,
        )

        oof_probability_positive = cross_val_predict(
            pipeline,
            features_primary_development[
                safe_baseline_historical_features
            ],
            target_primary_development,
            cv=primary_cv_splits,
            method="predict_proba",
            n_jobs=evaluation_n_jobs,
        )[:, 1]

    convergence_warning_count = sum(
        issubclass(
            warning.category,
            ConvergenceWarning,
        )
        for warning in captured_warnings
    )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    fold_iterations = [
        int(
            estimator.named_steps[
                "classifier"
            ].n_iter_[0]
        )
        for estimator in cv_results["estimator"]
    ]

    logistic_baseline_oof_probabilities[
        preprocessing_name
    ] = oof_probability_positive

    logistic_baseline_cv_results[
        preprocessing_name
    ] = cv_results

    logistic_baseline_records.append({
        "preprocessing": preprocessing_name,
        "cv_pr_auc_mean": cv_results["test_score"].mean(),
        "cv_pr_auc_std": cv_results["test_score"].std(),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "oof_roc_auc": evaluation_metrics["roc_auc"],
        "recall": evaluation_metrics["recall"],
        "precision": evaluation_metrics["precision"],
        "f1_score": evaluation_metrics["f1_score"],
        "specificity": evaluation_metrics["specificity"],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": evaluation_metrics[
            "predicted_positive_rate"
        ],
        "brier_score": evaluation_metrics["brier_score"],
        "pr_auc_improvement_over_dummy": (
            cv_results["test_score"].mean()
            - dummy_reference_pr_auc
        ),
        "pr_auc_multiple_vs_dummy": (
            cv_results["test_score"].mean()
            / dummy_reference_pr_auc
        ),
        "maximum_fold_iterations": max(fold_iterations),
        "convergence_warning_count": (
            convergence_warning_count
        ),
        "all_folds_converged": (
            max(fold_iterations)
            < LOGISTIC_BASELINE_MAX_ITER
            and convergence_warning_count == 0
        ),
    })

logistic_baseline_results = pd.DataFrame(
    logistic_baseline_records
)

display(logistic_baseline_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["preprocessing"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "oof_roc_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "specificity": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_improvement_over_dummy": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "maximum_fold_iterations": "{:,.0f}",
        "convergence_warning_count": "{:,.0f}",
    })
)

preprocessing,cv_pr_auc_mean,cv_pr_auc_std,oof_pr_auc,oof_roc_auc,recall,precision,f1_score,specificity,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_improvement_over_dummy,pr_auc_multiple_vs_dummy,maximum_fold_iterations,convergence_warning_count,all_folds_converged
Scaled,0.0731,0.0050,0.0712,0.4994,0.0105,0.8421,0.0208,0.9999,0.5052,0.07%,0.0519,0.0178,1.32×,40,0,True
Unscaled,0.0742,0.0056,0.0726,0.4999,0.0112,0.8095,0.0220,0.9998,0.5055,0.08%,0.0518,0.0188,1.34×,"2,000",8,False


In [32]:
# Inspect convergence and preprocessing effects for scaled baseline
with warnings.catch_warnings(record=True) as captured_warnings:
    warnings.simplefilter("always", ConvergenceWarning)

    logistic_baseline_development_fit = clone(
        logistic_baseline_scaled_pipeline
    )

    logistic_baseline_development_fit.fit(
        features_primary_development[
            safe_baseline_historical_features
        ],
        target_primary_development,
    )

convergence_warnings = [
    warning
    for warning in captured_warnings
    if issubclass(
        warning.category,
        ConvergenceWarning,
    )
]

development_fit_iterations = int(
    logistic_baseline_development_fit.named_steps[
        "classifier"
    ].n_iter_[0]
)

transformed_baseline_feature_names = (
    logistic_baseline_development_fit.named_steps[
        "preprocessor"
    ].get_feature_names_out()
)

baseline_categorical_features = [
    feature
    for feature in safe_baseline_historical_features
    if feature in categorical_features
]

baseline_non_categorical_count = (
    len(safe_baseline_historical_features)
    - len(baseline_categorical_features)
)

encoded_categorical_output_count = (
    len(transformed_baseline_feature_names)
    - baseline_non_categorical_count
)

logistic_baseline_diagnostics = pd.DataFrame({
    "diagnostic": [
        "Source feature count",
        "Categorical source feature count",
        "One-hot encoded categorical output count",
        "Final transformed feature count",
        "Development fit iterations",
        "Maximum allowed iterations",
        "Convergence warning count",
        "Development fit converged",
    ],
    "value": [
        len(safe_baseline_historical_features),
        len(baseline_categorical_features),
        encoded_categorical_output_count,
        len(transformed_baseline_feature_names),
        development_fit_iterations,
        LOGISTIC_BASELINE_MAX_ITER,
        len(convergence_warnings),
        str(
            development_fit_iterations
            < LOGISTIC_BASELINE_MAX_ITER
            and len(convergence_warnings) == 0
        ),
    ],
})

display(logistic_baseline_diagnostics.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["diagnostic"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "value": format_validation_value,
    })
)

diagnostic,value
Source feature count,10
Categorical source feature count,2
One-hot encoded categorical output count,8
Final transformed feature count,16
Development fit iterations,35
Maximum allowed iterations,"2,000"
Convergence warning count,0
Development fit converged,True


In [33]:
# Validate standard logistic regression baseline
logistic_baseline_probability_positive = (
    logistic_baseline_oof_probabilities["Scaled"]
)

logistic_baseline_confusion_matrix = (
    create_confusion_matrix_summary(
        target_true=target_primary_development,
        probability_positive=(
            logistic_baseline_probability_positive
        ),
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )
)

display(logistic_baseline_confusion_matrix.style
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_table_styles([{
        "selector": "th",
        "props": [
            ("text-align", "center"),
            ("padding", "8px 16px"),
        ],
    }])
    .format("{:,.0f}")
)

scaled_logistic_result = logistic_baseline_results.loc[
    logistic_baseline_results["preprocessing"].eq("Scaled")
].iloc[0]

logistic_baseline_validation = pd.DataFrame({
    "validation_check": [
        "Safe baseline feature count",
        "Only safe baseline features used",
        "Out-of-fold predictions match development size",
        "No missing out-of-fold probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "Unweighted classifier used",
        "All scaled CV folds converged",
        "Development fit converged",
        "Scaled model exceeds dummy PR-AUC",
        "Final test set remained unused",
    ],
    "expected": [
        10,
        "True",
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(safe_baseline_historical_features),
        str(
            set(safe_baseline_historical_features)
            .issubset(leakage_safe_predictor_columns)
        ),
        len(logistic_baseline_probability_positive),
        np.isnan(
            logistic_baseline_probability_positive
        ).sum(),
        str(np.all(
            (logistic_baseline_probability_positive >= 0)
            & (logistic_baseline_probability_positive <= 1)
        )),
        len(primary_cv_splits),
        str(
            logistic_baseline_scaled_pipeline.named_steps[
                "classifier"
            ].class_weight is None
        ),
        str(
            scaled_logistic_result["all_folds_converged"]
        ),
        str(
            development_fit_iterations
            < LOGISTIC_BASELINE_MAX_ITER
            and len(convergence_warnings) == 0
        ),
        str(
            scaled_logistic_result["cv_pr_auc_mean"]
            > dummy_reference_pr_auc
        ),
        "True",
    ],
})

logistic_baseline_validation["passed"] = [
    len(safe_baseline_historical_features) == 10,
    set(safe_baseline_historical_features).issubset(
        leakage_safe_predictor_columns
    ),
    (
        len(logistic_baseline_probability_positive)
        == len(target_primary_development)
    ),
    not np.isnan(
        logistic_baseline_probability_positive
    ).any(),
    np.all(
        (logistic_baseline_probability_positive >= 0)
        & (logistic_baseline_probability_positive <= 1)
    ),
    len(primary_cv_splits) == CV_FOLDS,
    (
        logistic_baseline_scaled_pipeline.named_steps[
            "classifier"
        ].class_weight is None
    ),
    bool(
        scaled_logistic_result["all_folds_converged"]
    ),
    (
        development_fit_iterations
        < LOGISTIC_BASELINE_MAX_ITER
        and len(convergence_warnings) == 0
    ),
    (
        scaled_logistic_result["cv_pr_auc_mean"]
        > dummy_reference_pr_auc
    ),
    True,
]

print("")

display(logistic_baseline_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_logistic_baseline_checks = (
    logistic_baseline_validation.loc[
        ~logistic_baseline_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_logistic_baseline_checks:
    raise AssertionError(
        "Logistic regression baseline validation failed for: "
        + ", ".join(failed_logistic_baseline_checks)
    )

print(
    "\nStandard logistic regression baseline evaluated "
    "and validated successfully."
)

,Predicted Class 0,Predicted Class 1
Actual Class 0,"25,996",3
Actual Class 1,"1,507",16


validation_check,expected,actual,passed
Safe baseline feature count,10,10,True
Only safe baseline features used,True,True,True
Out-of-fold predictions match development size,"27,522","27,522",True
No missing out-of-fold probabilities,0,0,True
All probabilities between 0 and 1,True,True,True
Same cross-validation folds reused,5,5,True
Unweighted classifier used,True,True,True
All scaled CV folds converged,True,True,True
Development fit converged,True,True,True
Scaled model exceeds dummy PR-AUC,True,True,True



Standard logistic regression baseline evaluated and validated successfully.


# Logistic Regression Baseline Notes

The standard logistic regression baseline showed some predictive signal, with a cross-validated PR-AUC of approximately `0.0731` compared with the dummy baseline of about `0.0553`.

However, several limitations should be kept in mind:

- The default `0.50` probability threshold is extremely conservative and results in very few positive predictions.
- Recall is very low because only a small fraction of actual donors are classified as positive at the default threshold.
- The high precision value should not be overinterpreted because it is based on very few predicted-positive records.
- The current model is intentionally unweighted, so class-imbalance strategies have not yet been tested.
- Only the 10-feature Safe Baseline Historical Set is being used at this stage.
- Regularization has not yet been evaluated.
- Probability calibration has not yet been assessed.
- Outreach-capacity performance, such as donor capture within the top 1%, 5%, 10%, or 20% of ranked records, has not yet been evaluated.
- The pooled out-of-fold ROC-AUC is close to `0.50`, so later comparisons should also examine fold-level ROC-AUC alongside PR-AUC.
- The unscaled logistic regression diagnostic failed to converge within 2,000 iterations, while the scaled version converged quickly with no warnings. This supports retaining scaling for logistic regression.

These issues are expected at this baseline stage and will be addressed later in the notebook.

In [34]:
# Define regularized logistic regression configurations
regularized_logistic_configurations = {
    "L2": {
        "penalty": "l2",
        "solver": "lbfgs",
        "class_weight": None,
        "tol": 1e-4,
    },
    "L2 Balanced": {
        "penalty": "l2",
        "solver": "lbfgs",
        "class_weight": "balanced",
        "tol": 1e-4,
    },
    "L1": {
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": None,
        "tol": 1e-4,
    },
    "L1 Balanced": {
        "penalty": "l1",
        "solver": "liblinear",
        "class_weight": "balanced",
        "tol": 1e-4,
    },
    "Elastic Net": {
        "penalty": "elasticnet",
        "solver": "saga",
        "class_weight": None,
        "l1_ratio": 0.50,
        "tol": 1e-3,
    },
    "Elastic Net Balanced": {
        "penalty": "elasticnet",
        "solver": "saga",
        "class_weight": "balanced",
        "l1_ratio": 0.50,
        "tol": 1e-3,
    },
}

regularized_logistic_pipelines = {}

for model_name, configuration in regularized_logistic_configurations.items():
    classifier_parameters = {
        "penalty": configuration["penalty"],
        "solver": configuration["solver"],
        "class_weight": configuration["class_weight"],
        "C": 1.0,
        "tol": configuration["tol"],
        "max_iter": REGULARIZED_LOGISTIC_MAX_ITER,
        "random_state": RANDOM_STATE,
    }

    if configuration["penalty"] == "elasticnet":
        classifier_parameters["l1_ratio"] = configuration[
            "l1_ratio"
        ]

    regularized_logistic_pipelines[model_name] = Pipeline([
        (
            "preprocessor",
            build_linear_preprocessor(
                safe_baseline_historical_features
            ),
        ),
        (
            "classifier",
            LogisticRegression(**classifier_parameters),
        ),
    ])

features_regularized_logistic_development = (
    features_primary_development[
        safe_baseline_historical_features
    ].copy()
)

In [35]:
# Define reusable parallel fitting functions
def evaluate_regularized_logistic_fold(
    model_name,
    pipeline,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = clone(pipeline)

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        fold_pipeline.fit(
            features_regularized_logistic_development.iloc[
                train_indices
            ],
            target_primary_development.iloc[
                train_indices
            ],
        )

    fold_probability_positive = fold_pipeline.predict_proba(
        features_regularized_logistic_development.iloc[
            validation_indices
        ]
    )[:, 1]

    fold_target = target_primary_development.iloc[
        validation_indices
    ]

    classifier = fold_pipeline.named_steps[
        "classifier"
    ]

    iteration_count = int(classifier.n_iter_[0])

    convergence_warning_count = sum(
        issubclass(
            warning.category,
            ConvergenceWarning,
        )
        for warning in captured_warnings
    )

    return {
        "model": model_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": fold_probability_positive,
        "pr_auc": average_precision_score(
            fold_target,
            fold_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_target,
            fold_probability_positive,
        ),
        "iterations": iteration_count,
        "convergence_warning_count": (
            convergence_warning_count
        ),
        "converged": (
            iteration_count < classifier.max_iter
            and convergence_warning_count == 0
        ),
    }


def fit_regularized_logistic_development(
    model_name,
    pipeline,
):
    development_fit = clone(pipeline)

    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter(
            "always",
            ConvergenceWarning,
        )

        development_fit.fit(
            features_regularized_logistic_development,
            target_primary_development,
        )

    classifier = development_fit.named_steps[
        "classifier"
    ]

    iteration_count = int(classifier.n_iter_[0])

    convergence_warning_count = sum(
        issubclass(
            warning.category,
            ConvergenceWarning,
        )
        for warning in captured_warnings
    )

    coefficient_values = classifier.coef_[0]

    return {
        "model": model_name,
        "development_fit": development_fit,
        "iterations": iteration_count,
        "convergence_warning_count": (
            convergence_warning_count
        ),
        "converged": (
            iteration_count < classifier.max_iter
            and convergence_warning_count == 0
        ),
        "nonzero_coefficient_count": np.count_nonzero(
            np.abs(coefficient_values) > 1e-8
        ),
        "total_coefficient_count": len(
            coefficient_values
        ),
    }

In [36]:
# Evaluate regularized logistic regression configurations in parallel

cv_tasks = [
    (
        model_name,
        pipeline,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name, pipeline
    in regularized_logistic_pipelines.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    cv_parallel_results = joblib.Parallel(
        n_jobs=REGULARIZED_LOGISTIC_N_JOBS,
    )(
        joblib.delayed(
            evaluate_regularized_logistic_fold
        )(
            model_name,
            pipeline,
            fold_number,
            train_indices,
            validation_indices,
        )
        for (
            model_name,
            pipeline,
            fold_number,
            train_indices,
            validation_indices,
        ) in cv_tasks
    )

    development_parallel_results = joblib.Parallel(
        n_jobs=REGULARIZED_LOGISTIC_N_JOBS,
    )(
        joblib.delayed(
            fit_regularized_logistic_development
        )(
            model_name,
            pipeline,
        )
        for model_name, pipeline
        in regularized_logistic_pipelines.items()
    )

development_results_by_model = {
    result["model"]: result
    for result in development_parallel_results
}

regularized_logistic_records = []
regularized_logistic_oof_probabilities = {}
regularized_logistic_development_fits = {}

for model_name in regularized_logistic_pipelines:
    model_fold_results = sorted(
        [
            result
            for result in cv_parallel_results
            if result["model"] == model_name
        ],
        key=lambda result: result["fold"],
    )

    oof_probability_positive = np.full(
        len(target_primary_development),
        np.nan,
        dtype=float,
    )

    for fold_result in model_fold_results:
        oof_probability_positive[
            fold_result["validation_indices"]
        ] = fold_result["probability_positive"]

    if np.isnan(oof_probability_positive).any():
        raise AssertionError(
            f"Incomplete OOF probabilities for {model_name}."
        )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    development_result = (
        development_results_by_model[model_name]
    )

    configuration = (
        regularized_logistic_configurations[
            model_name
        ]
    )

    all_cv_folds_converged = all(
        result["converged"]
        for result in model_fold_results
    )

    development_fit_converged = (
        development_result["converged"]
    )

    regularized_logistic_oof_probabilities[
        model_name
    ] = oof_probability_positive

    regularized_logistic_development_fits[
        model_name
    ] = development_result[
        "development_fit"
    ]

    regularized_logistic_records.append({
        "model": model_name,
        "penalty": configuration["penalty"],
        "solver": configuration["solver"],
        "class_weight": (
            configuration["class_weight"]
            or "None"
        ),
        "cv_pr_auc_mean": np.mean([
            result["pr_auc"]
            for result in model_fold_results
        ]),
        "cv_pr_auc_std": np.std([
            result["pr_auc"]
            for result in model_fold_results
        ]),
        "cv_roc_auc_mean": np.mean([
            result["roc_auc"]
            for result in model_fold_results
        ]),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "recall": evaluation_metrics["recall"],
        "precision": evaluation_metrics["precision"],
        "f1_score": evaluation_metrics["f1_score"],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": evaluation_metrics[
            "predicted_positive_rate"
        ],
        "brier_score": evaluation_metrics[
            "brier_score"
        ],
        "pr_auc_multiple_vs_dummy": (
            np.mean([
                result["pr_auc"]
                for result in model_fold_results
            ])
            / dummy_reference_pr_auc
        ),
        "pr_auc_improvement_vs_standard": (
            np.mean([
                result["pr_auc"]
                for result in model_fold_results
            ])
            - scaled_logistic_result[
                "cv_pr_auc_mean"
            ]
        ),
        "maximum_fold_iterations": max(
            result["iterations"]
            for result in model_fold_results
        ),
        "cv_convergence_warning_count": sum(
            result["convergence_warning_count"]
            for result in model_fold_results
        ),
        "all_cv_folds_converged": (
            all_cv_folds_converged
        ),
        "development_fit_converged": (
            development_fit_converged
        ),
        "eligible_for_selection": (
            all_cv_folds_converged
            and development_fit_converged
        ),
        "nonzero_coefficient_count": (
            development_result[
                "nonzero_coefficient_count"
            ]
        ),
        "total_coefficient_count": (
            development_result[
                "total_coefficient_count"
            ]
        ),
    })

regularized_logistic_results = pd.DataFrame(
    regularized_logistic_records
).sort_values(
    "cv_pr_auc_mean",
    ascending=False,
).reset_index(drop=True)

display(regularized_logistic_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["model"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_improvement_vs_standard": "{:+.4f}",
        "maximum_fold_iterations": "{:,.0f}",
        "cv_convergence_warning_count": "{:,.0f}",
        "nonzero_coefficient_count": "{:,.0f}",
        "total_coefficient_count": "{:,.0f}",
    })
)

model,penalty,solver,class_weight,cv_pr_auc_mean,cv_pr_auc_std,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_improvement_vs_standard,maximum_fold_iterations,cv_convergence_warning_count,all_cv_folds_converged,development_fit_converged,eligible_for_selection,nonzero_coefficient_count,total_coefficient_count
L1,l1,liblinear,None,0.0743,0.0053,0.5032,0.0725,0.0112,0.9444,0.0221,0.5056,0.07%,0.0518,1.34×,+0.0011,21,0,True,True,True,12,16
L1 Balanced,l1,liblinear,balanced,0.0738,0.0051,0.5001,0.0724,0.3565,0.0559,0.0966,0.5019,35.30%,0.2468,1.33×,+0.0006,"2,000",4,False,False,False,14,16
L2 Balanced,l2,lbfgs,balanced,0.0727,0.0053,0.4996,0.0711,0.3592,0.0559,0.0968,0.5020,35.55%,0.2471,1.31×,-0.0004,51,0,True,True,True,16,16
L2,l2,lbfgs,None,0.0725,0.0050,0.5006,0.0707,0.0098,0.8824,0.0195,0.5049,0.06%,0.0518,1.31×,-0.0007,37,0,True,True,True,16,16
Elastic Net Balanced,elasticnet,saga,balanced,0.0720,0.0053,0.5007,0.0710,0.3697,0.0562,0.0975,0.5028,36.43%,0.2473,1.30×,-0.0011,817,0,True,True,True,15,16
Elastic Net,elasticnet,saga,None,0.0705,0.0054,0.5020,0.0696,0.0098,0.9375,0.0195,0.5049,0.06%,0.0518,1.27×,-0.0027,300,0,True,True,True,16,16


In [37]:
# Validate regularized logistic regression experiment
eligible_regularized_logistic_results = (
    regularized_logistic_results.loc[
        regularized_logistic_results[
            "eligible_for_selection"
        ]
    ].copy()
)

if eligible_regularized_logistic_results.empty:
    raise AssertionError(
        "No regularized logistic regression configuration "
        "completed with valid convergence."
    )

best_regularized_logistic_model = (
    eligible_regularized_logistic_results
    .sort_values(
        "cv_pr_auc_mean",
        ascending=False,
    )
    .iloc[0]["model"]
)

nonconverged_regularized_models = (
    regularized_logistic_results.loc[
        ~regularized_logistic_results[
            "eligible_for_selection"
        ],
        "model",
    ].tolist()
)

regularized_logistic_validation = pd.DataFrame({
    "validation_check": [
        "Regularized configuration count",
        "Cross-validation fit count",
        "Development fit count",
        "Safe baseline feature count",
        "All configurations produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "L2 configurations evaluated",
        "L1 configurations evaluated",
        "Elastic-net configurations evaluated",
        "Class-weighted configurations evaluated",
        "Convergence status recorded for every configuration",
        "At least one configuration eligible for selection",
        "Non-converged configurations excluded from selection",
        "Best eligible model exceeds dummy PR-AUC",
        "Final test set remained unused",
    ],
    "expected": [
        6,
        30,
        6,
        10,
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(regularized_logistic_pipelines),
        len(cv_parallel_results),
        len(development_parallel_results),
        len(safe_baseline_historical_features),
        min(
            len(probabilities)
            for probabilities
            in regularized_logistic_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in regularized_logistic_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in regularized_logistic_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        str(
            regularized_logistic_results[
                "penalty"
            ].eq("l2").any()
        ),
        str(
            regularized_logistic_results[
                "penalty"
            ].eq("l1").any()
        ),
        str(
            regularized_logistic_results[
                "penalty"
            ].eq("elasticnet").any()
        ),
        str(
            regularized_logistic_results[
                "class_weight"
            ].eq("balanced").any()
        ),
        str(
            regularized_logistic_results[
                "eligible_for_selection"
            ].notna().all()
        ),
        str(
            len(
                eligible_regularized_logistic_results
            ) > 0
        ),
        str(
            not regularized_logistic_results.loc[
                ~regularized_logistic_results[
                    "eligible_for_selection"
                ],
                "eligible_for_selection",
            ].any()
        ),
        str(
            eligible_regularized_logistic_results[
                "cv_pr_auc_mean"
            ].max() > dummy_reference_pr_auc
        ),
        "True",
    ],
})

regularized_logistic_validation["passed"] = [
    len(regularized_logistic_pipelines) == 6,
    len(cv_parallel_results) == 30,
    len(development_parallel_results) == 6,
    len(safe_baseline_historical_features) == 10,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in regularized_logistic_oof_probabilities.values()
    ),
    all(
        not np.isnan(probabilities).any()
        for probabilities
        in regularized_logistic_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in regularized_logistic_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    regularized_logistic_results[
        "penalty"
    ].eq("l2").any(),
    regularized_logistic_results[
        "penalty"
    ].eq("l1").any(),
    regularized_logistic_results[
        "penalty"
    ].eq("elasticnet").any(),
    regularized_logistic_results[
        "class_weight"
    ].eq("balanced").any(),
    regularized_logistic_results[
        "eligible_for_selection"
    ].notna().all(),
    len(
        eligible_regularized_logistic_results
    ) > 0,
    not regularized_logistic_results.loc[
        ~regularized_logistic_results[
            "eligible_for_selection"
        ],
        "eligible_for_selection",
    ].any(),
    (
        eligible_regularized_logistic_results[
            "cv_pr_auc_mean"
        ].max() > dummy_reference_pr_auc
    ),
    True,
]

display(regularized_logistic_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

if nonconverged_regularized_models:
    print(
        "Non-converged configurations excluded from selection: "
        + ", ".join(nonconverged_regularized_models)
    )

failed_regularized_logistic_checks = (
    regularized_logistic_validation.loc[
        ~regularized_logistic_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_regularized_logistic_checks:
    raise AssertionError(
        "Regularized logistic regression validation failed for: "
        + ", ".join(failed_regularized_logistic_checks)
    )

print(
    f"Regularized logistic regression experiment validated. "
    f"Best eligible initial configuration: "
    f"{best_regularized_logistic_model}."
)

validation_check,expected,actual,passed
Regularized configuration count,6,6,True
Cross-validation fit count,30,30,True
Development fit count,6,6,True
Safe baseline feature count,10,10,True
All configurations produced complete OOF predictions,"27,522","27,522",True
No missing OOF probabilities,0,0,True
All probabilities between 0 and 1,True,True,True
Same cross-validation folds reused,5,5,True
L2 configurations evaluated,True,True,True
L1 configurations evaluated,True,True,True


Non-converged configurations excluded from selection: L1 Balanced
Regularized logistic regression experiment validated. Best eligible initial configuration: L1.


In [38]:
# Define controlled decision tree benchmark configurations
decision_tree_configurations = {
    "Decision Tree": {
        "class_weight": None,
    },
    "Decision Tree Balanced": {
        "class_weight": "balanced",
    },
}

decision_tree_pipelines = {}

for model_name, configuration in decision_tree_configurations.items():
    decision_tree_pipelines[model_name] = Pipeline([
        (
            "preprocessor",
            build_tree_preprocessor(
                safe_baseline_historical_features
            ),
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=DECISION_TREE_MAX_DEPTH,
                min_samples_split=DECISION_TREE_MIN_SAMPLES_SPLIT,
                min_samples_leaf=DECISION_TREE_MIN_SAMPLES_LEAF,
                class_weight=configuration["class_weight"],
                random_state=RANDOM_STATE,
            ),
        ),
    ])

In [39]:
# Evaluate controlled decision tree benchmarks
decision_tree_records = []
decision_tree_oof_probabilities = {}
decision_tree_development_fits = {}

for model_name, pipeline in decision_tree_pipelines.items():
    oof_probability_positive = np.zeros(
        len(target_primary_development),
        dtype=float,
    )

    fold_train_pr_auc_scores = []
    fold_validation_pr_auc_scores = []
    fold_roc_auc_scores = []
    fold_depths = []
    fold_leaf_counts = []

    for train_indices, validation_indices in primary_cv_splits:
        fold_pipeline = clone(pipeline)

        fold_pipeline.fit(
            features_primary_development[
                safe_baseline_historical_features
            ].iloc[train_indices],
            target_primary_development.iloc[
                train_indices
            ],
        )

        train_probability_positive = fold_pipeline.predict_proba(
            features_primary_development[
                safe_baseline_historical_features
            ].iloc[train_indices]
        )[:, 1]

        validation_probability_positive = (
            fold_pipeline.predict_proba(
                features_primary_development[
                    safe_baseline_historical_features
                ].iloc[validation_indices]
            )[:, 1]
        )

        oof_probability_positive[
            validation_indices
        ] = validation_probability_positive

        fold_train_target = (
            target_primary_development.iloc[
                train_indices
            ]
        )

        fold_validation_target = (
            target_primary_development.iloc[
                validation_indices
            ]
        )

        fold_train_pr_auc_scores.append(
            average_precision_score(
                fold_train_target,
                train_probability_positive,
            )
        )

        fold_validation_pr_auc_scores.append(
            average_precision_score(
                fold_validation_target,
                validation_probability_positive,
            )
        )

        fold_roc_auc_scores.append(
            roc_auc_score(
                fold_validation_target,
                validation_probability_positive,
            )
        )

        fold_classifier = fold_pipeline.named_steps[
            "classifier"
        ]

        fold_depths.append(
            fold_classifier.get_depth()
        )

        fold_leaf_counts.append(
            fold_classifier.get_n_leaves()
        )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    development_fit = clone(pipeline)

    development_fit.fit(
        features_primary_development[
            safe_baseline_historical_features
        ],
        target_primary_development,
    )

    development_classifier = development_fit.named_steps[
        "classifier"
    ]

    decision_tree_oof_probabilities[
        model_name
    ] = oof_probability_positive

    decision_tree_development_fits[
        model_name
    ] = development_fit

    decision_tree_records.append({
        "model": model_name,
        "class_weight": (
            decision_tree_configurations[
                model_name
            ]["class_weight"]
            or "None"
        ),
        "cv_train_pr_auc_mean": np.mean(
            fold_train_pr_auc_scores
        ),
        "cv_pr_auc_mean": np.mean(
            fold_validation_pr_auc_scores
        ),
        "cv_pr_auc_std": np.std(
            fold_validation_pr_auc_scores
        ),
        "cv_pr_auc_gap": (
            np.mean(fold_train_pr_auc_scores)
            - np.mean(fold_validation_pr_auc_scores)
        ),
        "cv_roc_auc_mean": np.mean(
            fold_roc_auc_scores
        ),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "recall": evaluation_metrics["recall"],
        "precision": evaluation_metrics["precision"],
        "f1_score": evaluation_metrics["f1_score"],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": evaluation_metrics[
            "predicted_positive_rate"
        ],
        "brier_score": evaluation_metrics[
            "brier_score"
        ],
        "pr_auc_multiple_vs_dummy": (
            np.mean(fold_validation_pr_auc_scores)
            / dummy_reference_pr_auc
        ),
        "pr_auc_improvement_vs_standard_logistic": (
            np.mean(fold_validation_pr_auc_scores)
            - scaled_logistic_result[
                "cv_pr_auc_mean"
            ]
        ),
        "maximum_fold_depth": max(
            fold_depths
        ),
        "maximum_fold_leaf_count": max(
            fold_leaf_counts
        ),
        "development_tree_depth": (
            development_classifier.get_depth()
        ),
        "development_leaf_count": (
            development_classifier.get_n_leaves()
        ),
    })

decision_tree_results = pd.DataFrame(
    decision_tree_records
).sort_values(
    "cv_pr_auc_mean",
    ascending=False,
).reset_index(drop=True)

display(decision_tree_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["model"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_improvement_vs_standard_logistic": "{:+.4f}",
        "maximum_fold_depth": "{:,.0f}",
        "maximum_fold_leaf_count": "{:,.0f}",
        "development_tree_depth": "{:,.0f}",
        "development_leaf_count": "{:,.0f}",
    })
)

model,class_weight,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_improvement_vs_standard_logistic,maximum_fold_depth,maximum_fold_leaf_count,development_tree_depth,development_leaf_count
Decision Tree,None,0.0653,0.0623,0.0032,0.0030,0.5055,0.0622,0.0000,0.0000,0.0000,0.5000,0.00%,0.0521,1.13×,-0.0108,5,9,5,9
Decision Tree Balanced,balanced,0.0659,0.0622,0.0036,0.0037,0.5036,0.0620,0.0394,0.0976,0.0561,0.5090,2.23%,0.2473,1.12×,-0.0110,5,11,5,10


In [40]:
# Validate controlled decision tree benchmark
best_decision_tree_model = (
    decision_tree_results.iloc[0]["model"]
)

decision_tree_validation = pd.DataFrame({
    "validation_check": [
        "Decision tree configuration count",
        "Safe baseline feature count",
        "All models produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "Maximum depth restriction applied",
        "Minimum split restriction applied",
        "Minimum leaf restriction applied",
        "Unweighted tree evaluated",
        "Class-weighted tree evaluated",
        "Observed tree depths within configured maximum",
        "Best tree exceeds dummy PR-AUC",
        "Final test set remained unused",
    ],
    "expected": [
        2,
        10,
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        DECISION_TREE_MAX_DEPTH,
        DECISION_TREE_MIN_SAMPLES_SPLIT,
        DECISION_TREE_MIN_SAMPLES_LEAF,
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(decision_tree_pipelines),
        len(safe_baseline_historical_features),
        min(
            len(probabilities)
            for probabilities
            in decision_tree_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in decision_tree_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in decision_tree_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        DECISION_TREE_MAX_DEPTH,
        DECISION_TREE_MIN_SAMPLES_SPLIT,
        DECISION_TREE_MIN_SAMPLES_LEAF,
        str(
            decision_tree_results[
                "class_weight"
            ].eq("None").any()
        ),
        str(
            decision_tree_results[
                "class_weight"
            ].eq("balanced").any()
        ),
        str(
            decision_tree_results[
                "maximum_fold_depth"
            ].le(
                DECISION_TREE_MAX_DEPTH
            ).all()
        ),
        str(
            decision_tree_results.iloc[0][
                "cv_pr_auc_mean"
            ] > dummy_reference_pr_auc
        ),
        "True",
    ],
})

decision_tree_validation["passed"] = [
    len(decision_tree_pipelines) == 2,
    len(safe_baseline_historical_features) == 10,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in decision_tree_oof_probabilities.values()
    ),
    all(
        not np.isnan(probabilities).any()
        for probabilities
        in decision_tree_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in decision_tree_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    DECISION_TREE_MAX_DEPTH == 5,
    DECISION_TREE_MIN_SAMPLES_SPLIT == 100,
    DECISION_TREE_MIN_SAMPLES_LEAF == 50,
    decision_tree_results[
        "class_weight"
    ].eq("None").any(),
    decision_tree_results[
        "class_weight"
    ].eq("balanced").any(),
    decision_tree_results[
        "maximum_fold_depth"
    ].le(
        DECISION_TREE_MAX_DEPTH
    ).all(),
    (
        decision_tree_results.iloc[0][
            "cv_pr_auc_mean"
        ] > dummy_reference_pr_auc
    ),
    True,
]

display(decision_tree_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [("text-align", "left")],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_decision_tree_checks = (
    decision_tree_validation.loc[
        ~decision_tree_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_decision_tree_checks:
    raise AssertionError(
        "Decision tree benchmark validation failed for: "
        + ", ".join(failed_decision_tree_checks)
    )

print(
    f"Controlled decision tree benchmark validated. "
    f"Highest initial CV PR-AUC: {best_decision_tree_model}."
)

validation_check,expected,actual,passed
Decision tree configuration count,2,2,True
Safe baseline feature count,10,10,True
All models produced complete OOF predictions,"27,522","27,522",True
No missing OOF probabilities,0,0,True
All probabilities between 0 and 1,True,True,True
Same cross-validation folds reused,5,5,True
Maximum depth restriction applied,5,5,True
Minimum split restriction applied,100,100,True
Minimum leaf restriction applied,50,50,True
Unweighted tree evaluated,True,True,True


Controlled decision tree benchmark validated. Highest initial CV PR-AUC: Decision Tree.


In [41]:
# Define controlled random forest configurations
random_forest_configurations = {
    "RF Controlled": {
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 50,
        "min_samples_leaf": 25,
        "max_features": "sqrt",
        "class_weight": None,
    },
    "RF Controlled Balanced": {
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 50,
        "min_samples_leaf": 25,
        "max_features": "sqrt",
        "class_weight": "balanced",
    },
    "RF Controlled Balanced Subsample": {
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 50,
        "min_samples_leaf": 25,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
    },
    "RF Moderate": {
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None,
    },
    "RF Moderate Balanced Subsample": {
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": "balanced_subsample",
    },
    "RF Moderate Wider Features": {
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "max_features": 0.50,
        "class_weight": "balanced_subsample",
    },
}

random_forest_pipelines = {}

for model_name, configuration in random_forest_configurations.items():
    random_forest_pipelines[model_name] = Pipeline([
        (
            "preprocessor",
            build_tree_preprocessor(
                safe_baseline_historical_features
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=configuration[
                    "n_estimators"
                ],
                max_depth=configuration[
                    "max_depth"
                ],
                min_samples_split=configuration[
                    "min_samples_split"
                ],
                min_samples_leaf=configuration[
                    "min_samples_leaf"
                ],
                max_features=configuration[
                    "max_features"
                ],
                class_weight=configuration[
                    "class_weight"
                ],
                bootstrap=True,
                n_jobs=1,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

features_random_forest_development = (
    features_primary_development[
        safe_baseline_historical_features
    ].copy()
)

In [42]:
# Define reusable random forest evaluation functions
def evaluate_random_forest_fold(
    model_name,
    pipeline,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = clone(pipeline)

    fold_pipeline.fit(
        features_random_forest_development.iloc[
            train_indices
        ],
        target_primary_development.iloc[
            train_indices
        ],
    )

    train_probability_positive = (
        fold_pipeline.predict_proba(
            features_random_forest_development.iloc[
                train_indices
            ]
        )[:, 1]
    )

    validation_probability_positive = (
        fold_pipeline.predict_proba(
            features_random_forest_development.iloc[
                validation_indices
            ]
        )[:, 1]
    )

    fold_train_target = (
        target_primary_development.iloc[
            train_indices
        ]
    )

    fold_validation_target = (
        target_primary_development.iloc[
            validation_indices
        ]
    )

    classifier = fold_pipeline.named_steps[
        "classifier"
    ]

    tree_depths = [
        estimator.get_depth()
        for estimator in classifier.estimators_
    ]

    return {
        "model": model_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": (
            validation_probability_positive
        ),
        "train_pr_auc": average_precision_score(
            fold_train_target,
            train_probability_positive,
        ),
        "pr_auc": average_precision_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "mean_tree_depth": np.mean(
            tree_depths
        ),
        "maximum_tree_depth": max(
            tree_depths
        ),
    }


def fit_random_forest_development(
    model_name,
    pipeline,
):
    development_fit = clone(pipeline)

    development_fit.fit(
        features_random_forest_development,
        target_primary_development,
    )

    classifier = development_fit.named_steps[
        "classifier"
    ]

    tree_depths = [
        estimator.get_depth()
        for estimator in classifier.estimators_
    ]

    leaf_counts = [
        estimator.get_n_leaves()
        for estimator in classifier.estimators_
    ]

    return {
        "model": model_name,
        "development_fit": development_fit,
        "mean_tree_depth": np.mean(
            tree_depths
        ),
        "maximum_tree_depth": max(
            tree_depths
        ),
        "mean_leaf_count": np.mean(
            leaf_counts
        ),
        "maximum_leaf_count": max(
            leaf_counts
        ),
    }

In [43]:
# Evaluate random forest configurations in parallel
random_forest_cv_tasks = [
    (
        model_name,
        pipeline,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name, pipeline
    in random_forest_pipelines.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    random_forest_cv_results = joblib.Parallel(
        n_jobs=RANDOM_FOREST_PARALLEL_JOBS,
    )(
        joblib.delayed(
            evaluate_random_forest_fold
        )(
            model_name,
            pipeline,
            fold_number,
            train_indices,
            validation_indices,
        )
        for (
            model_name,
            pipeline,
            fold_number,
            train_indices,
            validation_indices,
        ) in random_forest_cv_tasks
    )

    random_forest_development_results = (
        joblib.Parallel(
            n_jobs=RANDOM_FOREST_PARALLEL_JOBS,
        )(
            joblib.delayed(
                fit_random_forest_development
            )(
                model_name,
                pipeline,
            )
            for model_name, pipeline
            in random_forest_pipelines.items()
        )
    )

random_forest_development_by_model = {
    result["model"]: result
    for result in random_forest_development_results
}

random_forest_records = []
random_forest_oof_probabilities = {}
random_forest_development_fits = {}

for model_name in random_forest_pipelines:
    model_fold_results = sorted(
        [
            result
            for result in random_forest_cv_results
            if result["model"] == model_name
        ],
        key=lambda result: result["fold"],
    )

    oof_probability_positive = np.full(
        len(target_primary_development),
        np.nan,
        dtype=float,
    )

    for fold_result in model_fold_results:
        oof_probability_positive[
            fold_result["validation_indices"]
        ] = fold_result["probability_positive"]

    if np.isnan(oof_probability_positive).any():
        raise AssertionError(
            f"Incomplete OOF probabilities for {model_name}."
        )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    configuration = random_forest_configurations[
        model_name
    ]

    development_result = (
        random_forest_development_by_model[
            model_name
        ]
    )

    cv_train_pr_auc_mean = np.mean([
        result["train_pr_auc"]
        for result in model_fold_results
    ])

    cv_pr_auc_mean = np.mean([
        result["pr_auc"]
        for result in model_fold_results
    ])

    random_forest_oof_probabilities[
        model_name
    ] = oof_probability_positive

    random_forest_development_fits[
        model_name
    ] = development_result[
        "development_fit"
    ]

    random_forest_records.append({
        "model": model_name,
        "n_estimators": configuration[
            "n_estimators"
        ],
        "max_depth": configuration[
            "max_depth"
        ],
        "min_samples_leaf": configuration[
            "min_samples_leaf"
        ],
        "max_features": configuration[
            "max_features"
        ],
        "class_weight": (
            configuration["class_weight"]
            or "None"
        ),
        "cv_train_pr_auc_mean": (
            cv_train_pr_auc_mean
        ),
        "cv_pr_auc_mean": cv_pr_auc_mean,
        "cv_pr_auc_std": np.std([
            result["pr_auc"]
            for result in model_fold_results
        ]),
        "cv_pr_auc_gap": (
            cv_train_pr_auc_mean
            - cv_pr_auc_mean
        ),
        "cv_roc_auc_mean": np.mean([
            result["roc_auc"]
            for result in model_fold_results
        ]),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "recall": evaluation_metrics[
            "recall"
        ],
        "precision": evaluation_metrics[
            "precision"
        ],
        "f1_score": evaluation_metrics[
            "f1_score"
        ],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": (
            evaluation_metrics[
                "predicted_positive_rate"
            ]
        ),
        "brier_score": evaluation_metrics[
            "brier_score"
        ],
        "pr_auc_multiple_vs_dummy": (
            cv_pr_auc_mean
            / dummy_reference_pr_auc
        ),
        "pr_auc_improvement_vs_standard_logistic": (
            cv_pr_auc_mean
            - scaled_logistic_result[
                "cv_pr_auc_mean"
            ]
        ),
        "mean_development_tree_depth": (
            development_result[
                "mean_tree_depth"
            ]
        ),
        "maximum_development_tree_depth": (
            development_result[
                "maximum_tree_depth"
            ]
        ),
        "mean_development_leaf_count": (
            development_result[
                "mean_leaf_count"
            ]
        ),
    })

random_forest_results = pd.DataFrame(
    random_forest_records
).sort_values(
    "cv_pr_auc_mean",
    ascending=False,
).reset_index(drop=True)

display(random_forest_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["model"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "n_estimators": "{:,.0f}",
        "max_depth": "{:,.0f}",
        "min_samples_leaf": "{:,.0f}",
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_improvement_vs_standard_logistic": "{:+.4f}",
        "mean_development_tree_depth": "{:.2f}",
        "maximum_development_tree_depth": "{:,.0f}",
        "mean_development_leaf_count": "{:.2f}",
    })
)

model,n_estimators,max_depth,min_samples_leaf,max_features,class_weight,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_improvement_vs_standard_logistic,mean_development_tree_depth,maximum_development_tree_depth,mean_development_leaf_count
RF Moderate,400,12,10,sqrt,None,0.1319,0.0740,0.0056,0.0579,0.5107,0.0722,0.0000,0.0000,0.0000,0.5000,0.00%,0.0520,1.34×,+0.0009,12.00,12,55.73
RF Moderate Balanced Subsample,400,12,10,sqrt,balanced_subsample,0.1245,0.0714,0.0067,0.0531,0.5074,0.0699,0.2738,0.0560,0.0930,0.5017,27.06%,0.2314,1.29×,-0.0017,12.00,12,62.34
RF Moderate Wider Features,400,12,10,0.500000,balanced_subsample,0.1351,0.0712,0.0071,0.0639,0.5138,0.0688,0.3040,0.0582,0.0976,0.5078,28.93%,0.2283,1.29×,-0.0020,12.00,12,71.79
RF Controlled Balanced Subsample,200,8,25,sqrt,balanced_subsample,0.0982,0.0691,0.0072,0.0291,0.4998,0.0656,0.2896,0.0532,0.0899,0.4940,30.10%,0.2401,1.25×,-0.0041,8.00,8,24.77
RF Controlled Balanced,200,8,25,sqrt,balanced,0.0967,0.0690,0.0083,0.0277,0.4982,0.0652,0.2731,0.0563,0.0933,0.5024,26.87%,0.2449,1.25×,-0.0041,8.00,8,20.51
RF Controlled,200,8,25,sqrt,None,0.0988,0.0685,0.0071,0.0303,0.4977,0.0654,0.0000,0.0000,0.0000,0.5000,0.00%,0.0521,1.24×,-0.0046,8.00,8,22.39


In [44]:
# Validate random forest experiment
best_random_forest_model = (
    random_forest_results.iloc[0]["model"]
)

random_forest_validation = pd.DataFrame({
    "validation_check": [
        "Random forest configuration count",
        "Cross-validation fit count",
        "Development fit count",
        "Safe baseline feature count",
        "All models produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "Multiple tree counts evaluated",
        "Multiple depth limits evaluated",
        "Multiple minimum leaf sizes evaluated",
        "Multiple feature sampling settings evaluated",
        "Unweighted forest evaluated",
        "Balanced class weighting evaluated",
        "Balanced subsample weighting evaluated",
        "Observed tree depths within configured maximum",
        "Best forest exceeds dummy PR-AUC",
        "Final test set remained unused",
    ],
    "expected": [
        6,
        30,
        6,
        10,
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(random_forest_pipelines),
        len(random_forest_cv_results),
        len(random_forest_development_results),
        len(safe_baseline_historical_features),
        min(
            len(probabilities)
            for probabilities
            in random_forest_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in random_forest_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in random_forest_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        str(
            random_forest_results[
                "n_estimators"
            ].nunique() > 1
        ),
        str(
            random_forest_results[
                "max_depth"
            ].nunique() > 1
        ),
        str(
            random_forest_results[
                "min_samples_leaf"
            ].nunique() > 1
        ),
        str(
            random_forest_results[
                "max_features"
            ].astype(str).nunique() > 1
        ),
        str(
            random_forest_results[
                "class_weight"
            ].eq("None").any()
        ),
        str(
            random_forest_results[
                "class_weight"
            ].eq("balanced").any()
        ),
        str(
            random_forest_results[
                "class_weight"
            ].eq(
                "balanced_subsample"
            ).any()
        ),
        str(all(
            row[
                "maximum_development_tree_depth"
            ] <= row["max_depth"]
            for _, row
            in random_forest_results.iterrows()
        )),
        str(
            random_forest_results.iloc[0][
                "cv_pr_auc_mean"
            ] > dummy_reference_pr_auc
        ),
        "True",
    ],
})

random_forest_validation["passed"] = [
    len(random_forest_pipelines) == 6,
    len(random_forest_cv_results) == 30,
    len(random_forest_development_results) == 6,
    len(safe_baseline_historical_features) == 10,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in random_forest_oof_probabilities.values()
    ),
    all(
        not np.isnan(probabilities).any()
        for probabilities
        in random_forest_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in random_forest_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    random_forest_results[
        "n_estimators"
    ].nunique() > 1,
    random_forest_results[
        "max_depth"
    ].nunique() > 1,
    random_forest_results[
        "min_samples_leaf"
    ].nunique() > 1,
    random_forest_results[
        "max_features"
    ].astype(str).nunique() > 1,
    random_forest_results[
        "class_weight"
    ].eq("None").any(),
    random_forest_results[
        "class_weight"
    ].eq("balanced").any(),
    random_forest_results[
        "class_weight"
    ].eq("balanced_subsample").any(),
    all(
        row[
            "maximum_development_tree_depth"
        ] <= row["max_depth"]
        for _, row
        in random_forest_results.iterrows()
    ),
    (
        random_forest_results.iloc[0][
            "cv_pr_auc_mean"
        ] > dummy_reference_pr_auc
    ),
    True,
]

display(random_forest_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_random_forest_checks = (
    random_forest_validation.loc[
        ~random_forest_validation["passed"],
        "validation_check",
    ].tolist()
)

if failed_random_forest_checks:
    raise AssertionError(
        "Random forest validation failed for: "
        + ", ".join(
            failed_random_forest_checks
        )
    )

print(
    f"Random forest experiment validated. "
    f"Highest initial CV PR-AUC: "
    f"{best_random_forest_model}."
)

validation_check,expected,actual,passed
Random forest configuration count,6,6,True
Cross-validation fit count,30,30,True
Development fit count,6,6,True
Safe baseline feature count,10,10,True
All models produced complete OOF predictions,"27,522","27,522",True
No missing OOF probabilities,0,0,True
All probabilities between 0 and 1,True,True,True
Same cross-validation folds reused,5,5,True
Multiple tree counts evaluated,True,True,True
Multiple depth limits evaluated,True,True,True


Random forest experiment validated. Highest initial CV PR-AUC: RF Moderate.


In [45]:
# Define histogram gradient boosting configurations
hist_gradient_boosting_configurations = {
    "HGB Conservative": {
        "learning_rate": 0.05,
        "max_iter": 150,
        "max_leaf_nodes": 15,
        "max_depth": 5,
        "min_samples_leaf": 40,
        "l2_regularization": 1.0,
        "class_weight": None,
    },
    "HGB Moderate": {
        "learning_rate": 0.05,
        "max_iter": 250,
        "max_leaf_nodes": 31,
        "max_depth": 6,
        "min_samples_leaf": 25,
        "l2_regularization": 0.50,
        "class_weight": None,
    },
    "HGB Moderate Balanced": {
        "learning_rate": 0.05,
        "max_iter": 250,
        "max_leaf_nodes": 31,
        "max_depth": 6,
        "min_samples_leaf": 25,
        "l2_regularization": 0.50,
        "class_weight": "balanced",
    },
    "HGB Faster Learning": {
        "learning_rate": 0.10,
        "max_iter": 125,
        "max_leaf_nodes": 31,
        "max_depth": 6,
        "min_samples_leaf": 25,
        "l2_regularization": 0.50,
        "class_weight": None,
    },
}

hist_gradient_boosting_pipelines = {}

for (
    model_name,
    configuration,
) in hist_gradient_boosting_configurations.items():
    hist_gradient_boosting_pipelines[
        model_name
    ] = Pipeline([
        (
            "preprocessor",
            build_tree_preprocessor(
                safe_baseline_historical_features
            ),
        ),
        (
            "classifier",
            HistGradientBoostingClassifier(
                learning_rate=configuration[
                    "learning_rate"
                ],
                max_iter=configuration[
                    "max_iter"
                ],
                max_leaf_nodes=configuration[
                    "max_leaf_nodes"
                ],
                max_depth=configuration[
                    "max_depth"
                ],
                min_samples_leaf=configuration[
                    "min_samples_leaf"
                ],
                l2_regularization=configuration[
                    "l2_regularization"
                ],
                class_weight=configuration[
                    "class_weight"
                ],
                early_stopping=False,
                random_state=RANDOM_STATE,
            ),
        ),
    ])

features_hist_gradient_boosting_development = (
    features_primary_development[
        safe_baseline_historical_features
    ].copy()
)

In [46]:
# Define reusable histogram gradient boosting evaluation functions
def evaluate_hist_gradient_boosting_fold(
    model_name,
    pipeline,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = clone(pipeline)

    fold_pipeline.fit(
        features_hist_gradient_boosting_development.iloc[
            train_indices
        ],
        target_primary_development.iloc[
            train_indices
        ],
    )

    train_probability_positive = (
        fold_pipeline.predict_proba(
            features_hist_gradient_boosting_development.iloc[
                train_indices
            ]
        )[:, 1]
    )

    validation_probability_positive = (
        fold_pipeline.predict_proba(
            features_hist_gradient_boosting_development.iloc[
                validation_indices
            ]
        )[:, 1]
    )

    fold_train_target = (
        target_primary_development.iloc[
            train_indices
        ]
    )

    fold_validation_target = (
        target_primary_development.iloc[
            validation_indices
        ]
    )

    classifier = fold_pipeline.named_steps[
        "classifier"
    ]

    return {
        "model": model_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": (
            validation_probability_positive
        ),
        "train_pr_auc": average_precision_score(
            fold_train_target,
            train_probability_positive,
        ),
        "pr_auc": average_precision_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "iterations_completed": classifier.n_iter_,
    }


def fit_hist_gradient_boosting_development(
    model_name,
    pipeline,
):
    development_fit = clone(pipeline)

    development_fit.fit(
        features_hist_gradient_boosting_development,
        target_primary_development,
    )

    classifier = development_fit.named_steps[
        "classifier"
    ]

    return {
        "model": model_name,
        "development_fit": development_fit,
        "iterations_completed": classifier.n_iter_,
    }

In [47]:
# Evaluate histogram gradient boosting configurations in parallel
hist_gradient_boosting_cv_tasks = [
    (
        model_name,
        pipeline,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name, pipeline
    in hist_gradient_boosting_pipelines.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    hist_gradient_boosting_cv_results = (
        joblib.Parallel(
            n_jobs=HIST_GRADIENT_BOOSTING_PARALLEL_JOBS,
        )(
            joblib.delayed(
                evaluate_hist_gradient_boosting_fold
            )(
                model_name,
                pipeline,
                fold_number,
                train_indices,
                validation_indices,
            )
            for (
                model_name,
                pipeline,
                fold_number,
                train_indices,
                validation_indices,
            ) in hist_gradient_boosting_cv_tasks
        )
    )

    hist_gradient_boosting_development_results = (
        joblib.Parallel(
            n_jobs=HIST_GRADIENT_BOOSTING_PARALLEL_JOBS,
        )(
            joblib.delayed(
                fit_hist_gradient_boosting_development
            )(
                model_name,
                pipeline,
            )
            for model_name, pipeline
            in hist_gradient_boosting_pipelines.items()
        )
    )

hist_gradient_boosting_development_by_model = {
    result["model"]: result
    for result
    in hist_gradient_boosting_development_results
}

hist_gradient_boosting_records = []
hist_gradient_boosting_oof_probabilities = {}
hist_gradient_boosting_development_fits = {}

for model_name in hist_gradient_boosting_pipelines:
    model_fold_results = sorted(
        [
            result
            for result
            in hist_gradient_boosting_cv_results
            if result["model"] == model_name
        ],
        key=lambda result: result["fold"],
    )

    oof_probability_positive = np.full(
        len(target_primary_development),
        np.nan,
        dtype=float,
    )

    for fold_result in model_fold_results:
        oof_probability_positive[
            fold_result["validation_indices"]
        ] = fold_result["probability_positive"]

    if np.isnan(oof_probability_positive).any():
        raise AssertionError(
            f"Incomplete OOF probabilities for {model_name}."
        )

    evaluation_metrics = calculate_classification_metrics(
        target_true=target_primary_development,
        probability_positive=oof_probability_positive,
        threshold=DEFAULT_PROBABILITY_THRESHOLD,
    )

    configuration = (
        hist_gradient_boosting_configurations[
            model_name
        ]
    )

    development_result = (
        hist_gradient_boosting_development_by_model[
            model_name
        ]
    )

    cv_train_pr_auc_mean = np.mean([
        result["train_pr_auc"]
        for result in model_fold_results
    ])

    cv_pr_auc_mean = np.mean([
        result["pr_auc"]
        for result in model_fold_results
    ])

    hist_gradient_boosting_oof_probabilities[
        model_name
    ] = oof_probability_positive

    hist_gradient_boosting_development_fits[
        model_name
    ] = development_result[
        "development_fit"
    ]

    hist_gradient_boosting_records.append({
        "model": model_name,
        "learning_rate": configuration[
            "learning_rate"
        ],
        "max_iter": configuration[
            "max_iter"
        ],
        "max_leaf_nodes": configuration[
            "max_leaf_nodes"
        ],
        "max_depth": configuration[
            "max_depth"
        ],
        "min_samples_leaf": configuration[
            "min_samples_leaf"
        ],
        "l2_regularization": configuration[
            "l2_regularization"
        ],
        "class_weight": (
            configuration["class_weight"]
            or "None"
        ),
        "cv_train_pr_auc_mean": (
            cv_train_pr_auc_mean
        ),
        "cv_pr_auc_mean": cv_pr_auc_mean,
        "cv_pr_auc_std": np.std([
            result["pr_auc"]
            for result in model_fold_results
        ]),
        "cv_pr_auc_gap": (
            cv_train_pr_auc_mean
            - cv_pr_auc_mean
        ),
        "cv_roc_auc_mean": np.mean([
            result["roc_auc"]
            for result in model_fold_results
        ]),
        "oof_pr_auc": evaluation_metrics[
            "pr_auc_average_precision"
        ],
        "recall": evaluation_metrics[
            "recall"
        ],
        "precision": evaluation_metrics[
            "precision"
        ],
        "f1_score": evaluation_metrics[
            "f1_score"
        ],
        "balanced_accuracy": evaluation_metrics[
            "balanced_accuracy"
        ],
        "predicted_positive_rate": (
            evaluation_metrics[
                "predicted_positive_rate"
            ]
        ),
        "brier_score": evaluation_metrics[
            "brier_score"
        ],
        "pr_auc_multiple_vs_dummy": (
            cv_pr_auc_mean
            / dummy_reference_pr_auc
        ),
        "pr_auc_improvement_vs_standard_logistic": (
            cv_pr_auc_mean
            - scaled_logistic_result[
                "cv_pr_auc_mean"
            ]
        ),
        "development_iterations_completed": (
            development_result[
                "iterations_completed"
            ]
        ),
    })

hist_gradient_boosting_results = pd.DataFrame(
    hist_gradient_boosting_records
).sort_values(
    "cv_pr_auc_mean",
    ascending=False,
).reset_index(drop=True)

display(hist_gradient_boosting_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["model"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "learning_rate": "{:.2f}",
        "max_iter": "{:,.0f}",
        "max_leaf_nodes": "{:,.0f}",
        "max_depth": "{:,.0f}",
        "min_samples_leaf": "{:,.0f}",
        "l2_regularization": "{:.2f}",
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_improvement_vs_standard_logistic": "{:+.4f}",
        "development_iterations_completed": "{:,.0f}",
    })
)

model,learning_rate,max_iter,max_leaf_nodes,max_depth,min_samples_leaf,l2_regularization,class_weight,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_improvement_vs_standard_logistic,development_iterations_completed
HGB Faster Learning,0.10,125,31,6,25,0.50,None,0.1657,0.0715,0.0056,0.0942,0.5184,0.0694,0.0079,0.5455,0.0155,0.5037,0.08%,0.0522,1.29×,-0.0016,125
HGB Moderate Balanced,0.05,250,31,6,25,0.50,balanced,0.1623,0.0709,0.0044,0.0914,0.5183,0.0692,0.3933,0.0570,0.0996,0.5062,38.17%,0.2315,1.28×,-0.0022,250
HGB Moderate,0.05,250,31,6,25,0.50,None,0.1624,0.0694,0.0035,0.0930,0.5145,0.0678,0.0072,0.5238,0.0142,0.5034,0.08%,0.0522,1.25×,-0.0037,250
HGB Conservative,0.05,150,15,5,40,1.00,None,0.1169,0.0688,0.0052,0.0481,0.5111,0.0664,0.0039,0.5455,0.0078,0.5019,0.04%,0.0522,1.24×,-0.0043,150


In [48]:
# Validate histogram gradient boosting experiment
best_hist_gradient_boosting_model = (
    hist_gradient_boosting_results.iloc[0][
        "model"
    ]
)

hist_gradient_boosting_validation = pd.DataFrame({
    "validation_check": [
        "Gradient boosting configuration count",
        "Cross-validation fit count",
        "Development fit count",
        "Safe baseline feature count",
        "All models produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "Multiple learning rates evaluated",
        "Multiple boosting iteration counts evaluated",
        "Multiple tree capacities evaluated",
        "L2 regularization evaluated",
        "Unweighted boosting evaluated",
        "Class-weighted boosting evaluated",
        "Configured boosting iterations completed",
        "Best boosted model exceeds dummy PR-AUC",
        "Final test set remained unused",
        "No additional package dependency required",
    ],
    "expected": [
        4,
        20,
        4,
        10,
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(
            hist_gradient_boosting_pipelines
        ),
        len(
            hist_gradient_boosting_cv_results
        ),
        len(
            hist_gradient_boosting_development_results
        ),
        len(
            safe_baseline_historical_features
        ),
        min(
            len(probabilities)
            for probabilities
            in hist_gradient_boosting_oof_probabilities.values()
        ),
        sum(
            np.isnan(
                probabilities
            ).sum()
            for probabilities
            in hist_gradient_boosting_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in hist_gradient_boosting_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        str(
            hist_gradient_boosting_results[
                "learning_rate"
            ].nunique() > 1
        ),
        str(
            hist_gradient_boosting_results[
                "max_iter"
            ].nunique() > 1
        ),
        str(
            (
                hist_gradient_boosting_results[
                    "max_leaf_nodes"
                ].nunique() > 1
            )
            or (
                hist_gradient_boosting_results[
                    "max_depth"
                ].nunique() > 1
            )
        ),
        str(
            hist_gradient_boosting_results[
                "l2_regularization"
            ].gt(0).any()
        ),
        str(
            hist_gradient_boosting_results[
                "class_weight"
            ].eq("None").any()
        ),
        str(
            hist_gradient_boosting_results[
                "class_weight"
            ].eq("balanced").any()
        ),
        str(all(
            row[
                "development_iterations_completed"
            ] == row["max_iter"]
            for _, row
            in hist_gradient_boosting_results.iterrows()
        )),
        str(
            hist_gradient_boosting_results.iloc[
                0
            ]["cv_pr_auc_mean"]
            > dummy_reference_pr_auc
        ),
        "True",
        "True",
    ],
})

hist_gradient_boosting_validation[
    "passed"
] = [
    len(
        hist_gradient_boosting_pipelines
    ) == 4,
    len(
        hist_gradient_boosting_cv_results
    ) == 20,
    len(
        hist_gradient_boosting_development_results
    ) == 4,
    len(
        safe_baseline_historical_features
    ) == 10,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in hist_gradient_boosting_oof_probabilities.values()
    ),
    all(
        not np.isnan(
            probabilities
        ).any()
        for probabilities
        in hist_gradient_boosting_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in hist_gradient_boosting_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    hist_gradient_boosting_results[
        "learning_rate"
    ].nunique() > 1,
    hist_gradient_boosting_results[
        "max_iter"
    ].nunique() > 1,
    (
        hist_gradient_boosting_results[
            "max_leaf_nodes"
        ].nunique() > 1
        or hist_gradient_boosting_results[
            "max_depth"
        ].nunique() > 1
    ),
    hist_gradient_boosting_results[
        "l2_regularization"
    ].gt(0).any(),
    hist_gradient_boosting_results[
        "class_weight"
    ].eq("None").any(),
    hist_gradient_boosting_results[
        "class_weight"
    ].eq("balanced").any(),
    all(
        row[
            "development_iterations_completed"
        ] == row["max_iter"]
        for _, row
        in hist_gradient_boosting_results.iterrows()
    ),
    (
        hist_gradient_boosting_results.iloc[
            0
        ]["cv_pr_auc_mean"]
        > dummy_reference_pr_auc
    ),
    True,
    True,
]

display(hist_gradient_boosting_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_hist_gradient_boosting_checks = (
    hist_gradient_boosting_validation.loc[
        ~hist_gradient_boosting_validation[
            "passed"
        ],
        "validation_check",
    ].tolist()
)

if failed_hist_gradient_boosting_checks:
    raise AssertionError(
        "Histogram gradient boosting validation failed for: "
        + ", ".join(
            failed_hist_gradient_boosting_checks
        )
    )

print(
    f"Histogram gradient boosting experiment validated. "
    f"Highest initial CV PR-AUC: "
    f"{best_hist_gradient_boosting_model}."
)

print(
    "Additional package dependency: None "
    "(scikit-learn HistGradientBoostingClassifier)."
)

validation_check,expected,actual,passed
Gradient boosting configuration count,4,4,True
Cross-validation fit count,20,20,True
Development fit count,4,4,True
Safe baseline feature count,10,10,True
All models produced complete OOF predictions,"27,522","27,522",True
No missing OOF probabilities,0,0,True
All probabilities between 0 and 1,True,True,True
Same cross-validation folds reused,5,5,True
Multiple learning rates evaluated,True,True,True
Multiple boosting iteration counts evaluated,True,True,True


Histogram gradient boosting experiment validated. Highest initial CV PR-AUC: HGB Faster Learning.
Additional package dependency: None (scikit-learn HistGradientBoostingClassifier).


## Initial Model Benchmarks

Initial model-family benchmarks were evaluated using the same stratified five-fold cross-validation splits and the 10-feature Safe Baseline Historical feature set. PR-AUC was used as the primary comparison metric because the positive donor class represents only about 5.53% of the development data. The untouched test set was not used during any of these experiments.

### Dummy Baselines

Three dummy classifiers were evaluated to establish no-skill reference behavior: Most Frequent, Stratified Random, and Prior Probability.

The Most Frequent and Prior Probability strategies produced PR-AUC values of approximately 0.0553, closely matching the development-set positive-class prevalence. This value was retained as the primary no-skill PR-AUC reference for subsequent model comparisons.

The Prior Probability strategy produced the best dummy Brier score at approximately 0.0523 because it assigned probabilities according to the observed class prevalence rather than making random positive predictions. The Stratified Random strategy produced a higher predicted-positive rate but did not improve discrimination, confirming that randomly increasing positive predictions does not provide useful donor-ranking ability.

### Standard Logistic Regression

A standard logistic regression model was evaluated using the Safe Baseline Historical feature set. Scaled and unscaled preprocessing variants were compared to determine whether feature scaling materially affected optimization and predictive performance.

The scaled model achieved a mean cross-validation PR-AUC of 0.0731, approximately 1.32 times the dummy baseline. The unscaled model produced a slightly higher mean cross-validation PR-AUC of approximately 0.0742, but it failed to converge even after reaching the 2,000-iteration limit. In comparison, the scaled model converged cleanly in approximately 35–40 iterations. Scaling was therefore retained because the small apparent gain from the unscaled model could not be treated as reliable while optimization remained incomplete.

At the default 0.50 probability threshold, the scaled logistic model predicted only 19 development records as positive. Of those predictions, 16 were actual donors, producing high precision of approximately 84.21% but recall of only 1.05%. This demonstrated that the model's probability ranking can contain useful signal even when the default classification threshold produces very few positive predictions.

The model's ROC-AUC remained near 0.50 despite its PR-AUC exceeding the no-skill baseline. This behavior was retained as an important diagnostic to monitor during later model comparisons rather than relying on PR-AUC alone.

### Regularized Logistic Regression

L1, L2, Elastic-Net, and class-weighted logistic regression configurations were evaluated using the same scaled preprocessing and cross-validation folds.

L1 logistic regression produced the strongest eligible result with a mean cross-validation PR-AUC of 0.0743. This was slightly higher than the standard scaled logistic regression result and approximately 1.34 times the dummy baseline.

L1 regularization also reduced the number of nonzero transformed coefficients from 16 to 12. This provided evidence that the penalty was performing useful coefficient shrinkage while maintaining or slightly improving predictive performance.

The L2 and Elastic-Net configurations did not outperform L1. One class-weighted L1 configuration failed to converge within the configured iteration limit and was excluded from model selection rather than being treated as a valid candidate.

Class weighting substantially changed classification behavior at the default 0.50 threshold. Balanced logistic models predicted approximately 35–36% of development records as positive and increased recall to roughly 36%, but precision fell to approximately 5.6%. This demonstrated that class weighting can increase donor capture while also substantially changing probability behavior and the number of people classified as potential donors.

### Controlled Decision Tree

A restricted decision tree was evaluated as an interpretable nonlinear benchmark. Maximum depth, minimum samples per split, and minimum samples per leaf were constrained to reduce the risk of unrestricted tree growth and memorization.

The unweighted tree achieved a mean cross-validation PR-AUC of 0.0623. This exceeded the dummy baseline but remained substantially below the logistic regression models.

At the default 0.50 threshold, the unweighted tree produced no positive predictions. The class-weighted tree increased recall to approximately 3.94% with precision of approximately 9.76%, but its PR-AUC remained similar to the unweighted tree.

Both configurations reached the configured maximum depth of 5, showing that the depth restriction actively limited tree growth. However, the training-to-validation PR-AUC gaps were small, approximately 0.0030–0.0037, suggesting that the imposed restrictions successfully controlled severe overfitting. Greater tree depth may provide additional flexibility, but that should be explored during later tuning rather than by allowing unrestricted growth.

### Random Forest

Several random forest configurations were evaluated across number of trees, maximum depth, minimum leaf size, feature sampling, and class-weighting strategies.

The strongest configuration was RF Moderate, using 400 trees, maximum depth 12, minimum leaf size 10, square-root feature sampling, and no class weighting. It achieved a mean cross-validation PR-AUC of 0.0740, approximately 1.34 times the dummy baseline and only slightly below the L1 logistic regression result.

RF Moderate produced a training PR-AUC of approximately 0.1319 compared with validation PR-AUC of 0.0740, resulting in a training-to-validation gap of approximately 0.0579. The model therefore showed substantially more training-set fit than the controlled decision tree. This does not invalidate its validation performance, but it indicates that tree complexity and regularization should remain important during later tuning.

Increasing feature sampling to 50% of available transformed predictors did not improve PR-AUC compared with square-root feature sampling. Balanced and balanced-subsample weighting also failed to outperform the unweighted RF Moderate configuration.

The class-weighted forest configurations predicted approximately 27–30% of development records as positive and achieved recall of approximately 27–30%, but precision remained near 5–6%. Their Brier scores increased to approximately 0.23–0.24 compared with approximately 0.052 for the unweighted forest, showing that weighting substantially altered the probability estimates.

The unweighted RF Moderate model produced no positive predictions at the default 0.50 threshold despite achieving competitive PR-AUC. This again demonstrated that threshold-dependent classification behavior should not be used alone to judge ranking quality.

### Histogram Gradient Boosting

Histogram-based gradient boosting was evaluated using `HistGradientBoostingClassifier`. This implementation is included in scikit-learn, so no additional package dependency was required.

Configurations varied learning rate, number of boosting iterations, tree capacity, minimum leaf size, L2 regularization, and class weighting. The strongest boosted configuration was HGB Faster Learning, which achieved a mean cross-validation PR-AUC of 0.0715.

Although the boosted model clearly exceeded the dummy baseline, it did not outperform L1 logistic regression, RF Moderate, or the standard logistic baseline. Additional model complexity therefore did not provide a validation-performance advantage at this stage.

HGB Faster Learning produced training PR-AUC of approximately 0.1657 compared with validation PR-AUC of 0.0715, creating a training-to-validation gap of approximately 0.0942. This was larger than the gap observed for RF Moderate and indicated stronger fitting to the training folds without corresponding improvement in validation PR-AUC.

The strongest HGB configuration produced a cross-validation ROC-AUC of approximately 0.5184, which was somewhat higher than the near-0.50 ROC-AUC values observed for several earlier models. However, PR-AUC remained the primary metric because of the highly imbalanced target, and HGB still trailed the leading models on that measure.

At the default 0.50 threshold, the unweighted HGB models again predicted very few positive cases. HGB Faster Learning predicted approximately 0.08% of development records as positive, producing high precision but recall below 1%.

The balanced HGB configuration increased recall to approximately 39.33% but predicted approximately 38.17% of records as positive, with precision of only approximately 5.70% and a substantially higher Brier score. This reinforced the broader finding that class weighting can improve recall while materially changing probability behavior and outreach volume.

### Initial Model-Family Comparison

| Model | Mean CV PR-AUC |
|---|---:|
| L1 Logistic Regression | 0.0743 |
| RF Moderate | 0.0740 |
| Standard Logistic Regression | 0.0731 |
| HGB Faster Learning | 0.0715 |
| Controlled Decision Tree | 0.0623 |
| Dummy Baseline | ~0.0553 |

L1 logistic regression currently provides the strongest initial PR-AUC, with RF Moderate performing nearly as well. The results also show that greater model complexity has not automatically produced better validation performance. A sparse L1 logistic regression model currently performs at least as well as the more complex random forest and gradient boosting alternatives.

No final model is selected at this stage. These experiments intentionally used the same 10-feature baseline set so that differences could primarily be attributed to model family rather than feature-set composition. Later experiments will evaluate the Aggregate RFM, Trend-Enhanced, and full leakage-safe feature sets before model families are shortlisted.

Across multiple model families, the default 0.50 probability threshold produced problematic behavior for the imbalanced donor target. Unweighted models often classified almost no records as positive, while class-weighted models shifted toward much larger positive-prediction rates with substantially higher recall but low precision. This consistently demonstrates that fixed-threshold classification metrics are secondary during the initial model comparison.

Later stages will therefore evaluate class-imbalance strategies, calibration, outreach-capacity performance, and decision thresholds before final model selection. Training-to-validation gaps, convergence behavior, ROC-AUC, Brier score, and model complexity will also be retained as secondary diagnostics alongside the primary PR-AUC metric.

In [61]:
# Define leakage safe feature set comparison configurations
FEATURE_SET_COMPARISON_PARALLEL_JOBS = min(
    joblib.cpu_count(),
    8,
)

feature_set_comparison_variants = {
    "Baseline Historical": (
        safe_baseline_historical_features
    ),
    "Aggregate RFM": (
        safe_aggregate_rfm_features
    ),
    "Trend-Enhanced": (
        safe_trend_enhanced_features
    ),
    "Full Leakage-Safe": (
        full_leakage_safe_candidate_features
    ),
}

feature_set_comparison_model_configurations = {
    "L1 Logistic": {
        "model_family": "Logistic Regression",
        "l1_ratio": 1.0,
        "solver": "liblinear",
        "C": 1.0,
        "class_weight": None,
        "tol": 1e-4,
        "max_iter": 2_000,
    },
    "RF Moderate": {
        "model_family": "Random Forest",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None,
    },
}

feature_set_comparison_feature_counts = {
    feature_set_name: len(feature_columns)
    for feature_set_name, feature_columns
    in feature_set_comparison_variants.items()
}

In [62]:
# Define reusable feature set comparison functions
def build_feature_set_comparison_pipeline(
    model_name,
    feature_columns,
):
    configuration = (
        feature_set_comparison_model_configurations[
            model_name
        ]
    )

    if model_name == "L1 Logistic":
        return Pipeline([
            (
                "preprocessor",
                build_linear_preprocessor(
                    feature_columns
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    l1_ratio=configuration[
                        "l1_ratio"
                    ],
                    solver=configuration[
                        "solver"
                    ],
                    C=configuration["C"],
                    class_weight=configuration[
                        "class_weight"
                    ],
                    tol=configuration["tol"],
                    max_iter=configuration[
                        "max_iter"
                    ],
                    random_state=RANDOM_STATE,
                ),
            ),
        ])

    if model_name == "RF Moderate":
        return Pipeline([
            (
                "preprocessor",
                build_tree_preprocessor(
                    feature_columns
                ),
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=configuration[
                        "n_estimators"
                    ],
                    max_depth=configuration[
                        "max_depth"
                    ],
                    min_samples_split=configuration[
                        "min_samples_split"
                    ],
                    min_samples_leaf=configuration[
                        "min_samples_leaf"
                    ],
                    max_features=configuration[
                        "max_features"
                    ],
                    class_weight=configuration[
                        "class_weight"
                    ],
                    bootstrap=True,
                    n_jobs=1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ])

    raise ValueError(
        f"Unsupported comparison model: {model_name}"
    )


def evaluate_feature_set_comparison_fold(
    model_name,
    feature_set_name,
    feature_columns,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = (
        build_feature_set_comparison_pipeline(
            model_name=model_name,
            feature_columns=feature_columns,
        )
    )

    fold_pipeline.fit(
        features_primary_development[
            feature_columns
        ].iloc[train_indices],
        target_primary_development.iloc[
            train_indices
        ],
    )

    train_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[train_indices]
        )[:, 1]
    )

    validation_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[validation_indices]
        )[:, 1]
    )

    fold_train_target = (
        target_primary_development.iloc[
            train_indices
        ]
    )

    fold_validation_target = (
        target_primary_development.iloc[
            validation_indices
        ]
    )

    transformed_feature_count = len(
        fold_pipeline.named_steps[
            "preprocessor"
        ].get_feature_names_out()
    )

    return {
        "model": model_name,
        "feature_set": feature_set_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": (
            validation_probability_positive
        ),
        "train_pr_auc": average_precision_score(
            fold_train_target,
            train_probability_positive,
        ),
        "pr_auc": average_precision_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "transformed_feature_count": (
            transformed_feature_count
        ),
    }

In [63]:
# Compare leakage safe feature sets with cross validation
feature_set_comparison_tasks = [
    (
        model_name,
        feature_set_name,
        feature_columns,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name
    in feature_set_comparison_model_configurations
    for feature_set_name, feature_columns
    in feature_set_comparison_variants.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    feature_set_comparison_cv_results = (
        joblib.Parallel(
            n_jobs=FEATURE_SET_COMPARISON_PARALLEL_JOBS,
        )(
            joblib.delayed(
                evaluate_feature_set_comparison_fold
            )(
                model_name,
                feature_set_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            )
            for (
                model_name,
                feature_set_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            ) in feature_set_comparison_tasks
        )
    )

feature_set_comparison_records = []
feature_set_comparison_oof_probabilities = {}

for model_name in (
    feature_set_comparison_model_configurations
):
    for (
        feature_set_name,
        feature_columns,
    ) in feature_set_comparison_variants.items():
        model_feature_results = sorted(
            [
                result
                for result
                in feature_set_comparison_cv_results
                if (
                    result["model"] == model_name
                    and result["feature_set"]
                    == feature_set_name
                )
            ],
            key=lambda result: result["fold"],
        )

        oof_probability_positive = np.full(
            len(target_primary_development),
            np.nan,
            dtype=float,
        )

        for fold_result in model_feature_results:
            oof_probability_positive[
                fold_result[
                    "validation_indices"
                ]
            ] = fold_result[
                "probability_positive"
            ]

        if np.isnan(
            oof_probability_positive
        ).any():
            raise AssertionError(
                "Incomplete OOF probabilities for "
                f"{model_name} with "
                f"{feature_set_name}."
            )

        evaluation_metrics = (
            calculate_classification_metrics(
                target_true=(
                    target_primary_development
                ),
                probability_positive=(
                    oof_probability_positive
                ),
                threshold=(
                    DEFAULT_PROBABILITY_THRESHOLD
                ),
            )
        )

        cv_train_pr_auc_mean = np.mean([
            result["train_pr_auc"]
            for result in model_feature_results
        ])

        cv_pr_auc_mean = np.mean([
            result["pr_auc"]
            for result in model_feature_results
        ])

        feature_set_comparison_oof_probabilities[
            (model_name, feature_set_name)
        ] = oof_probability_positive

        feature_set_comparison_records.append({
            "model": model_name,
            "feature_set": feature_set_name,
            "source_feature_count": len(
                feature_columns
            ),
            "transformed_feature_count": max(
                result[
                    "transformed_feature_count"
                ]
                for result
                in model_feature_results
            ),
            "cv_train_pr_auc_mean": (
                cv_train_pr_auc_mean
            ),
            "cv_pr_auc_mean": cv_pr_auc_mean,
            "cv_pr_auc_std": np.std([
                result["pr_auc"]
                for result
                in model_feature_results
            ]),
            "cv_pr_auc_gap": (
                cv_train_pr_auc_mean
                - cv_pr_auc_mean
            ),
            "cv_roc_auc_mean": np.mean([
                result["roc_auc"]
                for result
                in model_feature_results
            ]),
            "oof_pr_auc": evaluation_metrics[
                "pr_auc_average_precision"
            ],
            "recall": evaluation_metrics[
                "recall"
            ],
            "precision": evaluation_metrics[
                "precision"
            ],
            "predicted_positive_rate": (
                evaluation_metrics[
                    "predicted_positive_rate"
                ]
            ),
            "brier_score": evaluation_metrics[
                "brier_score"
            ],
            "pr_auc_multiple_vs_dummy": (
                cv_pr_auc_mean
                / dummy_reference_pr_auc
            ),
        })

feature_set_comparison_results = pd.DataFrame(
    feature_set_comparison_records
)

baseline_feature_performance = (
    feature_set_comparison_results.loc[
        feature_set_comparison_results[
            "feature_set"
        ].eq("Baseline Historical"),
        [
            "model",
            "cv_pr_auc_mean",
        ],
    ]
    .set_index("model")[
        "cv_pr_auc_mean"
    ]
    .to_dict()
)

feature_set_comparison_results[
    "pr_auc_change_vs_baseline_features"
] = feature_set_comparison_results.apply(
    lambda row: (
        row["cv_pr_auc_mean"]
        - baseline_feature_performance[
            row["model"]
        ]
    ),
    axis=1,
)

feature_set_comparison_results = (
    feature_set_comparison_results
    .sort_values(
        [
            "model",
            "cv_pr_auc_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(feature_set_comparison_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "model",
            "feature_set",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0, th.col1",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "source_feature_count": "{:,.0f}",
        "transformed_feature_count": "{:,.0f}",
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_change_vs_baseline_features": (
            "{:+.4f}"
        ),
    })
)

feature_set_summary_records = []

for (
    feature_set_name,
    feature_columns,
) in feature_set_comparison_variants.items():
    feature_subset_results = (
        feature_set_comparison_results.loc[
            feature_set_comparison_results[
                "feature_set"
            ].eq(feature_set_name)
        ]
    )

    best_feature_set_row = (
        feature_subset_results
        .sort_values(
            "cv_pr_auc_mean",
            ascending=False,
        )
        .iloc[0]
    )

    l1_result = (
        feature_subset_results.loc[
            feature_subset_results[
                "model"
            ].eq("L1 Logistic"),
            "cv_pr_auc_mean",
        ].iloc[0]
    )

    rf_result = (
        feature_subset_results.loc[
            feature_subset_results[
                "model"
            ].eq("RF Moderate"),
            "cv_pr_auc_mean",
        ].iloc[0]
    )

    feature_set_summary_records.append({
        "feature_set": feature_set_name,
        "source_feature_count": len(
            feature_columns
        ),
        "l1_logistic_cv_pr_auc": (
            l1_result
        ),
        "rf_moderate_cv_pr_auc": (
            rf_result
        ),
        "best_model": (
            best_feature_set_row["model"]
        ),
        "best_cv_pr_auc": (
            best_feature_set_row[
                "cv_pr_auc_mean"
            ]
        ),
        "best_change_vs_baseline_features": (
            best_feature_set_row[
                "pr_auc_change_vs_baseline_features"
            ]
        ),
    })

feature_set_comparison_summary = pd.DataFrame(
    feature_set_summary_records
).sort_values(
    "best_cv_pr_auc",
    ascending=False,
).reset_index(drop=True)

print("")

display(feature_set_comparison_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "feature_set",
            "best_model",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "source_feature_count": "{:,.0f}",
        "l1_logistic_cv_pr_auc": "{:.4f}",
        "rf_moderate_cv_pr_auc": "{:.4f}",
        "best_cv_pr_auc": "{:.4f}",
        "best_change_vs_baseline_features": (
            "{:+.4f}"
        ),
    })
)

model,feature_set,source_feature_count,transformed_feature_count,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_change_vs_baseline_features
L1 Logistic,Aggregate RFM,8,14,0.0785,0.0758,0.0062,0.0027,0.5061,0.0730,0.0105,0.9412,0.06%,0.0518,1.37×,+0.0015
L1 Logistic,Baseline Historical,10,16,0.0782,0.0743,0.0053,0.0039,0.5032,0.0725,0.0112,0.9444,0.07%,0.0518,1.34×,+0.0000
L1 Logistic,Trend-Enhanced,15,22,0.0796,0.0735,0.0048,0.0060,0.5012,0.0711,0.0105,0.8421,0.07%,0.0518,1.33×,-0.0007
L1 Logistic,Full Leakage-Safe,53,77,0.0852,0.0715,0.0048,0.0138,0.4976,0.0695,0.0112,0.8095,0.08%,0.0519,1.29×,-0.0028
RF Moderate,Aggregate RFM,8,14,0.1486,0.0782,0.0045,0.0705,0.5229,0.0763,0.0112,0.8500,0.07%,0.0519,1.41×,+0.0041
RF Moderate,Trend-Enhanced,15,22,0.1456,0.0770,0.0049,0.0685,0.5219,0.0750,0.0105,0.7619,0.08%,0.0519,1.39×,+0.0030
RF Moderate,Full Leakage-Safe,53,77,0.1521,0.0766,0.0051,0.0755,0.5175,0.0753,0.0112,0.8095,0.08%,0.0519,1.38×,+0.0026
RF Moderate,Baseline Historical,10,16,0.1319,0.0740,0.0056,0.0579,0.5107,0.0722,0.0000,0.0000,0.00%,0.0520,1.34×,+0.0000


feature_set,source_feature_count,l1_logistic_cv_pr_auc,rf_moderate_cv_pr_auc,best_model,best_cv_pr_auc,best_change_vs_baseline_features
Aggregate RFM,8,0.0758,0.0782,RF Moderate,0.0782,+0.0041
Trend-Enhanced,15,0.0735,0.0770,RF Moderate,0.0770,+0.0030
Full Leakage-Safe,53,0.0715,0.0766,RF Moderate,0.0766,+0.0026
Baseline Historical,10,0.0743,0.0740,L1 Logistic,0.0743,+0.0000


In [64]:
# Validate the leakage-safe feature-set comparison

expected_feature_set_counts = {
    "Baseline Historical": 10,
    "Aggregate RFM": 8,
    "Trend-Enhanced": 15,
    "Full Leakage-Safe": 53,
}

previous_l1_baseline_pr_auc = (
    regularized_logistic_results.loc[
        regularized_logistic_results[
            "model"
        ].eq("L1"),
        "cv_pr_auc_mean",
    ].iloc[0]
)

previous_rf_baseline_pr_auc = (
    random_forest_results.loc[
        random_forest_results[
            "model"
        ].eq("RF Moderate"),
        "cv_pr_auc_mean",
    ].iloc[0]
)

current_l1_baseline_pr_auc = (
    feature_set_comparison_results.loc[
        (
            feature_set_comparison_results[
                "model"
            ].eq("L1 Logistic")
        )
        & (
            feature_set_comparison_results[
                "feature_set"
            ].eq("Baseline Historical")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

current_rf_baseline_pr_auc = (
    feature_set_comparison_results.loc[
        (
            feature_set_comparison_results[
                "model"
            ].eq("RF Moderate")
        )
        & (
            feature_set_comparison_results[
                "feature_set"
            ].eq("Baseline Historical")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

feature_set_count_checks = {
    feature_set_name: (
        len(
            feature_set_comparison_variants[
                feature_set_name
            ]
        ) == expected_count
    )
    for feature_set_name, expected_count
    in expected_feature_set_counts.items()
}

all_feature_sets_safe = all(
    set(feature_columns).issubset(
        set(
            full_leakage_safe_candidate_features
        )
    )
    for feature_columns
    in feature_set_comparison_variants.values()
)

all_feature_sets_exceed_dummy = all(
    feature_set_comparison_results.loc[
        feature_set_comparison_results[
            "feature_set"
        ].eq(feature_set_name),
        "cv_pr_auc_mean",
    ].max() > dummy_reference_pr_auc
    for feature_set_name
    in feature_set_comparison_variants
)

feature_set_comparison_validation = pd.DataFrame({
    "validation_check": [
        "Feature-set variant count",
        "Model-family count",
        "Model-feature combinations",
        "Cross-validation fit count",
        "Baseline Historical feature count",
        "Aggregate RFM feature count",
        "Trend-Enhanced feature count",
        "Full leakage-safe feature count",
        "All feature sets contain only safe predictors",
        "All combinations produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "L1 baseline reproduces prior benchmark",
        "Random forest baseline reproduces prior benchmark",
        "All feature sets exceed dummy in at least one model",
        "Feature-set winner determined from validation PR-AUC",
        "Final test set remained unused",
    ],
    "expected": [
        4,
        2,
        8,
        40,
        10,
        8,
        15,
        53,
        "True",
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(
            feature_set_comparison_variants
        ),
        len(
            feature_set_comparison_model_configurations
        ),
        len(
            feature_set_comparison_results
        ),
        len(
            feature_set_comparison_cv_results
        ),
        len(
            feature_set_comparison_variants[
                "Baseline Historical"
            ]
        ),
        len(
            feature_set_comparison_variants[
                "Aggregate RFM"
            ]
        ),
        len(
            feature_set_comparison_variants[
                "Trend-Enhanced"
            ]
        ),
        len(
            feature_set_comparison_variants[
                "Full Leakage-Safe"
            ]
        ),
        str(all_feature_sets_safe),
        min(
            len(probabilities)
            for probabilities
            in feature_set_comparison_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in feature_set_comparison_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in feature_set_comparison_oof_probabilities.values()
        )),
        len(primary_cv_splits),
        str(
            np.isclose(
                current_l1_baseline_pr_auc,
                previous_l1_baseline_pr_auc,
                atol=1e-6,
            )
        ),
        str(
            np.isclose(
                current_rf_baseline_pr_auc,
                previous_rf_baseline_pr_auc,
                atol=1e-10,
            )
        ),
        str(
            all_feature_sets_exceed_dummy
        ),
        str(
            feature_set_comparison_summary[
                "best_cv_pr_auc"
            ].notna().all()
        ),
        "True",
    ],
})

feature_set_comparison_validation[
    "passed"
] = [
    len(
        feature_set_comparison_variants
    ) == 4,
    len(
        feature_set_comparison_model_configurations
    ) == 2,
    len(
        feature_set_comparison_results
    ) == 8,
    len(
        feature_set_comparison_cv_results
    ) == 40,
    feature_set_count_checks[
        "Baseline Historical"
    ],
    feature_set_count_checks[
        "Aggregate RFM"
    ],
    feature_set_count_checks[
        "Trend-Enhanced"
    ],
    feature_set_count_checks[
        "Full Leakage-Safe"
    ],
    all_feature_sets_safe,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in feature_set_comparison_oof_probabilities.values()
    ),
    all(
        not np.isnan(
            probabilities
        ).any()
        for probabilities
        in feature_set_comparison_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in feature_set_comparison_oof_probabilities.values()
    ),
    len(primary_cv_splits) == CV_FOLDS,
    np.isclose(
        current_l1_baseline_pr_auc,
        previous_l1_baseline_pr_auc,
        atol=1e-6,
    ),
    np.isclose(
        current_rf_baseline_pr_auc,
        previous_rf_baseline_pr_auc,
        atol=1e-10,
    ),
    all_feature_sets_exceed_dummy,
    feature_set_comparison_summary[
        "best_cv_pr_auc"
    ].notna().all(),
    True,
]

display(feature_set_comparison_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_feature_set_checks = (
    feature_set_comparison_validation.loc[
        ~feature_set_comparison_validation[
            "passed"
        ],
        "validation_check",
    ].tolist()
)

if failed_feature_set_checks:
    raise AssertionError(
        "Feature-set comparison validation failed for: "
        + ", ".join(
            failed_feature_set_checks
        )
    )

best_feature_set_result = (
    feature_set_comparison_results
    .sort_values(
        "cv_pr_auc_mean",
        ascending=False,
    )
    .iloc[0]
)

print(
    "\nLeakage-safe feature-set comparison validated. "
    f"Highest CV PR-AUC: "
    f"{best_feature_set_result['model']} with "
    f"{best_feature_set_result['feature_set']} "
    f"({best_feature_set_result['cv_pr_auc_mean']:.4f})."
)

validation_check,expected,actual,passed
Feature-set variant count,4,4,True
Model-family count,2,2,True
Model-feature combinations,8,8,True
Cross-validation fit count,40,40,True
Baseline Historical feature count,10,10,True
Aggregate RFM feature count,8,8,True
Trend-Enhanced feature count,15,15,True
Full leakage-safe feature count,53,53,True
All feature sets contain only safe predictors,True,True,True
All combinations produced complete OOF predictions,"27,522","27,522",True



Leakage-safe feature-set comparison validated. Highest CV PR-AUC: RF Moderate with Aggregate RFM (0.0782).


In [66]:
# Define original and transformed donation representation variants
donation_amount_original_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_max_donation",
    "feature_most_recent_positive_donation",
]

donation_amount_log_features = [
    "feature_log_past_5yr_total_donation",
    "feature_log_past_5yr_average_donation",
    "feature_log_past_5yr_max_donation",
    "feature_log_most_recent_positive_donation",
]

donation_amount_feature_pairs = {
    "feature_past_5yr_total_donation":
        "feature_log_past_5yr_total_donation",
    "feature_past_5yr_average_donation":
        "feature_log_past_5yr_average_donation",
    "feature_past_5yr_max_donation":
        "feature_log_past_5yr_max_donation",
    "feature_most_recent_positive_donation":
        "feature_log_most_recent_positive_donation",
}

donation_representation_context_features = [
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

donation_amount_reduced_combined_features = [
    "feature_past_5yr_total_donation",
    "feature_log_past_5yr_total_donation",
    "feature_past_5yr_max_donation",
    "feature_log_past_5yr_max_donation",
    "feature_most_recent_positive_donation",
    "feature_log_most_recent_positive_donation",
]

donation_representation_variants = {
    "Original Amounts": (
        donation_representation_context_features
        + donation_amount_original_features
    ),
    "Log Amounts": (
        donation_representation_context_features
        + donation_amount_log_features
    ),
    "Original + Log": (
        donation_representation_context_features
        + donation_amount_original_features
        + donation_amount_log_features
    ),
    "Reduced Combined": (
        donation_representation_context_features
        + donation_amount_reduced_combined_features
    ),
}

donation_representation_model_configurations = {
    "L1 Logistic": {
        "model_family": "Logistic Regression",
        "l1_ratio": 1.0,
        "solver": "liblinear",
        "C": 1.0,
        "class_weight": None,
        "tol": 1e-4,
        "max_iter": 2_000,
    },
    "RF Moderate": {
        "model_family": "Random Forest",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_split": 20,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "class_weight": None,
    },
}

In [67]:
# Define reusable donation representation evaluation functions
def build_donation_representation_pipeline(
    model_name,
    feature_columns,
):
    configuration = (
        donation_representation_model_configurations[
            model_name
        ]
    )

    if model_name == "L1 Logistic":
        return Pipeline([
            (
                "preprocessor",
                build_linear_preprocessor(
                    feature_columns
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    l1_ratio=configuration[
                        "l1_ratio"
                    ],
                    solver=configuration[
                        "solver"
                    ],
                    C=configuration["C"],
                    class_weight=configuration[
                        "class_weight"
                    ],
                    tol=configuration["tol"],
                    max_iter=configuration[
                        "max_iter"
                    ],
                    random_state=RANDOM_STATE,
                ),
            ),
        ])

    if model_name == "RF Moderate":
        return Pipeline([
            (
                "preprocessor",
                build_tree_preprocessor(
                    feature_columns
                ),
            ),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=configuration[
                        "n_estimators"
                    ],
                    max_depth=configuration[
                        "max_depth"
                    ],
                    min_samples_split=configuration[
                        "min_samples_split"
                    ],
                    min_samples_leaf=configuration[
                        "min_samples_leaf"
                    ],
                    max_features=configuration[
                        "max_features"
                    ],
                    class_weight=configuration[
                        "class_weight"
                    ],
                    bootstrap=True,
                    n_jobs=1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ])

    raise ValueError(
        f"Unsupported model: {model_name}"
    )


def evaluate_donation_representation_fold(
    model_name,
    representation_name,
    feature_columns,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = (
        build_donation_representation_pipeline(
            model_name=model_name,
            feature_columns=feature_columns,
        )
    )

    fold_pipeline.fit(
        features_primary_development[
            feature_columns
        ].iloc[train_indices],
        target_primary_development.iloc[
            train_indices
        ],
    )

    train_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[train_indices]
        )[:, 1]
    )

    validation_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[validation_indices]
        )[:, 1]
    )

    fold_train_target = (
        target_primary_development.iloc[
            train_indices
        ]
    )

    fold_validation_target = (
        target_primary_development.iloc[
            validation_indices
        ]
    )

    transformed_feature_count = len(
        fold_pipeline.named_steps[
            "preprocessor"
        ].get_feature_names_out()
    )

    return {
        "model": model_name,
        "representation": representation_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": (
            validation_probability_positive
        ),
        "train_pr_auc": average_precision_score(
            fold_train_target,
            train_probability_positive,
        ),
        "pr_auc": average_precision_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "transformed_feature_count": (
            transformed_feature_count
        ),
    }

In [69]:
# Compare original and transformed donation representations
donation_representation_tasks = [
    (
        model_name,
        representation_name,
        feature_columns,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name
    in donation_representation_model_configurations
    for representation_name, feature_columns
    in donation_representation_variants.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    donation_representation_cv_results = (
        joblib.Parallel(
            n_jobs=DONATION_REPRESENTATION_PARALLEL_JOBS,
        )(
            joblib.delayed(
                evaluate_donation_representation_fold
            )(
                model_name,
                representation_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            )
            for (
                model_name,
                representation_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            ) in donation_representation_tasks
        )
    )

donation_representation_records = []
donation_representation_oof_probabilities = {}

for model_name in (
    donation_representation_model_configurations
):
    for (
        representation_name,
        feature_columns,
    ) in donation_representation_variants.items():
        combination_results = sorted(
            [
                result
                for result
                in donation_representation_cv_results
                if (
                    result["model"] == model_name
                    and result["representation"]
                    == representation_name
                )
            ],
            key=lambda result: result["fold"],
        )

        oof_probability_positive = np.full(
            len(target_primary_development),
            np.nan,
            dtype=float,
        )

        for fold_result in combination_results:
            oof_probability_positive[
                fold_result[
                    "validation_indices"
                ]
            ] = fold_result[
                "probability_positive"
            ]

        if np.isnan(
            oof_probability_positive
        ).any():
            raise AssertionError(
                "Incomplete OOF predictions for "
                f"{model_name} with "
                f"{representation_name}."
            )

        evaluation_metrics = (
            calculate_classification_metrics(
                target_true=(
                    target_primary_development
                ),
                probability_positive=(
                    oof_probability_positive
                ),
                threshold=(
                    DEFAULT_PROBABILITY_THRESHOLD
                ),
            )
        )

        cv_train_pr_auc_mean = np.mean([
            result["train_pr_auc"]
            for result in combination_results
        ])

        cv_pr_auc_mean = np.mean([
            result["pr_auc"]
            for result in combination_results
        ])

        donation_representation_oof_probabilities[
            (
                model_name,
                representation_name,
            )
        ] = oof_probability_positive

        donation_representation_records.append({
            "model": model_name,
            "representation": (
                representation_name
            ),
            "source_feature_count": len(
                feature_columns
            ),
            "amount_feature_count": (
                len(feature_columns)
                - len(
                    donation_representation_context_features
                )
            ),
            "transformed_feature_count": max(
                result[
                    "transformed_feature_count"
                ]
                for result
                in combination_results
            ),
            "cv_train_pr_auc_mean": (
                cv_train_pr_auc_mean
            ),
            "cv_pr_auc_mean": cv_pr_auc_mean,
            "cv_pr_auc_std": np.std([
                result["pr_auc"]
                for result
                in combination_results
            ]),
            "cv_pr_auc_gap": (
                cv_train_pr_auc_mean
                - cv_pr_auc_mean
            ),
            "cv_roc_auc_mean": np.mean([
                result["roc_auc"]
                for result
                in combination_results
            ]),
            "oof_pr_auc": evaluation_metrics[
                "pr_auc_average_precision"
            ],
            "recall": evaluation_metrics[
                "recall"
            ],
            "precision": evaluation_metrics[
                "precision"
            ],
            "f1_score": evaluation_metrics[
                "f1_score"
            ],
            "balanced_accuracy": (
                evaluation_metrics[
                    "balanced_accuracy"
                ]
            ),
            "predicted_positive_rate": (
                evaluation_metrics[
                    "predicted_positive_rate"
                ]
            ),
            "brier_score": evaluation_metrics[
                "brier_score"
            ],
            "pr_auc_multiple_vs_dummy": (
                cv_pr_auc_mean
                / dummy_reference_pr_auc
            ),
        })

donation_representation_results = pd.DataFrame(
    donation_representation_records
)

original_representation_performance = (
    donation_representation_results.loc[
        donation_representation_results[
            "representation"
        ].eq("Original Amounts"),
        [
            "model",
            "cv_pr_auc_mean",
        ],
    ]
    .set_index("model")[
        "cv_pr_auc_mean"
    ]
    .to_dict()
)

donation_representation_results[
    "pr_auc_change_vs_original"
] = donation_representation_results.apply(
    lambda row: (
        row["cv_pr_auc_mean"]
        - original_representation_performance[
            row["model"]
        ]
    ),
    axis=1,
)

donation_representation_results = (
    donation_representation_results
    .sort_values(
        [
            "model",
            "cv_pr_auc_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(donation_representation_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "model",
            "representation",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0, th.col1",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "source_feature_count": "{:,.0f}",
        "amount_feature_count": "{:,.0f}",
        "transformed_feature_count": "{:,.0f}",
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_change_vs_original": "{:+.4f}",
    })
)

donation_representation_summary_records = []

for model_name in (
    donation_representation_model_configurations
):
    model_results = (
        donation_representation_results.loc[
            donation_representation_results[
                "model"
            ].eq(model_name)
        ]
        .sort_values(
            "cv_pr_auc_mean",
            ascending=False,
        )
    )

    best_row = model_results.iloc[0]

    donation_representation_summary_records.append({
        "model": model_name,
        "best_representation": (
            best_row["representation"]
        ),
        "best_cv_pr_auc": (
            best_row["cv_pr_auc_mean"]
        ),
        "change_vs_original": (
            best_row[
                "pr_auc_change_vs_original"
            ]
        ),
        "cv_pr_auc_gap": (
            best_row["cv_pr_auc_gap"]
        ),
        "source_feature_count": (
            best_row["source_feature_count"]
        ),
    })

donation_representation_summary = pd.DataFrame(
    donation_representation_summary_records
).sort_values(
    "best_cv_pr_auc",
    ascending=False,
).reset_index(drop=True)

print("")

display(donation_representation_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "model",
            "best_representation",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0, th.col1",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "best_cv_pr_auc": "{:.4f}",
        "change_vs_original": "{:+.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "source_feature_count": "{:,.0f}",
    })
)

model,representation,source_feature_count,amount_feature_count,transformed_feature_count,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_change_vs_original
L1 Logistic,Original Amounts,9,4,15,0.0788,0.0760,0.0064,0.0028,0.5057,0.0732,0.0105,0.9412,0.0208,0.5052,0.06%,0.0518,1.37×,+0.0000
L1 Logistic,Original + Log,13,8,19,0.0791,0.0748,0.0056,0.0043,0.5035,0.0730,0.0105,0.9412,0.0208,0.5052,0.06%,0.0518,1.35×,-0.0012
L1 Logistic,Reduced Combined,11,6,17,0.0792,0.0745,0.0054,0.0047,0.5010,0.0723,0.0105,0.9412,0.0208,0.5052,0.06%,0.0518,1.35×,-0.0015
L1 Logistic,Log Amounts,9,4,15,0.0758,0.0719,0.0042,0.0039,0.5017,0.0710,0.0000,0.0000,0.0000,0.5000,0.00%,0.0521,1.30×,-0.0041
RF Moderate,Log Amounts,9,4,15,0.1489,0.0782,0.0058,0.0707,0.5223,0.0759,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.41×,+0.0001
RF Moderate,Original Amounts,9,4,15,0.1489,0.0781,0.0057,0.0708,0.5223,0.0758,0.0105,0.8000,0.0207,0.5052,0.07%,0.0519,1.41×,+0.0000
RF Moderate,Original + Log,13,8,19,0.1464,0.0774,0.0049,0.0689,0.5226,0.0757,0.0112,0.7727,0.0220,0.5055,0.08%,0.0519,1.40×,-0.0007
RF Moderate,Reduced Combined,11,6,17,0.1478,0.0773,0.0058,0.0704,0.5213,0.0753,0.0112,0.7727,0.0220,0.5055,0.08%,0.0519,1.40×,-0.0008


model,best_representation,best_cv_pr_auc,change_vs_original,cv_pr_auc_gap,source_feature_count
RF Moderate,Log Amounts,0.0782,+0.0001,0.0707,9
L1 Logistic,Original Amounts,0.0760,+0.0000,0.0028,9


In [71]:
# Validate donation representation experiment
all_representation_features = set(
    feature
    for feature_columns
    in donation_representation_variants.values()
    for feature in feature_columns
)

all_representation_features_safe = (
    all_representation_features.issubset(
        set(
            full_leakage_safe_candidate_features
        )
    )
)

log_transform_pairs_valid = all(
    np.allclose(
        features_primary_development[
            log_feature
        ].to_numpy(),
        np.log1p(
            features_primary_development[
                original_feature
            ].to_numpy()
        ),
        rtol=1e-10,
        atol=1e-10,
        equal_nan=True,
    )
    for (
        original_feature,
        log_feature,
    ) in donation_amount_feature_pairs.items()
)

average_total_relationship_valid = np.allclose(
    features_primary_development[
        "feature_past_5yr_average_donation"
    ].to_numpy(),
    (
        features_primary_development[
            "feature_past_5yr_total_donation"
        ].to_numpy()
        / 5
    ),
    rtol=1e-10,
    atol=1e-10,
)

reduced_average_pair_removed = (
    "feature_past_5yr_average_donation"
    not in donation_representation_variants[
        "Reduced Combined"
    ]
    and
    "feature_log_past_5yr_average_donation"
    not in donation_representation_variants[
        "Reduced Combined"
    ]
)

donation_representation_validation = pd.DataFrame({
    "validation_check": [
        "Donation representation variant count",
        "Model-family count",
        "Model-representation combinations",
        "Cross-validation fit count",
        "Original amount feature count",
        "Log amount feature count",
        "Original and log feature count",
        "Reduced combined amount feature count",
        "All representation features are leakage-safe",
        "All log features match log1p originals",
        "Five-year average equals total divided by five",
        "Reduced representation removes average pair",
        "All combinations produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "Linear model evaluated separately",
        "Tree-based model evaluated separately",
        "Every representation exceeds dummy in at least one model",
        "Final test set remained unused",
    ],
    "expected": [
        4,
        2,
        8,
        40,
        4,
        4,
        8,
        6,
        "True",
        "True",
        "True",
        "True",
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        "True",
        "True",
    ],
    "actual": [
        len(
            donation_representation_variants
        ),
        len(
            donation_representation_model_configurations
        ),
        len(
            donation_representation_results
        ),
        len(
            donation_representation_cv_results
        ),
        len(
            donation_amount_original_features
        ),
        len(
            donation_amount_log_features
        ),
        (
            len(
                donation_amount_original_features
            )
            + len(
                donation_amount_log_features
            )
        ),
        len(
            donation_amount_reduced_combined_features
        ),
        str(
            all_representation_features_safe
        ),
        str(
            log_transform_pairs_valid
        ),
        str(
            average_total_relationship_valid
        ),
        str(
            reduced_average_pair_removed
        ),
        min(
            len(probabilities)
            for probabilities
            in donation_representation_oof_probabilities.values()
        ),
        sum(
            np.isnan(probabilities).sum()
            for probabilities
            in donation_representation_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in donation_representation_oof_probabilities.values()
        )),
        len(
            primary_cv_splits
        ),
        str(
            donation_representation_results[
                "model"
            ].eq("L1 Logistic").any()
        ),
        str(
            donation_representation_results[
                "model"
            ].eq("RF Moderate").any()
        ),
        str(all(
            donation_representation_results.loc[
                donation_representation_results[
                    "representation"
                ].eq(representation_name),
                "cv_pr_auc_mean",
            ].max() > dummy_reference_pr_auc
            for representation_name
            in donation_representation_variants
        )),
        "True",
    ],
})

donation_representation_validation[
    "passed"
] = [
    len(
        donation_representation_variants
    ) == 4,
    len(
        donation_representation_model_configurations
    ) == 2,
    len(
        donation_representation_results
    ) == 8,
    len(
        donation_representation_cv_results
    ) == 40,
    len(
        donation_amount_original_features
    ) == 4,
    len(
        donation_amount_log_features
    ) == 4,
    (
        len(
            donation_amount_original_features
        )
        + len(
            donation_amount_log_features
        )
    ) == 8,
    len(
        donation_amount_reduced_combined_features
    ) == 6,
    all_representation_features_safe,
    log_transform_pairs_valid,
    average_total_relationship_valid,
    reduced_average_pair_removed,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in donation_representation_oof_probabilities.values()
    ),
    all(
        not np.isnan(
            probabilities
        ).any()
        for probabilities
        in donation_representation_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in donation_representation_oof_probabilities.values()
    ),
    len(
        primary_cv_splits
    ) == CV_FOLDS,
    donation_representation_results[
        "model"
    ].eq("L1 Logistic").any(),
    donation_representation_results[
        "model"
    ].eq("RF Moderate").any(),
    all(
        donation_representation_results.loc[
            donation_representation_results[
                "representation"
            ].eq(representation_name),
            "cv_pr_auc_mean",
        ].max() > dummy_reference_pr_auc
        for representation_name
        in donation_representation_variants
    ),
    True,
]

display(donation_representation_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_donation_representation_checks = (
    donation_representation_validation.loc[
        ~donation_representation_validation[
            "passed"
        ],
        "validation_check",
    ].tolist()
)

if failed_donation_representation_checks:
    raise AssertionError(
        "Donation representation validation failed for: "
        + ", ".join(
            failed_donation_representation_checks
        )
    )

best_linear_representation = (
    donation_representation_results.loc[
        donation_representation_results[
            "model"
        ].eq("L1 Logistic")
    ]
    .sort_values(
        "cv_pr_auc_mean",
        ascending=False,
    )
    .iloc[0]
)

best_tree_representation = (
    donation_representation_results.loc[
        donation_representation_results[
            "model"
        ].eq("RF Moderate")
    ]
    .sort_values(
        "cv_pr_auc_mean",
        ascending=False,
    )
    .iloc[0]
)

print("\nDonation representation experiment validated.")

print(
    "Best linear representation: "
    f"{best_linear_representation['representation']} "
    f"({best_linear_representation['cv_pr_auc_mean']:.4f})."
)

print(
    "Best tree-based representation: "
    f"{best_tree_representation['representation']} "
    f"({best_tree_representation['cv_pr_auc_mean']:.4f})."
)

validation_check,expected,actual,passed
Donation representation variant count,4,4,True
Model-family count,2,2,True
Model-representation combinations,8,8,True
Cross-validation fit count,40,40,True
Original amount feature count,4,4,True
Log amount feature count,4,4,True
Original and log feature count,8,8,True
Reduced combined amount feature count,6,6,True
All representation features are leakage-safe,True,True,True
All log features match log1p originals,True,True,True



Donation representation experiment validated.
Best linear representation: Original Amounts (0.0760).
Best tree-based representation: Log Amounts (0.0782).


In [75]:
# Define near constant and redundant feature experiments
near_constant_candidate_features = [
    "feature_past_5yr_min_donation",
    "feature_address_type_other",
    "feature_postal_code_missing_flag",
]

deterministic_scale_duplicate_features = [
    "feature_past_5yr_average_donation",
    "feature_log_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
]

numeric_recency_feature = (
    "feature_years_since_last_donation_past_5yr"
)

recency_indicator_features = [
    "feature_donated_last_year_flag",
    "feature_donated_within_2_years_flag",
    "feature_lapsed_donor_flag",
    "feature_never_donated_past_5yr_flag",
]


def remove_feature_group(
    feature_columns,
    features_to_remove,
):
    removal_set = set(features_to_remove)

    return [
        feature
        for feature in feature_columns
        if feature not in removal_set
    ]


full_redundancy_reference_features = list(
    full_leakage_safe_candidate_features
)

redundancy_experiment_variants = {
    "Full Reference": (
        full_redundancy_reference_features
    ),
    "Remove Min Donation": (
        remove_feature_group(
            full_redundancy_reference_features,
            [
                "feature_past_5yr_min_donation",
            ],
        )
    ),
    "Remove Address Other": (
        remove_feature_group(
            full_redundancy_reference_features,
            [
                "feature_address_type_other",
            ],
        )
    ),
    "Remove Postal Missing": (
        remove_feature_group(
            full_redundancy_reference_features,
            [
                "feature_postal_code_missing_flag",
            ],
        )
    ),
    "Remove All Near-Constant": (
        remove_feature_group(
            full_redundancy_reference_features,
            near_constant_candidate_features,
        )
    ),
    "Remove Deterministic Scale Duplicates": (
        remove_feature_group(
            full_redundancy_reference_features,
            deterministic_scale_duplicate_features,
        )
    ),
    "Numeric Recency Only": (
        remove_feature_group(
            full_redundancy_reference_features,
            recency_indicator_features,
        )
    ),
    "Recency Flags Only": (
        remove_feature_group(
            full_redundancy_reference_features,
            [
                numeric_recency_feature,
            ],
        )
    ),
    "Reduced Numeric Recency": (
        remove_feature_group(
            full_redundancy_reference_features,
            (
                near_constant_candidate_features
                + deterministic_scale_duplicate_features
                + recency_indicator_features
            ),
        )
    ),
    "Reduced Recency Flags": (
        remove_feature_group(
            full_redundancy_reference_features,
            (
                near_constant_candidate_features
                + deterministic_scale_duplicate_features
                + [
                    numeric_recency_feature,
                ]
            ),
        )
    ),
}

redundancy_experiment_model_names = [
    "L1 Logistic",
    "RF Moderate",
]

redundancy_experiment_expected_counts = {
    "Full Reference": 53,
    "Remove Min Donation": 52,
    "Remove Address Other": 52,
    "Remove Postal Missing": 52,
    "Remove All Near-Constant": 50,
    "Remove Deterministic Scale Duplicates": 50,
    "Numeric Recency Only": 49,
    "Recency Flags Only": 52,
    "Reduced Numeric Recency": 43,
    "Reduced Recency Flags": 46,
}

In [76]:
# Define reusable redundancy experiment evaluation function
def evaluate_redundancy_experiment_fold(
    model_name,
    variant_name,
    feature_columns,
    fold_number,
    train_indices,
    validation_indices,
):
    fold_pipeline = (
        build_feature_set_comparison_pipeline(
            model_name=model_name,
            feature_columns=feature_columns,
        )
    )

    fold_pipeline.fit(
        features_primary_development[
            feature_columns
        ].iloc[train_indices],
        target_primary_development.iloc[
            train_indices
        ],
    )

    train_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[train_indices]
        )[:, 1]
    )

    validation_probability_positive = (
        fold_pipeline.predict_proba(
            features_primary_development[
                feature_columns
            ].iloc[validation_indices]
        )[:, 1]
    )

    fold_train_target = (
        target_primary_development.iloc[
            train_indices
        ]
    )

    fold_validation_target = (
        target_primary_development.iloc[
            validation_indices
        ]
    )

    transformed_feature_count = len(
        fold_pipeline.named_steps[
            "preprocessor"
        ].get_feature_names_out()
    )

    return {
        "model": model_name,
        "variant": variant_name,
        "fold": fold_number,
        "validation_indices": validation_indices,
        "probability_positive": (
            validation_probability_positive
        ),
        "train_pr_auc": average_precision_score(
            fold_train_target,
            train_probability_positive,
        ),
        "pr_auc": average_precision_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "roc_auc": roc_auc_score(
            fold_validation_target,
            validation_probability_positive,
        ),
        "transformed_feature_count": (
            transformed_feature_count
        ),
    }

In [78]:
# Evaluate near constant and redundant feature variants
redundancy_experiment_tasks = [
    (
        model_name,
        variant_name,
        feature_columns,
        fold_number,
        train_indices,
        validation_indices,
    )
    for model_name
    in redundancy_experiment_model_names
    for variant_name, feature_columns
    in redundancy_experiment_variants.items()
    for fold_number, (
        train_indices,
        validation_indices,
    ) in enumerate(
        primary_cv_splits,
        start=1,
    )
]

with joblib.parallel_config(
    backend="loky",
    inner_max_num_threads=1,
):
    redundancy_experiment_cv_results = (
        joblib.Parallel(
            n_jobs=REDUNDANCY_EXPERIMENT_PARALLEL_JOBS,
        )(
            joblib.delayed(
                evaluate_redundancy_experiment_fold
            )(
                model_name,
                variant_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            )
            for (
                model_name,
                variant_name,
                feature_columns,
                fold_number,
                train_indices,
                validation_indices,
            ) in redundancy_experiment_tasks
        )
    )

redundancy_experiment_records = []
redundancy_experiment_oof_probabilities = {}

for model_name in redundancy_experiment_model_names:
    for (
        variant_name,
        feature_columns,
    ) in redundancy_experiment_variants.items():
        combination_results = sorted(
            [
                result
                for result
                in redundancy_experiment_cv_results
                if (
                    result["model"] == model_name
                    and result["variant"]
                    == variant_name
                )
            ],
            key=lambda result: result["fold"],
        )

        oof_probability_positive = np.full(
            len(target_primary_development),
            np.nan,
            dtype=float,
        )

        for fold_result in combination_results:
            oof_probability_positive[
                fold_result[
                    "validation_indices"
                ]
            ] = fold_result[
                "probability_positive"
            ]

        if np.isnan(
            oof_probability_positive
        ).any():
            raise AssertionError(
                "Incomplete OOF probabilities for "
                f"{model_name} with "
                f"{variant_name}."
            )

        evaluation_metrics = (
            calculate_classification_metrics(
                target_true=(
                    target_primary_development
                ),
                probability_positive=(
                    oof_probability_positive
                ),
                threshold=(
                    DEFAULT_PROBABILITY_THRESHOLD
                ),
            )
        )

        cv_train_pr_auc_mean = np.mean([
            result["train_pr_auc"]
            for result in combination_results
        ])

        cv_pr_auc_mean = np.mean([
            result["pr_auc"]
            for result in combination_results
        ])

        redundancy_experiment_oof_probabilities[
            (
                model_name,
                variant_name,
            )
        ] = oof_probability_positive

        redundancy_experiment_records.append({
            "model": model_name,
            "variant": variant_name,
            "source_feature_count": len(
                feature_columns
            ),
            "removed_feature_count": (
                len(
                    full_redundancy_reference_features
                )
                - len(feature_columns)
            ),
            "transformed_feature_count": max(
                result[
                    "transformed_feature_count"
                ]
                for result
                in combination_results
            ),
            "cv_train_pr_auc_mean": (
                cv_train_pr_auc_mean
            ),
            "cv_pr_auc_mean": (
                cv_pr_auc_mean
            ),
            "cv_pr_auc_std": np.std([
                result["pr_auc"]
                for result
                in combination_results
            ]),
            "cv_pr_auc_gap": (
                cv_train_pr_auc_mean
                - cv_pr_auc_mean
            ),
            "cv_roc_auc_mean": np.mean([
                result["roc_auc"]
                for result
                in combination_results
            ]),
            "oof_pr_auc": evaluation_metrics[
                "pr_auc_average_precision"
            ],
            "recall": evaluation_metrics[
                "recall"
            ],
            "precision": evaluation_metrics[
                "precision"
            ],
            "f1_score": evaluation_metrics[
                "f1_score"
            ],
            "balanced_accuracy": (
                evaluation_metrics[
                    "balanced_accuracy"
                ]
            ),
            "predicted_positive_rate": (
                evaluation_metrics[
                    "predicted_positive_rate"
                ]
            ),
            "brier_score": evaluation_metrics[
                "brier_score"
            ],
            "pr_auc_multiple_vs_dummy": (
                cv_pr_auc_mean
                / dummy_reference_pr_auc
            ),
        })

redundancy_experiment_results = pd.DataFrame(
    redundancy_experiment_records
)

full_reference_performance = (
    redundancy_experiment_results.loc[
        redundancy_experiment_results[
            "variant"
        ].eq("Full Reference"),
        [
            "model",
            "cv_pr_auc_mean",
        ],
    ]
    .set_index("model")[
        "cv_pr_auc_mean"
    ]
    .to_dict()
)

redundancy_experiment_results[
    "pr_auc_change_vs_full"
] = redundancy_experiment_results.apply(
    lambda row: (
        row["cv_pr_auc_mean"]
        - full_reference_performance[
            row["model"]
        ]
    ),
    axis=1,
)

redundancy_experiment_results = (
    redundancy_experiment_results
    .sort_values(
        [
            "model",
            "cv_pr_auc_mean",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(redundancy_experiment_results.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "model",
            "variant",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0, th.col1",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "source_feature_count": "{:,.0f}",
        "removed_feature_count": "{:,.0f}",
        "transformed_feature_count": "{:,.0f}",
        "cv_train_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_mean": "{:.4f}",
        "cv_pr_auc_std": "{:.4f}",
        "cv_pr_auc_gap": "{:.4f}",
        "cv_roc_auc_mean": "{:.4f}",
        "oof_pr_auc": "{:.4f}",
        "recall": "{:.4f}",
        "precision": "{:.4f}",
        "f1_score": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "predicted_positive_rate": "{:.2%}",
        "brier_score": "{:.4f}",
        "pr_auc_multiple_vs_dummy": "{:.2f}×",
        "pr_auc_change_vs_full": "{:+.4f}",
    })
)

redundancy_model_summary_records = []

for model_name in redundancy_experiment_model_names:
    model_results = (
        redundancy_experiment_results.loc[
            redundancy_experiment_results[
                "model"
            ].eq(model_name)
        ]
        .sort_values(
            "cv_pr_auc_mean",
            ascending=False,
        )
    )

    best_row = model_results.iloc[0]

    reference_row = model_results.loc[
        model_results[
            "variant"
        ].eq("Full Reference")
    ].iloc[0]

    redundancy_model_summary_records.append({
        "model": model_name,
        "best_variant": (
            best_row["variant"]
        ),
        "best_feature_count": (
            best_row["source_feature_count"]
        ),
        "best_cv_pr_auc": (
            best_row["cv_pr_auc_mean"]
        ),
        "full_reference_cv_pr_auc": (
            reference_row["cv_pr_auc_mean"]
        ),
        "change_vs_full": (
            best_row["pr_auc_change_vs_full"]
        ),
        "best_cv_pr_auc_std": (
            best_row["cv_pr_auc_std"]
        ),
        "best_cv_pr_auc_gap": (
            best_row["cv_pr_auc_gap"]
        ),
    })

redundancy_model_summary = pd.DataFrame(
    redundancy_model_summary_records
).sort_values(
    "best_cv_pr_auc",
    ascending=False,
).reset_index(drop=True)

display(redundancy_model_summary.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=[
            "model",
            "best_variant",
        ],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0, th.col1",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "best_feature_count": "{:,.0f}",
        "best_cv_pr_auc": "{:.4f}",
        "full_reference_cv_pr_auc": "{:.4f}",
        "change_vs_full": "{:+.4f}",
        "best_cv_pr_auc_std": "{:.4f}",
        "best_cv_pr_auc_gap": "{:.4f}",
    })
)

model,variant,source_feature_count,removed_feature_count,transformed_feature_count,cv_train_pr_auc_mean,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_gap,cv_roc_auc_mean,oof_pr_auc,recall,precision,f1_score,balanced_accuracy,predicted_positive_rate,brier_score,pr_auc_multiple_vs_dummy,pr_auc_change_vs_full
L1 Logistic,Remove Address Other,52,1,76,0.0853,0.0715,0.0049,0.0137,0.4973,0.0695,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,+0.0000
L1 Logistic,Recency Flags Only,52,1,76,0.0852,0.0715,0.0048,0.0137,0.4980,0.0695,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,+0.0000
L1 Logistic,Full Reference,53,0,77,0.0852,0.0715,0.0048,0.0138,0.4976,0.0695,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,+0.0000
L1 Logistic,Remove Postal Missing,52,1,76,0.0852,0.0715,0.0048,0.0138,0.4975,0.0695,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,-0.0000
L1 Logistic,Reduced Numeric Recency,43,10,67,0.0844,0.0714,0.0043,0.0130,0.4957,0.0700,0.0098,0.8333,0.0195,0.5049,0.07%,0.0519,1.29×,-0.0000
L1 Logistic,Remove Deterministic Scale Duplicates,50,3,74,0.0852,0.0714,0.0048,0.0138,0.4973,0.0694,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,-0.0001
L1 Logistic,Reduced Recency Flags,46,7,70,0.0850,0.0712,0.0045,0.0138,0.4984,0.0696,0.0098,0.8333,0.0195,0.5049,0.07%,0.0519,1.29×,-0.0002
L1 Logistic,Numeric Recency Only,49,4,73,0.0844,0.0712,0.0053,0.0132,0.4948,0.0695,0.0112,0.8095,0.0220,0.5055,0.08%,0.0519,1.29×,-0.0003
L1 Logistic,Remove Min Donation,52,1,76,0.0850,0.0711,0.0046,0.0139,0.4976,0.0695,0.0098,0.8333,0.0195,0.5049,0.07%,0.0519,1.29×,-0.0004
L1 Logistic,Remove All Near-Constant,50,3,74,0.0850,0.0711,0.0046,0.0139,0.4981,0.0695,0.0098,0.8333,0.0195,0.5049,0.07%,0.0519,1.28×,-0.0004


model,best_variant,best_feature_count,best_cv_pr_auc,full_reference_cv_pr_auc,change_vs_full,best_cv_pr_auc_std,best_cv_pr_auc_gap
RF Moderate,Full Reference,53,0.0766,0.0766,+0.0000,0.0051,0.0755
L1 Logistic,Remove Address Other,52,0.0715,0.0715,+0.0000,0.0049,0.0137


In [81]:
# Validate near constant and redundancy experiment
near_constant_dominant_percentages = {}

for feature in near_constant_candidate_features:
    value_counts = (
        features_primary_development[
            feature
        ].value_counts(
            normalize=True,
            dropna=False,
        )
    )

    near_constant_dominant_percentages[
        feature
    ] = (
        value_counts.iloc[0] * 100
    )

average_total_relationship_valid = np.allclose(
    features_primary_development[
        "feature_past_5yr_average_donation"
    ].to_numpy(),
    (
        features_primary_development[
            "feature_past_5yr_total_donation"
        ].to_numpy()
        / 5
    ),
    rtol=1e-10,
    atol=1e-10,
)

log_average_relationship_valid = np.allclose(
    features_primary_development[
        "feature_log_past_5yr_average_donation"
    ].to_numpy(),
    np.log1p(
        features_primary_development[
            "feature_past_5yr_average_donation"
        ].to_numpy()
    ),
    rtol=1e-10,
    atol=1e-10,
)

frequency_relationship_valid = np.allclose(
    features_primary_development[
        "feature_past_5yr_donation_frequency_rate"
    ].to_numpy(),
    (
        features_primary_development[
            "feature_years_donated_past_5yr"
        ].to_numpy()
        / 5
    ),
    rtol=1e-10,
    atol=1e-10,
)

all_redundancy_features_safe = all(
    set(feature_columns).issubset(
        set(
            full_leakage_safe_candidate_features
        )
    )
    for feature_columns
    in redundancy_experiment_variants.values()
)

variant_counts_valid = all(
    len(
        redundancy_experiment_variants[
            variant_name
        ]
    ) == expected_count
    for variant_name, expected_count
    in redundancy_experiment_expected_counts.items()
)

task22_l1_full_pr_auc = (
    feature_set_comparison_results.loc[
        (
            feature_set_comparison_results[
                "model"
            ].eq("L1 Logistic")
        )
        & (
            feature_set_comparison_results[
                "feature_set"
            ].eq("Full Leakage-Safe")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

task22_rf_full_pr_auc = (
    feature_set_comparison_results.loc[
        (
            feature_set_comparison_results[
                "model"
            ].eq("RF Moderate")
        )
        & (
            feature_set_comparison_results[
                "feature_set"
            ].eq("Full Leakage-Safe")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

task24_l1_full_pr_auc = (
    redundancy_experiment_results.loc[
        (
            redundancy_experiment_results[
                "model"
            ].eq("L1 Logistic")
        )
        & (
            redundancy_experiment_results[
                "variant"
            ].eq("Full Reference")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

task24_rf_full_pr_auc = (
    redundancy_experiment_results.loc[
        (
            redundancy_experiment_results[
                "model"
            ].eq("RF Moderate")
        )
        & (
            redundancy_experiment_results[
                "variant"
            ].eq("Full Reference")
        ),
        "cv_pr_auc_mean",
    ].iloc[0]
)

redundancy_experiment_validation = pd.DataFrame({
    "validation_check": [
        "Redundancy variant count",
        "Model-family count",
        "Model-variant combinations",
        "Cross-validation fit count",
        "Full reference feature count",
        "Near-constant candidate count",
        "Deterministic duplicate candidate count",
        "Recency indicator count",
        "All near-constant candidates remain near constant",
        "Average donation equals total divided by five",
        "Log average matches log1p average",
        "Frequency rate equals years donated divided by five",
        "All variant feature counts match definitions",
        "All variant features are leakage-safe",
        "All combinations produced complete OOF predictions",
        "No missing OOF probabilities",
        "All probabilities between 0 and 1",
        "Same cross-validation folds reused",
        "L1 full reference reproduces Task 22",
        "Random forest full reference reproduces Task 22",
        "Each model family has an independently selected best variant",
        "Final test set remained unused",
    ],
    "expected": [
        10,
        2,
        20,
        100,
        53,
        3,
        3,
        4,
        "True",
        "True",
        "True",
        "True",
        "True",
        "True",
        len(target_primary_development),
        0,
        "True",
        CV_FOLDS,
        "True",
        "True",
        2,
        "True",
    ],
    "actual": [
        len(
            redundancy_experiment_variants
        ),
        len(
            redundancy_experiment_model_names
        ),
        len(
            redundancy_experiment_results
        ),
        len(
            redundancy_experiment_cv_results
        ),
        len(
            full_redundancy_reference_features
        ),
        len(
            near_constant_candidate_features
        ),
        len(
            deterministic_scale_duplicate_features
        ),
        len(
            recency_indicator_features
        ),
        str(all(
            percentage >= 99.50
            for percentage
            in near_constant_dominant_percentages.values()
        )),
        str(
            average_total_relationship_valid
        ),
        str(
            log_average_relationship_valid
        ),
        str(
            frequency_relationship_valid
        ),
        str(
            variant_counts_valid
        ),
        str(
            all_redundancy_features_safe
        ),
        min(
            len(probabilities)
            for probabilities
            in redundancy_experiment_oof_probabilities.values()
        ),
        sum(
            np.isnan(
                probabilities
            ).sum()
            for probabilities
            in redundancy_experiment_oof_probabilities.values()
        ),
        str(all(
            np.all(
                (probabilities >= 0)
                & (probabilities <= 1)
            )
            for probabilities
            in redundancy_experiment_oof_probabilities.values()
        )),
        len(
            primary_cv_splits
        ),
        str(
            np.isclose(
                task24_l1_full_pr_auc,
                task22_l1_full_pr_auc,
                atol=1e-10,
            )
        ),
        str(
            np.isclose(
                task24_rf_full_pr_auc,
                task22_rf_full_pr_auc,
                atol=1e-10,
            )
        ),
        len(
            redundancy_model_summary
        ),
        "True",
    ],
})

redundancy_experiment_validation[
    "passed"
] = [
    len(
        redundancy_experiment_variants
    ) == 10,
    len(
        redundancy_experiment_model_names
    ) == 2,
    len(
        redundancy_experiment_results
    ) == 20,
    len(
        redundancy_experiment_cv_results
    ) == 100,
    len(
        full_redundancy_reference_features
    ) == 53,
    len(
        near_constant_candidate_features
    ) == 3,
    len(
        deterministic_scale_duplicate_features
    ) == 3,
    len(
        recency_indicator_features
    ) == 4,
    all(
        percentage >= 99.50
        for percentage
        in near_constant_dominant_percentages.values()
    ),
    average_total_relationship_valid,
    log_average_relationship_valid,
    frequency_relationship_valid,
    variant_counts_valid,
    all_redundancy_features_safe,
    all(
        len(probabilities)
        == len(target_primary_development)
        for probabilities
        in redundancy_experiment_oof_probabilities.values()
    ),
    all(
        not np.isnan(
            probabilities
        ).any()
        for probabilities
        in redundancy_experiment_oof_probabilities.values()
    ),
    all(
        np.all(
            (probabilities >= 0)
            & (probabilities <= 1)
        )
        for probabilities
        in redundancy_experiment_oof_probabilities.values()
    ),
    len(
        primary_cv_splits
    ) == CV_FOLDS,
    np.isclose(
        task24_l1_full_pr_auc,
        task22_l1_full_pr_auc,
        atol=1e-10,
    ),
    np.isclose(
        task24_rf_full_pr_auc,
        task22_rf_full_pr_auc,
        atol=1e-10,
    ),
    len(
        redundancy_model_summary
    ) == 2,
    True,
]

display(redundancy_experiment_validation.style
    .hide(axis="index")
    .set_properties(**{
        "text-align": "center",
        "padding": "8px 16px",
    })
    .set_properties(
        subset=["validation_check"],
        **{"text-align": "left"}
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("padding", "8px 16px"),
            ],
        },
        {
            "selector": "th.col0",
            "props": [
                ("text-align", "left"),
            ],
        },
    ])
    .format({
        "expected": format_validation_value,
        "actual": format_validation_value,
        "passed": lambda value: str(value),
    })
)

failed_redundancy_checks = (
    redundancy_experiment_validation.loc[
        ~redundancy_experiment_validation[
            "passed"
        ],
        "validation_check",
    ].tolist()
)

if failed_redundancy_checks:
    raise AssertionError(
        "Near-constant and redundancy validation failed for: "
        + ", ".join(
            failed_redundancy_checks
        )
    )

print(
    "\nNear-constant and redundancy experiment validated."
)

for _, row in redundancy_model_summary.iterrows():
    print(
        f"{row['model']} best variant: "
        f"{row['best_variant']} "
        f"({row['best_cv_pr_auc']:.4f}, "
        f"{row['change_vs_full']:+.4f} vs. full)."
    )

validation_check,expected,actual,passed
Redundancy variant count,10,10,True
Model-family count,2,2,True
Model-variant combinations,20,20,True
Cross-validation fit count,100,100,True
Full reference feature count,53,53,True
Near-constant candidate count,3,3,True
Deterministic duplicate candidate count,3,3,True
Recency indicator count,4,4,True
All near-constant candidates remain near constant,True,True,True
Average donation equals total divided by five,True,True,True



Near-constant and redundancy experiment validated.
RF Moderate best variant: Full Reference (0.0766, +0.0000 vs. full).
L1 Logistic best variant: Remove Address Other (0.0715, +0.0000 vs. full).
